# ORB Strategy
***
A strategy that uses the ORB strategy to determine which timeframe and position is best to be traded.

**Pipeline:** libraries & config → data import → shared backtest engine → paper replication
(Zarattini, Barbon & Aziz 2024) → 13k-configuration search → analysis of the winner → winner vs SPY.
Every strategy in this notebook runs through the **same engine**, so results are directly comparable.

## Libraries & configuration

In [2]:
import itertools
import math
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

warnings.filterwarnings("ignore", message="All-NaN slice encountered")   # sessions with an empty window
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="invalid value encountered")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# ── Shared settings (every cell reads these) ────────────────────────────────
DATA_DIR = Path("/Users/andrew/main/trading/data/etfs")
SYMBOLS = ("CPER", "DIA", "GLD", "IEF", "IWM", "QQQ", "SHY", "SLV", "SPY", "TLT", "USO")
BENCHMARK = "SPY"
EQUITY_ETFS = {"SPY", "QQQ", "IWM", "DIA"}
INITIAL_EQUITY = 25_000.0
TICK = 0.01
COMMISSION = 0.0035                    # $/share/side (IBKR Pro tiered entry level — verify current)
SLIPPAGE_TICKS = 1                     # adverse ticks on every stop/market fill (the paper assumed 0)
IEX_SWITCH = pd.Timestamp("2022-03-07")  # data source changes from pitrading to IEX-only volume

## Data importing

In [3]:
# ═════════════════════════════════════════════════════════════════════════════
# DATA IMPORT — ETF 1-minute OHLCV
#
# Loads each ETF's 1-minute CSV into a DataFrame indexed by timestamp
# (Eastern Time wall-clock, bars labelled by bar-open time).
#
# Output: raw_1m[symbol] = DataFrame[open, high, low, close, volume, source]
# ═════════════════════════════════════════════════════════════════════════════
REQUIRED = ["datetime", "open", "high", "low", "close", "volume"]


def load_etf_1m(path: Path) -> pd.DataFrame:
    """Read one ETF CSV → sorted, de-duplicated, DatetimeIndexed OHLCV frame."""
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, engine="pyarrow")
    df.columns = df.columns.str.lower()
    missing = set(REQUIRED) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    df["datetime"] = pd.to_datetime(df["datetime"], format="%Y-%m-%d %H:%M:%S")
    df = df.set_index("datetime").sort_index()
    df = df[~df.index.duplicated(keep="last")]
    if "source" not in df.columns:
        df["source"] = "unknown"
    return df[["open", "high", "low", "close", "volume", "source"]]


raw_1m: dict[str, pd.DataFrame] = {sym: load_etf_1m(DATA_DIR / f"{sym}_clean_1min.csv") for sym in SYMBOLS}

display(pd.DataFrame({
    sym: {"bars": len(df), "first": df.index.min(), "last": df.index.max(),
          "sessions": df.index.normalize().nunique(),
          "iex_from": df.index[df["source"].eq("iex")].min()}
    for sym, df in raw_1m.items()
}).T)

,bars,first,last,sessions,iex_from
CPER,109059,2011-11-15 09:37:00,2026-09-21 15:59:00,3315,2022-03-07 09:30:00
DIA,2067170,2002-12-30 09:30:00,2026-09-21 15:58:00,5966,2022-03-07 09:30:00
GLD,1924936,2004-11-18 09:30:00,2026-09-21 15:59:00,5490,2022-03-07 09:30:00
IEF,1560831,2002-12-30 09:31:00,2026-09-21 15:57:00,5966,2022-03-07 09:30:00
IWM,2217669,2002-12-30 09:30:00,2026-09-21 15:57:00,5966,2022-03-07 09:30:00
QQQ,2245015,2002-12-30 09:30:00,2026-09-21 15:56:00,5968,2022-03-07 09:30:00
SHY,1523666,2002-12-30 09:30:00,2026-09-21 15:59:00,5966,2022-03-07 09:30:00
SLV,1752691,2006-04-28 09:32:00,2026-09-21 15:59:00,5128,2022-03-07 09:33:00
SPY,2250959,2002-12-30 09:30:00,2026-09-21 15:58:00,5968,2022-03-07 09:30:00
TLT,2005004,2002-12-30 09:31:00,2026-09-21 15:59:00,5966,2022-03-07 09:30:00


## Backtest engine
One engine shared by every strategy below. Fill rules are conservative and identical everywhere:
* **Entry:** a stop order fills at `max(level, bar open)`; a bar-close entry fills at that bar's close.
* **Stops** are rounded away from entry. A stop-order entry is checked for a stop on its own bar. A gap through
  a stop fills at the bar open. Breakeven and trailing stops use the best price up to the *previous* bar.
* **Targets** need a one-tick trade-through. The stop wins same-bar ties.
* **Time exit** at the close of the exit bar, or of the last bar that exists before it.
* **Costs:** commission per share per side, plus `slip_ticks` adverse ticks on every stop/market fill.
* **Ex-ante features:** everything a filter uses is known before the trade (daily features are shifted one session).

In [4]:
RTH_OPEN, N_BARS = 9 * 60 + 30, 390            # column j = bar opening at 09:30 + j minutes
OR_WINDOWS = (5, 10, 15, 30, 60)
LOOKBACK = 14
MIN_OR_COVERAGE = 0.80                         # share of OR minutes that must have a bar
MIN_SESSION_COVERAGE = 0.80
MAX_OPEN_DELAY = 2                             # first bar must open by 09:32
MIN_PRICE = 5.0
MIN_STOP_TICKS = 3
TOL = TICK / 4                                 # float32 comparison tolerance


def _first_true(mask: np.ndarray) -> np.ndarray:
    """Index of the first True per row, -1 if none."""
    i = mask.argmax(axis=1)
    return np.where(mask[np.arange(len(mask)), i], i, -1)


# ── 1. Session panels: rows = sessions, columns = minutes after 09:30 (NaN = missing bar) ──
def build_panel(df: pd.DataFrame) -> dict:
    minute = df.index.hour * 60 + df.index.minute - RTH_OPEN
    df = df.loc[(minute >= 0) & (minute < N_BARS)]
    px = df[["open", "high", "low", "close"]]
    df = df.loc[(px > 0).all(axis=1) & (df["high"] >= df["low"]) & np.isfinite(px).all(axis=1)]
    col = (df.index.hour * 60 + df.index.minute - RTH_OPEN).to_numpy()
    codes, dates = pd.factorize(df.index.normalize(), sort=True)

    def mat(values: np.ndarray) -> np.ndarray:
        m = np.full((len(dates), N_BARS), np.nan, dtype=np.float32)
        m[codes, col] = values
        return m

    O, H, L, C, V = (mat(df[k].to_numpy(np.float64)) for k in ("open", "high", "low", "close", "volume"))
    present = ~np.isnan(C)
    has_bar = present.any(axis=1)
    first = np.where(has_bar, present.argmax(axis=1), -1)
    last = np.where(has_bar, N_BARS - 1 - present[:, ::-1].argmax(axis=1), -1)
    coverage = present.sum(axis=1) / np.maximum(last + 1, 1)
    rows = np.arange(len(dates))
    daily = pd.DataFrame({
        "open": np.where(has_bar, O[rows, np.maximum(first, 0)], np.nan),
        "high": np.nanmax(H, axis=1), "low": np.nanmin(L, axis=1),
        "close": np.where(has_bar, C[rows, np.maximum(last, 0)], np.nan),
        "volume": np.nansum(V, axis=1),
    }, index=pd.DatetimeIndex(dates, name="date")).astype(float).round(6)
    daily["source"] = df["source"].astype(str).groupby(codes).first().to_numpy()
    session_ok = has_bar & (first <= MAX_OPEN_DELAY) & (coverage >= MIN_SESSION_COVERAGE)
    return dict(dates=daily.index, O=O, H=H, L=L, C=C, V=V, first=first, last=last,
                daily=daily, session_ok=session_ok)


# ── 2. Ex-ante daily features (all shifted: known before 09:30) ──
def daily_features(d: pd.DataFrame) -> pd.DataFrame:
    pc = d["close"].shift(1)
    tr = pd.concat([d["high"] - d["low"], (d["high"] - pc).abs(), (d["low"] - pc).abs()], axis=1).max(axis=1)
    atr14 = tr.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
    atr50 = tr.ewm(alpha=1 / 50, adjust=False, min_periods=50).mean()
    rng = d["high"] - d["low"]
    f = pd.DataFrame(index=d.index)
    f["atr"] = atr14.shift(1)
    f["avg_vol"] = d["volume"].rolling(LOOKBACK, min_periods=LOOKBACK).mean().shift(1)
    f["gap_atr"] = (d["open"] - pc) / f["atr"]
    f["trend"] = np.sign(d["close"] - d["close"].rolling(20, min_periods=20).mean()).shift(1)
    f["atr_ratio"] = (atr14 / atr50).shift(1)
    f["nr7"] = (rng <= rng.rolling(7, min_periods=7).min()).shift(1, fill_value=False).astype(bool)
    f["open"] = d["open"]
    return f


# ── 3. Opening-range cache per window (known at 09:30 + n) ──
def or_cache(p: dict, n: int) -> dict:
    O, H, L, C, V, d = p["O"], p["H"], p["L"], p["C"], p["V"], p["daily"]
    rows = np.arange(len(d))
    present = ~np.isnan(C[:, :n])
    last_in = n - 1 - present[:, ::-1].argmax(axis=1)
    ok = p["session_ok"] & (present.mean(axis=1) >= MIN_OR_COVERAGE) & (p["last"] > n)
    o_open = O[rows, np.maximum(p["first"], 0)].astype(float)
    o_close = C[rows, last_in].astype(float)
    vol = pd.Series(np.nansum(V[:, :n], axis=1), index=d.index)
    # Relative volume vs the previous 14 valid sessions; the OR must include the 09:30 print
    # (~40 % of 5-min OR volume); computed within each data-source regime (IEX switch).
    rv_ok = ok & ~np.isnan(V[:, 0])
    base = (vol.where(rv_ok).groupby(d["source"].to_numpy())
            .transform(lambda s: s.dropna().rolling(LOOKBACK).mean().shift(1).reindex(s.index)))
    return dict(ok=ok,
                hi=np.nanmax(H[:, :n], axis=1).astype(float), lo=np.nanmin(L[:, :n], axis=1).astype(float),
                dir=np.where(ok, np.sign(np.round(o_close - o_open, 6)), 0).astype(int),
                rv=(vol / base).where(rv_ok & (base > 0)).to_numpy())


t0 = time.perf_counter()
panels = {sym: build_panel(df) for sym, df in raw_1m.items()}
FEAT = {s: daily_features(p["daily"]) for s, p in panels.items()}
ORC = {s: {n: or_cache(p, n) for n in OR_WINDOWS} for s, p in panels.items()}
CAL = pd.DatetimeIndex(sorted(set().union(*[set(p["dates"]) for p in panels.values()])))
YEARS = CAL.year.to_numpy()
YEAR_IDS = np.unique(YEARS)
YIDX = np.searchsorted(YEAR_IDS, YEARS)
R_SPY = panels[BENCHMARK]["daily"]["close"].reindex(CAL).ffill().pct_change().fillna(0.0).to_numpy()


# ── 4. Strategy definition ──
@dataclass(frozen=True)
class Logic:
    n: int                 # opening-range minutes
    dir_mode: str          # "candle" | "both" | "candle+gap" | "candle+trend"
    buffer: float          # entry beyond the OR, × ATR
    confirm: str           # "touch" (stop order) | "close" (bar-close confirmation)
    cutoff: int            # last entry column (minutes after 09:30)
    stop: tuple            # ("atr", k) | ("or_opp", 0) | ("or_mid", 0)
    be_R: float            # move stop to breakeven after +x R (0 = off)
    trail_atr: float       # trailing stop, × ATR from best price (0 = off)
    target_R: float        # profit target in R (0 = none)
    exit_col: int          # time exit column (389 = 15:59 bar → 16:00)

    def valid(self) -> bool:
        return self.cutoff > self.n and self.exit_col > self.n + 5 and not (self.be_R and self.trail_atr)


@dataclass(frozen=True)
class Filt:
    rv_min: float          # relative volume in the OR
    or_atr_max: float      # OR width / ATR
    gap_min: float         # |gap| / ATR
    regime: str            # "any" | "expanding" | "contracting" (ATR14 / ATR50)
    nr7: bool              # previous day was NR7
    universe: str          # "all" | "equity"
    top_n: int | None      # cross-sectional cap by relative volume
    min_avg_vol: float = 0.0   # paper liquidity filter (shares/day, previous 14 days)
    min_atr: float = 0.0       # paper volatility filter ($, ATR14 of previous days)


# ── 5. Vectorised simulator: one Logic × one asset, all sessions at once ──
#     Prices are mapped to "signed" space (shorts negated) so one code path serves both directions.
def simulate(sym: str, lg: Logic, slip_ticks: float = SLIPPAGE_TICKS) -> pd.DataFrame:
    p, f, oc = panels[sym], FEAT[sym], ORC[sym][lg.n]
    atr = f["atr"].to_numpy()
    base_ok = oc["ok"] & np.isfinite(atr) & (atr > 0) & (f["open"].to_numpy() > MIN_PRICE)
    cd = oc["dir"]
    if lg.dir_mode == "candle":
        dsel = cd
    elif lg.dir_mode == "candle+gap":
        dsel = np.where(np.sign(np.nan_to_num(f["gap_atr"].to_numpy())) == cd, cd, 0)
    elif lg.dir_mode == "candle+trend":
        dsel = np.where(np.nan_to_num(f["trend"].to_numpy()) == cd, cd, 0)
    else:
        dsel = np.where(base_ok, 2, 0)                      # "both": evaluate both sides
    rows = np.flatnonzero(base_ok & (dsel != 0))
    if rows.size == 0:
        return pd.DataFrame()

    n, slip = lg.n, slip_ticks * TICK
    O, H, L, C = (p[k][rows, n:lg.exit_col + 1] for k in ("O", "H", "L", "C"))
    j = np.arange(O.shape[1])[None, :]
    r_ = np.arange(len(rows))
    # last bar that actually exists at or before the time exit
    last_j = np.where(~np.isnan(C) & (j <= (p["last"][rows] - n)[:, None]), j, -1).max(axis=1)
    entry_lim = np.minimum(lg.cutoff - n, last_j - 1)
    a = atr[rows]
    hi, lo = oc["hi"][rows], oc["lo"][rows]
    trig_long = np.ceil(np.round((hi + lg.buffer * a) / TICK, 6)) * TICK
    trig_short = np.floor(np.round((lo - lg.buffer * a) / TICK, 6)) * TICK

    def scan(sign: float):
        ts = sign * (trig_long if sign > 0 else trig_short)
        Hs, Cs, Os = (H, C, O) if sign > 0 else (-L, -C, -O)
        win = j <= entry_lim[:, None]
        if lg.confirm == "touch":
            e = _first_true((Hs >= ts[:, None] - TOL) & win)
            fill = np.maximum(ts, np.round(Os[r_, np.maximum(e, 0)] / TICK) * TICK) + slip
        else:
            e = _first_true((Cs > ts[:, None] + TOL) & win)
            fill = np.round(Cs[r_, np.maximum(e, 0)] / TICK) * TICK + slip
        return e, fill

    if lg.dir_mode == "both":
        eL, fL = scan(1.0)
        eS, fS = scan(-1.0)
        eL_, eS_ = np.where(eL < 0, 10**6, eL), np.where(eS < 0, 10**6, eS)
        d = np.where(eL_ < eS_, 1.0, np.where(eS_ < eL_, -1.0, 0.0))     # same-bar double break → skip
        e = np.where(d > 0, eL, np.where(d < 0, eS, -1))
        fill_s = np.where(d > 0, fL, fS)
    else:
        d = dsel[rows].astype(float)
        e = np.full(len(rows), -1)
        fill_s = np.full(len(rows), np.nan)
        for sgn in (1.0, -1.0):
            mk = d == sgn
            if mk.any():
                ee, ff = scan(sgn)
                e, fill_s = np.where(mk, ee, e), np.where(mk, ff, fill_s)
    trig_ok = (e >= 0) & (d != 0)

    dd = d[:, None]
    Hs, Ls = np.where(dd > 0, H, -L), np.where(dd > 0, L, -H)
    Os, Cs = np.where(dd > 0, O, -O), np.where(dd > 0, C, -C)

    kind, k = lg.stop
    if kind == "atr":
        stop_s = fill_s - k * a
    elif kind == "or_opp":
        stop_s = np.where(d > 0, lo, -hi) - TICK
    else:
        stop_s = np.where(d > 0, (hi + lo) / 2, -(hi + lo) / 2)
    stop_s = np.floor(np.round(stop_s / TICK, 6)) * TICK      # rounded away from entry
    risk = fill_s - stop_s
    trig_ok &= np.isfinite(risk) & (risk >= MIN_STOP_TICKS * TICK)

    within = j <= last_j[:, None]
    ec = np.maximum(e, 0)[:, None]
    first_live = ec if lg.confirm == "touch" else ec + 1       # a stop-order entry bar can already stop out
    fav = np.where(within & (j >= first_live), Hs, -np.inf)
    run = np.maximum.accumulate(np.nan_to_num(fav, nan=-np.inf), axis=1)
    prev_best = np.maximum(np.concatenate([np.full((len(rows), 1), -np.inf), run[:, :-1]], axis=1),
                           fill_s[:, None])
    level = np.broadcast_to(stop_s[:, None], Hs.shape).copy()
    if lg.be_R > 0:
        level = np.where(prev_best - fill_s[:, None] >= lg.be_R * risk[:, None],
                         np.maximum(level, fill_s[:, None]), level)
    if lg.trail_atr > 0:
        level = np.maximum(level, prev_best - lg.trail_atr * a[:, None])

    js = _first_true(within & (j >= first_live) & (Ls <= level + TOL))
    lv = level[r_, np.maximum(js, 0)]
    o_s = np.round(Os[r_, np.maximum(js, 0)] / TICK) * TICK
    stop_fill = np.where(js == e, lv, np.minimum(lv, o_s)) - slip
    if lg.target_R > 0:
        tgt = np.ceil(np.round((fill_s + lg.target_R * risk) / TICK, 6)) * TICK
        jt = _first_true(within & (j >= ec + 1) & (Hs >= tgt[:, None] + TICK - TOL))
        tgt_fill = np.maximum(tgt, np.round(Os[r_, np.maximum(jt, 0)] / TICK) * TICK)
    else:
        jt, tgt_fill = np.full(len(rows), -1), np.full(len(rows), np.nan)
    big = 10**6
    js_, jt_ = np.where(js >= 0, js, big), np.where(jt >= 0, jt, big)
    by_stop = (js_ < big) & (js_ <= jt_)
    by_tgt = (jt_ < big) & (jt_ < js_)
    eod_fill = np.round(Cs[r_, np.maximum(last_j, 0)] / TICK) * TICK - slip
    exit_s = np.where(by_stop, stop_fill, np.where(by_tgt, tgt_fill, eod_fill))
    exit_j = np.where(by_stop, js, np.where(by_tgt, jt, last_j))

    pnl_ps = exit_s - fill_s - 2 * COMMISSION                   # $ per share after costs
    trig_ok &= np.isfinite(pnl_ps)
    fr = f.iloc[rows]
    out = pd.DataFrame({
        "asset": sym, "date": p["dates"][rows], "dir": d.astype(int), "triggered": trig_ok,
        "entry_col": e + n, "exit_col": exit_j + n, "entry": np.abs(fill_s), "risk_ps": risk,
        "pnl_ps": pnl_ps, "R": np.where(trig_ok, pnl_ps / np.where(risk > 0, risk, np.nan), np.nan),
        "rv": oc["rv"][rows], "or_atr": (hi - lo) / a, "gap_atr": fr["gap_atr"].to_numpy(),
        "atr_ratio": fr["atr_ratio"].to_numpy(), "nr7": fr["nr7"].to_numpy(),
        "atr": a, "avg_vol": fr["avg_vol"].to_numpy(),
        "reason": np.where(by_stop, "stop", np.where(by_tgt, "target", "time")),
    })
    out["cal_idx"] = CAL.get_indexer(out["date"])
    return out


def candidates(lg: Logic, slip_ticks: float = SLIPPAGE_TICKS) -> pd.DataFrame:
    parts = [c for c in (simulate(s, lg, slip_ticks) for s in panels) if not c.empty]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def apply_filter(c: pd.DataFrame, fl: Filt) -> pd.DataFrame:
    """OR-time filters + top-N ranking on candidates (before triggering is known) → triggered trades."""
    m = (c["or_atr"] <= fl.or_atr_max) & (c["gap_atr"].abs().fillna(0) >= fl.gap_min)
    if fl.rv_min > 0:
        m &= c["rv"] >= fl.rv_min
    if fl.regime == "expanding":
        m &= c["atr_ratio"] > 1
    elif fl.regime == "contracting":
        m &= c["atr_ratio"] < 1
    if fl.nr7:
        m &= c["nr7"]
    if fl.universe == "equity":
        m &= c["asset"].isin(EQUITY_ETFS)
    if fl.min_avg_vol > 0:
        m &= c["avg_vol"] >= fl.min_avg_vol
    if fl.min_atr > 0:
        m &= c["atr"] > fl.min_atr
    c = c.loc[m]
    if fl.top_n:
        c = c.loc[c.groupby("date")["rv"].rank(ascending=False, method="first", na_option="bottom") <= fl.top_n]
    return c.loc[c["triggered"]]


# ── 6. Portfolio sizing ──
def fixed_risk_returns(t: pd.DataFrame, risk: float = 0.0025, pos_cap: float = 1.0,
                       gross_cap: float = 4.0) -> np.ndarray:
    """Fast fractional sizing: `risk` of equity per trade, ≤ pos_cap × equity notional per trade,
    and the day's total notional scaled down to ≤ gross_cap × equity."""
    assert np.isfinite(t["R"].to_numpy()).all(), "non-finite trade R reached the portfolio"
    ent, rps, ci = t["entry"].to_numpy(), t["risk_ps"].to_numpy(), t["cal_idx"].to_numpy()
    w = np.minimum(1.0, pos_cap * rps / (risk * ent))                   # per-trade notional cap
    notional = risk * w * ent / rps                                     # × equity
    day_gross = np.bincount(ci, weights=notional, minlength=len(CAL))
    w = w * np.minimum(1.0, gross_cap / np.maximum(day_gross, 1e-12))[ci]
    return np.bincount(ci, weights=risk * w * t["R"].to_numpy(), minlength=len(CAL))


def run_risk(t: pd.DataFrame, rk: dict) -> np.ndarray:
    """Sequential, integer-share simulation with risk controls (keys: risk, gross_lev, max_trades,
    daily_loss, dd_throttle, eq_filter; optional pos_cap = max notional per trade × equity)."""
    t = t.sort_values(["cal_idx", "entry_col"])
    ci, ent, rps, pps = (t[k].to_numpy() for k in ("cal_idx", "entry", "risk_ps", "pnl_ps"))
    ecol, xcol = t["entry_col"].to_numpy(), t["exit_col"].to_numpy()
    pos_cap = rk.get("pos_cap", math.inf)
    day_pnl = np.zeros(len(CAL))
    equity = peak = INITIAL_EQUITY
    starts = np.flatnonzero(np.r_[True, ci[1:] != ci[:-1]]) if len(ci) else np.array([], int)
    ends = np.r_[starts[1:], len(ci)] if len(ci) else np.array([], int)
    last_day, eq_window = 0, []
    for s, e_ in zip(starts, ends):
        day = ci[s]
        eq_window = (eq_window + [equity] * (day - last_day))[-50:]   # equity carried over idle sessions
        last_day = day
        if equity <= 0:
            break
        if rk["eq_filter"] and len(eq_window) >= 50 and equity < np.mean(eq_window) * (1 - 1e-9):
            continue
        mult = 0.5 if rk["dd_throttle"] and equity < peak * (1 - rk["dd_throttle"]) else 1.0
        sod, taken, opened = equity, 0, []                         # opened: (exit_col, notional, pnl)
        for i in range(s, e_):
            if taken >= rk["max_trades"]:
                break
            if rk["daily_loss"] and sum(pn for xc, _, pn in opened if xc < ecol[i]) <= -rk["daily_loss"] * sod:
                break
            in_use = sum(nv for xc, nv, _ in opened if xc > ecol[i])
            q = math.floor(min(rk["risk"] * mult * sod / rps[i],
                               pos_cap * sod / ent[i],
                               max(rk["gross_lev"] * sod - in_use, 0.0) / ent[i]))
            if q <= 0:
                continue
            opened.append((xcol[i], q * ent[i], q * pps[i]))
            taken += 1
        day_pnl[day] = sum(pn for _, _, pn in opened)
        equity = sod + day_pnl[day]
        peak = max(peak, equity)
    eq = INITIAL_EQUITY + np.cumsum(day_pnl)
    prev = np.r_[INITIAL_EQUITY, eq[:-1]]
    return np.where(prev > 0, eq / prev - 1, 0.0)


# ── 7. Metrics ──
def summarize(r: np.ndarray, mask: np.ndarray) -> tuple[float, float, float]:
    """(Sharpe, CAGR, max drawdown) of daily returns r[mask]; fast path for the search."""
    x = r[mask]
    sd = x.std()
    g = np.r_[1.0, np.cumprod(1 + x)]
    sharpe = x.mean() / sd * np.sqrt(252) if sd > 0 else np.nan
    cagr = g[-1] ** (252 / len(x)) - 1 if g[-1] > 0 else -1.0
    return sharpe, cagr, (g / np.maximum.accumulate(g) - 1).min()


def perf_stats(r: pd.Series, bench: pd.Series, trades: pd.DataFrame | None = None) -> dict:
    """Full performance report for a daily return series (every session, zeros included)."""
    g = pd.concat([pd.Series([1.0], index=[r.index[0] - pd.Timedelta(days=1)]), (1 + r).cumprod()])
    dd = g / g.cummax() - 1
    trough = dd.idxmin()
    rec = g.loc[trough:][g.loc[trough:] >= g.loc[:trough].max()]
    monthly = (1 + r).resample("ME").prod() - 1
    yearly = (1 + r).resample("YE").prod() - 1
    beta = np.cov(r, bench)[0, 1] / bench.var() if bench.var() > 0 else np.nan
    down = np.sqrt(np.mean(np.minimum(r, 0) ** 2))
    cagr = g.iloc[-1] ** (252 / len(r)) - 1 if g.iloc[-1] > 0 else -1.0
    out = {
        "Total return %": 100 * (g.iloc[-1] - 1), "CAGR %": 100 * cagr,
        "Volatility %": 100 * r.std() * np.sqrt(252),
        "Sharpe": r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan,
        "Sortino": r.mean() / down * np.sqrt(252) if down > 0 else np.nan,
        "Max drawdown %": 100 * dd.min(), "Calmar": cagr / abs(dd.min()) if dd.min() < 0 else np.nan,
        "Recovery (sessions)": len(r.loc[trough:rec.index[0]]) - 1 if len(rec) else np.nan,
        "Worst day %": 100 * r.min(), "Worst month %": 100 * monthly.min(),
        "Best year %": 100 * yearly.max(), "Worst year %": 100 * yearly.min(),
        "Up months %": 100 * (monthly > 0).mean(),
        "Corr. to SPY": r.corr(bench), "Beta to SPY": beta,
        "Alpha % (ann.)": 100 * (r.mean() - beta * bench.mean()) * 252,
    }
    if trades is not None:
        out["Trades"] = len(trades)
        out["Hit ratio %"] = 100 * (trades["R"] > 0).mean() if len(trades) else np.nan
    return out


print(f"Engine ready: {len(panels)} assets, {len(CAL):,} sessions "
      f"({CAL[0].date()} → {CAL[-1].date()}), built in {time.perf_counter() - t0:.0f}s")

Engine ready: 11 assets, 5,969 sessions (2002-12-30 → 2026-09-21), built in 5s


## Strategy importing
Imports the strategy directly from the research paper: Zarattini, Barbon & Aziz (2024), SSRN 4729284.

In [5]:
# ═════════════════════════════════════════════════════════════════════════════
# ZARATTINI, BARBON & AZIZ (2024) — paper replication on the ETF universe
#
#   • Universe filter (ex-ante): open > $5, avg volume of previous 14 days ≥ 1M shares, ATR(14) > $0.50.
#   • Direction: sign of the first n-minute candle; doji → no trade. Stop-order entry at the OR high/low.
#   • Stop 10 % of ATR(14) from entry; otherwise exit at the close. No target.
#   • Sizing: each position gets equity / N slots; a stop-out loses 1 % of that slot; 4× leverage per slot.
#   • "Stocks in Play": Relative Volume ≥ 100 %, top 20 by Relative Volume. Windows 5/15/30/60; COMBO = equal weight.
#   • Costs as in the paper: $0.0035/share, zero slippage. A 1-tick-slippage row shows cost sensitivity.
# ═════════════════════════════════════════════════════════════════════════════
PAPER_PERIOD = (pd.Timestamp("2016-01-01"), pd.Timestamp("2023-12-31"))
PAPER_SLOTS = len(panels)
PAPER_SIZING = dict(risk=0.01 / PAPER_SLOTS, pos_cap=4.0 / PAPER_SLOTS, gross_lev=4.0, max_trades=99,
                    daily_loss=0.0, dd_throttle=0.0, eq_filter=False)


def paper_logic(n: int) -> Logic:
    return Logic(n, "candle", 0.0, "touch", N_BARS - 1, ("atr", 0.10), 0.0, 0.0, 0.0, N_BARS - 1)


PAPER_BASE = Filt(0.0, math.inf, 0.0, "any", False, "all", None, min_avg_vol=1_000_000, min_atr=0.50)
PAPER_RV = Filt(1.0, math.inf, 0.0, "any", False, "all", 20, min_avg_vol=1_000_000, min_atr=0.50)

paper_cands = {n: candidates(paper_logic(n), slip_ticks=0) for n in (5, 15, 30, 60)}
paper_trades = {"5m ORB Base": apply_filter(paper_cands[5], PAPER_BASE)}
for n in (5, 15, 30, 60):
    paper_trades[f"{n}m ORB + Rel Vol"] = apply_filter(paper_cands[n], PAPER_RV)
paper_rets = pd.DataFrame({k: run_risk(t, PAPER_SIZING) for k, t in paper_trades.items()}, index=CAL)
paper_rets["COMBO (5/15/30/60 equal-weight)"] = paper_rets[[f"{n}m ORB + Rel Vol" for n in (5, 15, 30, 60)]].mean(axis=1)
paper_rets[f"{BENCHMARK} buy & hold"] = R_SPY
slip_test = apply_filter(candidates(paper_logic(5), slip_ticks=1), PAPER_RV)

print("Paper filters — share of tradable sessions passing each filter, by data-source era:")
filt_rows = []
for s, p in panels.items():
    f, ok = FEAT[s], p["session_ok"]
    for era, m in (("pre-IEX", ok & (p["dates"] < IEX_SWITCH)), ("IEX-only", ok & (p["dates"] >= IEX_SWITCH))):
        if m.any():
            filt_rows.append({"asset": s, "era": era, "sessions": int(m.sum()),
                              "avg vol ≥ 1M": (f["avg_vol"][m] >= 1_000_000).mean(),
                              "ATR > $0.50": (f["atr"][m] > 0.50).mean()})
display(pd.DataFrame(filt_rows).set_index(["asset", "era"]).round(2).unstack("era"))

bench = pd.Series(R_SPY, index=CAL)
for label, (lo, hi) in (("Full sample", (CAL[0], CAL[-1])), ("Paper period 2016–2023", PAPER_PERIOD)):
    m = (CAL >= lo) & (CAL <= hi)
    table = {k: perf_stats(paper_rets.loc[m, k], bench[m],
                           paper_trades[k].loc[paper_trades[k]["date"].between(lo, hi)] if k in paper_trades else None)
             for k in paper_rets}
    table["5m + Rel Vol, 1-tick slippage"] = perf_stats(pd.Series(run_risk(slip_test, PAPER_SIZING), index=CAL)[m],
                                                        bench[m], slip_test.loc[slip_test["date"].between(lo, hi)])
    print(f"\n{label}: {lo.date()} → {hi.date()}")
    display(pd.DataFrame(table).T[["Total return %", "CAGR %", "Volatility %", "Sharpe", "Hit ratio %",
                                   "Max drawdown %", "Worst day %", "Alpha % (ann.)", "Beta to SPY", "Trades"]].round(2))

styles = {"5m ORB Base": dict(color="#555555", width=1.5), "5m ORB + Rel Vol": dict(color="#1f77b4", width=2.5),
          "15m ORB + Rel Vol": dict(color="#2ca02c", width=1.2), "30m ORB + Rel Vol": dict(color="#9467bd", width=1.2),
          "60m ORB + Rel Vol": dict(color="#8c564b", width=1.2),
          "COMBO (5/15/30/60 equal-weight)": dict(color="#000000", width=2, dash="dot"),
          f"{BENCHMARK} buy & hold": dict(color="#d62728", width=3)}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04,
                    subplot_titles=("Equity ($, log scale)", "Drawdown (%)"))
for name in paper_rets:
    eq = INITIAL_EQUITY * (1 + paper_rets[name]).cumprod()
    fig.add_trace(go.Scatter(x=CAL, y=eq, name=name, line=styles[name], legendgroup=name), 1, 1)
    fig.add_trace(go.Scatter(x=CAL, y=100 * (eq / eq.cummax() - 1), name=name, showlegend=False,
                             line={**styles[name], "width": 1}, legendgroup=name), 2, 1)
for row in (1, 2):
    fig.add_vrect(x0=PAPER_PERIOD[0], x1=PAPER_PERIOD[1], fillcolor="#f2c94c", opacity=0.10, line_width=0, row=row, col=1)
fig.add_vline(x=IEX_SWITCH, line_dash="dash", line_color="grey", row=1, col=1)
fig.add_annotation(x=IEX_SWITCH, y=1, yref="y domain", text="data → IEX-only", showarrow=False,
                   xanchor="left", font=dict(size=10, color="grey"), row=1, col=1)
fig.add_annotation(x=PAPER_PERIOD[0], y=1, yref="y domain", text="paper period 2016–2023", showarrow=False,
                   xanchor="left", font=dict(size=10), row=1, col=1)
fig.update_yaxes(type="log", tickformat="$,.0f", row=1, col=1)
fig.update_layout(title=(f"Zarattini, Barbon & Aziz (2024) n-minute ORB on {len(panels)} ETFs — benchmark: {BENCHMARK} buy & hold"
                         f"<br><sup>Stop 10 % ATR · EoD exit · 1 % risk per slot (equity / {PAPER_SLOTS}) · 4× leverage · "
                         f"${COMMISSION}/share · zero slippage (as in the paper)</sup>"),
                  template="plotly_white", height=820, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="left", x=0))
fig.show()

Paper filters — share of tradable sessions passing each filter, by data-source era:


sessions         avg vol ≥ 1M         ATR > $0.50        
era   IEX-only pre-IEX     IEX-only pre-IEX    IEX-only pre-IEX
asset                                                          
CPER       2.0    33.0         0.00    0.00        1.00    0.03
DIA      169.0  4614.0         0.01    1.00        1.00    1.00
GLD      458.0  3982.0         0.03    1.00        1.00    0.98
IEF       53.0  2147.0         0.02    0.82        0.62    0.28
IWM     1056.0  4370.0         0.16    1.00        1.00    1.00
QQQ     1078.0  4540.0         0.05    1.00        1.00    0.88
SHY        NaN  2536.0          NaN    0.73         NaN    0.00
SLV      322.0  3504.0         0.34    1.00        0.92    0.28
SPY     1095.0  4588.0         0.75    1.00        1.00    1.00
TLT     1003.0  3543.0         0.49    1.00        1.00    0.98
USO       78.0  3671.0         0.04    0.71        1.00    1.00


Full sample: 2002-12-30 → 2026-09-21


,Total return %,CAGR %,Volatility %,Sharpe,Hit ratio %,Max drawdown %,Worst day %,Alpha % (ann.),Beta to SPY,Trades
5m ORB Base,-34.94,-1.80,7.29,-0.21,16.90,-39.58,-1.81,-1.31,-0.02,26138.0
5m ORB + Rel Vol,-9.47,-0.42,4.27,-0.08,15.87,-19.56,-1.28,-0.20,-0.01,10002.0
15m ORB + Rel Vol,-23.89,-1.15,4.09,-0.26,16.47,-29.07,-1.57,-1.04,-0.00,9893.0
30m ORB + Rel Vol,-16.82,-0.77,3.72,-0.19,17.12,-27.02,-0.71,-0.67,-0.00,9894.0
60m ORB + Rel Vol,-17.42,-0.80,3.52,-0.21,19.13,-28.42,-0.72,-0.70,-0.00,9078.0
COMBO (5/15/30/60 equal-weight),-16.30,-0.75,2.75,-0.26,NaN,-21.30,-0.89,-0.65,-0.01,NaN
SPY buy & hold,1158.97,11.29,18.51,0.67,NaN,-55.20,-11.63,-0.00,1.00,NaN
"5m + Rel Vol, 1-tick slippage",-64.89,-4.32,3.96,-1.10,14.29,-66.46,-1.26,-4.26,-0.01,10002.0



Paper period 2016–2023: 2016-01-01 → 2023-12-31


,Total return %,CAGR %,Volatility %,Sharpe,Hit ratio %,Max drawdown %,Worst day %,Alpha % (ann.),Beta to SPY,Trades
5m ORB Base,-8.99,-1.17,6.44,-0.15,17.63,-21.63,-1.81,-0.90,-0.01,9758.0
5m ORB + Rel Vol,-0.68,-0.08,4.09,-0.00,16.45,-12.39,-1.28,-0.11,0.01,3690.0
15m ORB + Rel Vol,0.66,0.08,4.08,0.04,17.80,-15.78,-1.57,0.01,0.01,3568.0
30m ORB + Rel Vol,3.02,0.37,3.52,0.12,18.38,-8.85,-0.52,0.35,0.01,3514.0
60m ORB + Rel Vol,-2.73,-0.35,3.22,-0.09,19.60,-9.95,-0.55,-0.39,0.01,3239.0
COMBO (5/15/30/60 equal-weight),0.34,0.04,2.57,0.03,NaN,-7.08,-0.89,-0.03,0.01,NaN
SPY buy & hold,152.36,12.29,18.65,0.72,NaN,-33.85,-11.63,-0.00,1.00,NaN
"5m + Rel Vol, 1-tick slippage",-21.58,-3.00,3.93,-0.76,15.20,-28.19,-1.26,-3.10,0.01,3690.0


## Bootstrapping
This will help determine what the best strategy variation and parameters are.

In [6]:
# ═════════════════════════════════════════════════════════════════════════════
# ORB VARIATION SEARCH — 13,000+ configurations + risk-management overlays
#
# Research basis for the search space (verify details against the sources):
#   • Zarattini, Barbon & Aziz (2024); Zarattini & Aziz (2023): 5-min ORB, ATR stops, EoD exit, Rel Vol.
#   • Tsai et al. (2019): short opening ranges work best on index products.
#   • Crabel (1990) / Fisher (2002, ACD): entry buffer beyond the range; NR7 days precede expansion.
#   • Holmberg et al. (2013); Lundström (2017): ORB profits rise with volatility.
#   • Wu et al. (2021): profit targets hurt; protective / trailing stops help.
#   • Gao, Han, Li & Zhou (2018, JFE) "Market intraday momentum": time-of-day effects.
#   • Practitioner risk controls: fixed-fractional risk, leverage cap, max trades/day, daily loss limit,
#     drawdown throttle, equity-curve filter.
#
# Protocol: selection uses 2003–2015 only (2016+ held out). Stage 1 = ~12,000 signal × exit × filter configs
# (fast fractional sizing, 4× gross cap). Stage 2 = ~1,400 risk overlays on the top 10 (integer shares).
# Reports: Deflated Sharpe, IS→OOS rank correlation, marginal effects, and walk-forward re-selection
# (pick the best config on all prior years, trade it the next year). Runtime ≈ 3 min; QUICK = True ≈ 30 s.
# ═════════════════════════════════════════════════════════════════════════════
QUICK = False
SEED = 7
N_LOGIC = 60 if QUICK else 500
FILTERS_PER_LOGIC = 6 if QUICK else 24
STAGE2_TOP_K = 3 if QUICK else 10
IS_END = pd.Timestamp("2015-12-31")
IS_MASK = CAL <= IS_END
MIN_IS_TRADES = 300
WF_MIN_TRAIN_YEARS = 5
WF_MIN_TRADES_PER_YEAR = 25
MAX_IS_DD = 0.25
STAGE1_SIZING = dict(risk=0.0025, pos_cap=1.0, gross_cap=4.0)

LOGIC_GRID = {
    "n":         [5, 10, 15, 30, 60],
    "dir_mode":  ["candle", "both", "candle+gap", "candle+trend"],
    "buffer":    [0.0, 0.02, 0.05],
    "confirm":   ["touch", "close"],
    "cutoff":    [60, 120, 389],                 # no new entries after 10:30 / 11:30 / close
    "stop":      [("atr", 0.05), ("atr", 0.10), ("atr", 0.15), ("atr", 0.25), ("atr", 0.50),
                  ("or_opp", 0.0), ("or_mid", 0.0)],
    "be_R":      [0.0, 1.0, 2.0],
    "trail_atr": [0.0, 0.25, 0.50],
    "target_R":  [0.0, 2.0, 5.0, 10.0],
    "exit_col":  [150, 270, 389],                # close of the 12:00 / 14:00 / 15:59 bar
}
FILTER_GRID = {
    "rv_min":     [0.0, 1.0, 1.5, 2.0, 3.0],
    "or_atr_max": [math.inf, 0.5, 0.3],
    "gap_min":    [0.0, 0.3],
    "regime":     ["any", "expanding", "contracting"],
    "nr7":        [False, True],
    "universe":   ["all", "equity"],
    "top_n":      [None, 3],
}
RISK_GRID = {
    "risk":        [0.0025, 0.005, 0.01],
    "gross_lev":   [2.0, 4.0],
    "max_trades":  [1, 3, 99],
    "daily_loss":  [0.0, 0.01],
    "dd_throttle": [0.0, 0.15],
    "eq_filter":   [False, True],
}
LOGIC_COLS = list(LOGIC_GRID)
FILTER_COLS = list(FILTER_GRID)

# ── Stage 1 ──────────────────────────────────────────────────────────────────
t_start = time.perf_counter()
rng = np.random.default_rng(SEED)
all_logics = [lg for lg in (Logic(*c) for c in itertools.product(*LOGIC_GRID.values())) if lg.valid()]
all_filters = [Filt(*c) for c in itertools.product(*FILTER_GRID.values())]
logic_pick = [all_logics[i] for i in rng.choice(len(all_logics), N_LOGIC, replace=False)]
print(f"Search space: {len(all_logics):,} logics × {len(all_filters):,} filters "
      f"→ sampling {N_LOGIC * FILTERS_PER_LOGIC:,} Stage-1 configs")

rows_out, year_stats = [], []                 # year_stats: per config (Σr, Σr², trades) per year
for li, lg in enumerate(logic_pick):
    cands = candidates(lg)
    if cands.empty:
        continue
    for fl in [all_filters[i] for i in rng.choice(len(all_filters), FILTERS_PER_LOGIC, replace=False)]:
        t = apply_filter(cands, fl)
        r = fixed_risk_returns(t, **STAGE1_SIZING)
        is_t = int((t["date"] <= IS_END).sum())
        s_is, s_oos = summarize(r, IS_MASK), summarize(r, ~IS_MASK)
        rows_out.append({**asdict(lg), "stop": f"{lg.stop[0]}:{lg.stop[1]}",
                         **{k: getattr(fl, k) for k in FILTER_COLS},
                         "is_trades": is_t, "oos_trades": len(t) - is_t,
                         "is_avgR": t.loc[t["date"] <= IS_END, "R"].mean(),
                         "oos_avgR": t.loc[t["date"] > IS_END, "R"].mean(),
                         "is_sharpe": s_is[0], "is_cagr": s_is[1], "is_mdd": s_is[2],
                         "oos_sharpe": s_oos[0], "oos_cagr": s_oos[1], "oos_mdd": s_oos[2],
                         "is_daily_sr": r[IS_MASK].mean() / r[IS_MASK].std() if r[IS_MASK].std() > 0 else np.nan,
                         "_logic": lg, "_filt": fl})
        year_stats.append(np.stack([np.bincount(YIDX, weights=r, minlength=len(YEAR_IDS)),
                                    np.bincount(YIDX, weights=r * r, minlength=len(YEAR_IDS)),
                                    np.bincount(YIDX[t["cal_idx"].to_numpy()], minlength=len(YEAR_IDS))]))
    if (li + 1) % max(1, N_LOGIC // 5) == 0:
        print(f"  {li + 1}/{N_LOGIC} logics  ({time.perf_counter() - t_start:.0f}s)")

res = pd.DataFrame(rows_out)
YS = np.stack(year_stats)
elig = res.loc[(res["is_trades"] >= MIN_IS_TRADES) & res["is_sharpe"].notna()]
if elig.empty:
    raise RuntimeError("No eligible configurations — lower MIN_IS_TRADES.")
rho, p_rho = stats.spearmanr(elig["is_sharpe"], elig["oos_sharpe"], nan_policy="omit")
top_dec = elig.nlargest(max(1, len(elig) // 10), "is_sharpe")
print(f"\nStage 1: {len(res):,} configs, {len(elig):,} with ≥ {MIN_IS_TRADES} IS trades. "
      f"IS→OOS rank correlation ρ = {rho:+.3f}; median OOS Sharpe: all {elig['oos_sharpe'].median():+.2f}, "
      f"top IS decile {top_dec['oos_sharpe'].median():+.2f}; OOS Sharpe > 0 for {100 * (elig['oos_sharpe'] > 0).mean():.0f} %")

show = LOGIC_COLS + FILTER_COLS + ["is_trades", "is_avgR", "is_sharpe", "oos_trades", "oos_avgR", "oos_sharpe",
                                   "oos_cagr", "oos_mdd"]
print("\nTop 15 Stage-1 configs by IN-SAMPLE Sharpe (selection never sees OOS columns):")
display(elig.nlargest(15, "is_sharpe")[show].round(3))

print("\nMarginal effects — median Sharpe of eligible configs per parameter value (OOS = 2016+):")
marg = []
for col in LOGIC_COLS + FILTER_COLS:
    key = elig[col].astype(str)
    g = elig.groupby(key)[["is_sharpe", "oos_sharpe"]].median()
    g["configs"] = key.value_counts()
    marg.append(g.assign(param=col).reset_index(names="value"))
display(pd.concat(marg).set_index(["param", "value"]).round(3))

# ── Stage 2: risk overlays ──────────────────────────────────────────────────
risk_grid = [dict(zip(RISK_GRID, c)) for c in itertools.product(*RISK_GRID.values())]
stage2_seeds = elig.sort_values("is_sharpe", ascending=False).drop_duplicates(subset=LOGIC_COLS).head(STAGE2_TOP_K)
s2_rows, s2_returns = [], {}
for sid, row in stage2_seeds.iterrows():
    trades = apply_filter(candidates(row["_logic"]), row["_filt"])
    for ri, rk in enumerate(risk_grid):
        r = run_risk(trades, rk)
        s_is, s_oos = summarize(r, IS_MASK), summarize(r, ~IS_MASK)
        s2_rows.append({"seed": sid, "ri": ri, **rk, "is_sharpe": s_is[0], "is_cagr": s_is[1], "is_mdd": s_is[2],
                        "is_calmar": s_is[1] / abs(s_is[2]) if s_is[2] < 0 else np.nan,
                        "oos_sharpe": s_oos[0], "oos_cagr": s_oos[1], "oos_mdd": s_oos[2],
                        "is_daily_sr": r[IS_MASK].mean() / r[IS_MASK].std() if r[IS_MASK].std() > 0 else np.nan})
        s2_returns[(sid, ri)] = r
s2 = pd.DataFrame(s2_rows)
s2_ok = s2.loc[s2["is_mdd"] >= -MAX_IS_DD].dropna(subset=["is_calmar"])
s2_ok = s2_ok if not s2_ok.empty else s2.dropna(subset=["is_calmar"])
best2 = s2_ok.loc[s2_ok["is_calmar"].idxmax()]
print(f"\nStage 2: {len(stage2_seeds)} configs × {len(risk_grid)} risk protocols = {len(s2):,} runs. "
      f"Selection: highest IS Calmar with IS MDD ≤ {MAX_IS_DD:.0%}. Top 10:")
display(s2_ok.nlargest(10, "is_calmar").drop(columns=["is_daily_sr", "ri"]).round(3))
print("Marginal effect of each risk control (median over Stage-2 runs):")
display(pd.concat([s2.groupby(c)[["is_sharpe", "is_mdd", "oos_sharpe", "oos_mdd"]].median().assign(param=c)
                   .reset_index(names="value") for c in RISK_GRID]).set_index(["param", "value"]).round(3))


# ── Honest estimates: Deflated Sharpe & walk-forward re-selection ───────────
def deflated_sharpe(r: np.ndarray, trial_srs: np.ndarray, n_trials: int) -> dict:
    sr = r.mean() / r.std()
    g = 0.5772156649
    sr0 = np.sqrt(np.nanvar(trial_srs, ddof=1)) * ((1 - g) * stats.norm.ppf(1 - 1 / n_trials)
                                                    + g * stats.norm.ppf(1 - 1 / (n_trials * np.e)))
    den = np.sqrt(max(1 - stats.skew(r) * sr + (stats.kurtosis(r, fisher=False) - 1) / 4 * sr ** 2, 1e-12))
    return {"DSR": stats.norm.cdf((sr - sr0) * np.sqrt(len(r) - 1) / den),
            "PSR (SR > 0)": stats.norm.cdf(sr * np.sqrt(len(r) - 1) / den), "trials": n_trials}


best1 = elig.loc[elig["is_sharpe"].idxmax()]
r_best1 = fixed_risk_returns(apply_filter(candidates(best1["_logic"]), best1["_filt"]), **STAGE1_SIZING)
r_best2 = s2_returns[(best2["seed"], int(best2["ri"]))]
n_trials = len(res) + len(s2)
trial_srs = np.r_[elig["is_daily_sr"].to_numpy(), s2["is_daily_sr"].to_numpy()]   # configs with meaningful Sharpe
print(f"\nDeflated Sharpe Ratio of the IS winners after {n_trials:,} trials "
      "(conservative here: much of the dispersion between configs is real, not noise):")
display(pd.DataFrame({"Stage-1 winner": deflated_sharpe(r_best1[IS_MASK], trial_srs, n_trials),
                      "Stage-2 winner": deflated_sharpe(r_best2[IS_MASK], trial_srs, n_trials)}).T)

wf_ret = np.zeros(len(CAL))
wf_log, wf_cache = [], {}
for yi, yr in enumerate(YEAR_IDS):
    if yi < WF_MIN_TRAIN_YEARS:
        continue
    s1_, s2_, ntr = YS[:, 0, :yi].sum(1), YS[:, 1, :yi].sum(1), YS[:, 2, :yi].sum(1)
    ndays = (YIDX < yi).sum()
    mu = s1_ / ndays
    score = np.where(ntr >= WF_MIN_TRADES_PER_YEAR * yi, mu / np.sqrt(np.maximum(s2_ / ndays - mu ** 2, 1e-18)), -np.inf)
    k = int(np.argmax(score))
    if k not in wf_cache:
        wf_cache[k] = fixed_risk_returns(apply_filter(candidates(res.iloc[k]["_logic"]), res.iloc[k]["_filt"]),
                                         **STAGE1_SIZING)
    m = YEARS == yr
    wf_ret[m] = wf_cache[k][m]
    wf_log.append({"trade_year": yr, "chosen_config": k, "train_sharpe": score[k] * np.sqrt(252),
                   "year_return_%": 100 * (np.prod(1 + wf_cache[k][m]) - 1)})
wf_log = pd.DataFrame(wf_log)
WF_MASK = np.isin(YEARS, wf_log["trade_year"])
print("\nWalk-forward re-selection log (config chosen using only prior years):")
display(wf_log.round(3))

series = {"Stage-1 IS winner": (r_best1, np.ones(len(CAL), bool)),
          "Stage-2 IS winner (with risk mgmt)": (r_best2, np.ones(len(CAL), bool)),
          "Walk-forward re-selection": (wf_ret, WF_MASK),
          f"{BENCHMARK} buy & hold": (R_SPY, np.ones(len(CAL), bool))}
summary = {}
for name, (r, active) in series.items():
    for label, mk in (("IS ≤ 2015", IS_MASK & active), ("OOS 2016+", ~IS_MASK & active)):
        if mk.any():
            sh, cg, md = summarize(r, mk)
            summary[(name, label)] = {"Sharpe": sh, "CAGR %": 100 * cg, "MDD %": 100 * md}
print("\nHeadline comparison (walk-forward rows only count the years it actually traded):")
display(pd.DataFrame(summary).T.round(2))
print("Stage-1 IS winner:", best1["_logic"], best1["_filt"], sep="\n  ")
print("Stage-2 risk protocol:", {k: (best2[k].item() if hasattr(best2[k], "item") else best2[k]) for k in RISK_GRID})
print(f"Search runtime {time.perf_counter() - t_start:.0f}s")

colors = {"Stage-1 IS winner": "#1f77b4", "Stage-2 IS winner (with risk mgmt)": "#2ca02c",
          "Walk-forward re-selection": "#9467bd", f"{BENCHMARK} buy & hold": "#d62728"}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04,
                    subplot_titles=("Equity ($, log scale)", "Drawdown (%)"))
for name, (r, _) in series.items():
    eq = INITIAL_EQUITY * np.cumprod(1 + r)
    w = 3 if name.startswith(BENCHMARK) else 2
    fig.add_trace(go.Scatter(x=CAL, y=eq, name=name, line=dict(color=colors[name], width=w), legendgroup=name), 1, 1)
    fig.add_trace(go.Scatter(x=CAL, y=100 * (eq / np.maximum.accumulate(eq) - 1), name=name, showlegend=False,
                             line=dict(color=colors[name], width=1), legendgroup=name), 2, 1)
for row in (1, 2):
    fig.add_vrect(x0=IS_END, x1=CAL[-1], fillcolor="grey", opacity=0.08, line_width=0, row=row, col=1)
fig.add_annotation(x=IS_END, y=1, yref="y domain", text="  out-of-sample (2016+)", showarrow=False,
                   xanchor="left", row=1, col=1)
fig.update_yaxes(type="log", tickformat="$,.0f", row=1, col=1)
fig.update_layout(title=f"ORB variation search — {n_trials:,} configurations, selected on 2003–2015 only",
                  template="plotly_white", height=820, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="left", x=0))
fig.show()

fig2 = go.Figure(go.Scattergl(x=elig["is_sharpe"], y=elig["oos_sharpe"], mode="markers",
                              marker=dict(size=4, opacity=0.35, color=elig["n"], colorscale="Viridis",
                                          colorbar=dict(title="OR min")),
                              text=elig["dir_mode"] + " · " + elig["stop"], name="configs"))
lim = [np.nanmin(elig[["is_sharpe", "oos_sharpe"]].to_numpy()), np.nanmax(elig[["is_sharpe", "oos_sharpe"]].to_numpy())]
fig2.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(dash="dash", color="grey"), name="IS = OOS"))
fig2.update_layout(title=f"Overfitting map — IS vs OOS Sharpe for {len(elig):,} configs (ρ = {rho:+.2f})",
                   xaxis_title="In-sample Sharpe (2003–2015)", yaxis_title="Out-of-sample Sharpe (2016+)",
                   template="plotly_white", height=520)
fig2.show()

Search space: 141,120 logics × 720 filters → sampling 12,000 Stage-1 configs
  100/500 logics  (30s)
  200/500 logics  (60s)
  300/500 logics  (93s)
  400/500 logics  (128s)
  500/500 logics  (160s)

Stage 1: 12,000 configs, 4,221 with ≥ 300 IS trades. IS→OOS rank correlation ρ = +0.611; median OOS Sharpe: all -0.09, top IS decile +0.35; OOS Sharpe > 0 for 44 %

Top 15 Stage-1 configs by IN-SAMPLE Sharpe (selection never sees OOS columns):


,n,dir_mode,buffer,confirm,cutoff,stop,be_R,trail_atr,target_R,exit_col,rv_min,or_atr_max,gap_min,regime,nr7,universe,top_n,is_trades,is_avgR,is_sharpe,oos_trades,oos_avgR,oos_sharpe,oos_cagr,oos_mdd
9665,10,candle+gap,0.00,close,60,atr:0.5,0.0,0.5,0.0,389,0.0,0.5,0.3,any,False,all,NaN,3044,0.084,1.161,3393,0.009,0.305,0.013,-0.089
1093,15,candle+gap,0.00,close,60,atr:0.5,0.0,0.0,10.0,389,1.0,0.5,0.3,expanding,False,all,3.0,629,0.222,1.129,677,-0.022,-0.067,-0.002,-0.070
9323,15,candle+gap,0.02,close,120,or_mid:0.0,2.0,0.0,0.0,270,0.0,0.5,0.3,any,False,equity,NaN,1425,0.211,1.081,1419,0.243,1.183,0.068,-0.061
4702,15,candle+gap,0.05,touch,60,or_opp:0.0,1.0,0.0,0.0,389,0.0,0.5,0.3,expanding,False,equity,3.0,535,0.328,1.014,586,0.143,0.668,0.019,-0.064
4530,10,candle+gap,0.05,touch,120,atr:0.25,0.0,0.5,0.0,389,0.0,0.3,0.3,any,False,all,NaN,2730,0.131,0.984,2804,0.020,0.281,0.016,-0.142
1081,15,candle+gap,0.00,close,60,atr:0.5,0.0,0.0,10.0,389,1.5,inf,0.3,any,False,all,NaN,669,0.173,0.984,815,0.021,0.158,0.005,-0.051
1914,15,candle+gap,0.02,close,120,or_mid:0.0,1.0,0.0,5.0,270,0.0,0.3,0.3,any,False,equity,3.0,880,0.199,0.975,830,0.264,1.223,0.045,-0.038
2032,15,candle+gap,0.02,touch,389,atr:0.5,1.0,0.0,0.0,389,1.0,0.5,0.3,any,False,equity,NaN,1049,0.161,0.972,927,0.071,0.476,0.015,-0.054
2449,30,candle+gap,0.00,close,120,atr:0.25,0.0,0.0,5.0,389,1.0,0.3,0.3,any,False,all,NaN,585,0.242,0.971,531,0.024,0.075,0.001,-0.051
4698,15,candle+gap,0.05,touch,60,or_opp:0.0,1.0,0.0,0.0,389,1.0,0.5,0.3,any,False,all,NaN,1490,0.164,0.961,1585,0.021,0.248,0.010,-0.122



Marginal effects — median Sharpe of eligible configs per parameter value (OOS = 2016+):


is_sharpe  oos_sharpe  configs
param      value                                       
n          10               -0.564      -0.205      921
           15               -0.338       0.054     1007
           30               -0.414       0.008      763
           5                -0.636      -0.268     1068
           60               -0.421      -0.054      462
dir_mode   both             -0.772      -0.231     1186
           candle           -0.497      -0.102     1289
           candle+gap       -0.111       0.160      828
           candle+trend     -0.503      -0.215      918
buffer     0.0              -0.450      -0.112     1415
           0.02             -0.485      -0.091     1318
           0.05             -0.536      -0.064     1488
confirm    close            -0.302       0.061     1952
           touch            -0.684      -0.244     2269
cutoff     120              -0.440      -0.051     1462
           389              -0.484      -0.113     1519
           60               -0.565      -0.111     1240
stop       atr:0.05         -2.438      -1.113      612
           atr:0.1          -1.020      -0.290      611
           atr:0.15         -0.508      -0.077      622
           atr:0.25         -0.194       0.119      634
           atr:0.5          -0.197       0.074      616
           or_mid:0.0       -0.305       0.011      583
           or_opp:0.0       -0.245       0.087      543
be_R       0.0              -0.511      -0.079     2569
           1.0              -0.467      -0.133      876
           2.0              -0.416      -0.096      776
trail_atr  0.0              -0.402      -0.078     2396
           0.25             -0.775      -0.227      887
           0.5              -0.402      -0.016      938
target_R   0.0              -0.361       0.024     1224
           10.0             -0.563      -0.140     1014
           2.0              -0.703      -0.254      927
           5.0              -0.419      -0.062     1056
exit_col   150              -0.668      -0.164     1337
           270              -0.444      -0.085     1485
           389              -0.349      -0.027     1399
rv_min     0.0              -0.458      -0.026     1977
           1.0              -0.507      -0.068     1289
           1.5              -0.568      -0.174      657
           2.0              -0.432      -0.294      291
           3.0              -0.436      -1.090        7
or_atr_max 0.3              -0.423      -0.126     1113
           0.5              -0.510      -0.111     1500
           inf              -0.509      -0.043     1608
gap_min    0.0              -0.653      -0.103     2378
           0.3              -0.281      -0.078     1843
regime     any              -0.505      -0.054     1713
           contracting      -0.495      -0.098     1339
           expanding        -0.452      -0.125     1169
nr7        False            -0.613      -0.120     3306
           True             -0.258      -0.005      915
universe   all              -0.501      -0.318     2524
           equity           -0.468       0.295     1697
top_n      3.0              -0.487      -0.085     2129
           nan              -0.483      -0.100     2092


Stage 2: 10 configs × 144 risk protocols = 1,440 runs. Selection: highest IS Calmar with IS MDD ≤ 25%. Top 10:


,seed,risk,gross_lev,max_trades,daily_loss,dd_throttle,eq_filter,is_sharpe,is_cagr,is_mdd,is_calmar,oos_sharpe,oos_cagr,oos_mdd
321,9323,0.002,4.0,3,0.00,0.00,True,1.206,0.045,-0.034,1.322,0.442,0.012,-0.039
323,9323,0.002,4.0,3,0.00,0.15,True,1.206,0.045,-0.034,1.322,0.442,0.012,-0.039
325,9323,0.002,4.0,3,0.01,0.00,True,1.206,0.045,-0.034,1.322,0.442,0.012,-0.039
327,9323,0.002,4.0,3,0.01,0.15,True,1.206,0.045,-0.034,1.322,0.442,0.012,-0.039
329,9323,0.002,4.0,99,0.00,0.00,True,1.172,0.046,-0.036,1.256,0.448,0.013,-0.039
331,9323,0.002,4.0,99,0.00,0.15,True,1.172,0.046,-0.036,1.256,0.448,0.013,-0.039
333,9323,0.002,4.0,99,0.01,0.00,True,1.172,0.046,-0.036,1.256,0.448,0.013,-0.039
335,9323,0.002,4.0,99,0.01,0.15,True,1.172,0.046,-0.036,1.256,0.448,0.013,-0.039
377,9323,0.005,4.0,99,0.00,0.00,True,0.991,0.063,-0.054,1.165,0.324,0.014,-0.079
379,9323,0.005,4.0,99,0.00,0.15,True,0.991,0.063,-0.054,1.165,0.324,0.014,-0.079


... (5 outputs left out)

,trade_year,chosen_config,train_sharpe,year_return_%
0,2007,40,0.773,-0.486
1,2008,1093,1.086,2.082
2,2009,9665,1.213,7.411
3,2010,9665,1.256,3.758
4,2011,9665,1.149,7.198
5,2012,9665,1.181,-1.525
6,2013,1093,1.157,1.347
7,2014,1093,1.094,6.909
8,2015,1093,1.229,-0.087
9,2016,9665,1.161,-5.287



Headline comparison (walk-forward rows only count the years it actually traded):


Sharpe  CAGR %  MDD %
Stage-1 IS winner                  IS ≤ 2015    1.16    5.30  -6.09
                                   OOS 2016+    0.31    1.35  -8.86
Stage-2 IS winner (with risk mgmt) IS ≤ 2015    1.21    4.49  -3.40
                                   OOS 2016+    0.44    1.22  -3.89
Walk-forward re-selection          IS ≤ 2015    0.75    2.91  -6.09
                                   OOS 2016+    1.03    5.77  -6.85
SPY buy & hold                     IS ≤ 2015    0.54    8.77 -55.20
                                   OOS 2016+    0.85   14.42 -33.85

Stage-1 IS winner:
  Logic(n=10, dir_mode='candle+gap', buffer=0.0, confirm='close', cutoff=60, stop=('atr', 0.5), be_R=0.0, trail_atr=0.5, target_R=0.0, exit_col=389)
  Filt(rv_min=0.0, or_atr_max=0.5, gap_min=0.3, regime='any', nr7=False, universe='all', top_n=None, min_avg_vol=0.0, min_atr=0.0)
Stage-2 risk protocol: {'risk': 0.0025, 'gross_lev': 4.0, 'max_trades': 3, 'daily_loss': 0.0, 'dd_throttle': 0.0, 'eq_filter': True}
Search runtime 174s


## The winning strategy: 15-minute gap-confirmed ORB on index ETFs
***

### In one sentence
When an index ETF gaps at the open **and** its first 15 minutes move the same way, buy (or short) a confirmed breakout of that 15-minute range. Risk half the range, move the stop to breakeven at +2R, and be flat by 14:00.

### Why it should work (the logic)
- **Gap + opening candle agree = one-sided order flow.** An overnight gap shows news and positioning. When the first 15 minutes keep pushing the same way, the imbalance is still active rather than fading.
- **A narrow opening range = compression before expansion.** A tight first 15 minutes (≤ 0.5 × ATR) leaves room for the day's move to extend, and keeps the stop small relative to the potential move.
- **Bar-close confirmation filters false breakouts.** A wick through the level isn't enough; a 1-minute bar has to *close* beyond it.
- **The edge comes from trend days.** The analysis showed trades average **+1.09R on strong trend days** and **−0.43R on quiet range days**. The rules are built so many small losses on range days are paid for by a few large trend-day wins. Win rate is about 42 %, average win +1.79R, average loss −0.92R.
- *These are hypotheses consistent with the results, not proven mechanisms.*

### Universe & eligibility (checked before 09:30)
| Rule | Value |
|---|---|
| Instruments | SPY, QQQ, IWM, DIA |
| Price | Previous close > $5 |
| Data quality | First bar by 09:32, ≥ 80 % of minutes present in the session and in the opening range |
| ATR | 14-day Wilder ATR, measured on **prior** days only |

### Entry rules (each ETF, each day)
1. **Opening range (OR):** the high and low of the 09:30–09:44 bars (15 minutes).
2. **Direction:** bullish if the 09:44 close is above the 09:30 open, bearish if below. A flat candle means no trade.
3. **Gap filter:** today's open vs yesterday's close must point the **same direction** as the OR candle, and |gap| ≥ **0.3 × ATR**.
4. **OR width filter:** OR high − OR low ≤ **0.5 × ATR**.
5. **Trigger:** long = OR high + **0.02 × ATR** (rounded up to the cent). Short = OR low − 0.02 × ATR (rounded down).
6. **Entry:** the first 1-minute bar from 09:45 that **closes** beyond the trigger. Enter at that close (plus 1 tick slippage).
7. **Entry window:** no new entries after the **11:30** bar.
8. **One trade per ETF per day.** No re-entries, and only the direction set by steps 2–3.

### Exit rules
| Exit | Rule |
|---|---|
| Initial stop | **OR midpoint**, rounded away from entry. Risk per share (1R) = entry − stop. |
| Breakeven | Once price has traded **+2R** beyond entry (checked on completed bars), move the stop to entry. |
| Profit target | **None.** Winners run until the stop or the time exit. |
| Time exit | Flat at the close of the **14:00** bar. Never held overnight. |
| Stop fills | At the stop price; at the bar's open if it gaps through the stop; 1 tick slippage. |
| Ties | If a bar hits both the stop and a better price, the stop is assumed to fill first (conservative). |

### Risk management & position sizing
| Control | Setting |
|---|---|
| Risk per trade | **1 % of start-of-day equity** (realistic version); 0.25 % in the search |
| Shares | floor(1 % × equity ÷ 1R per share), capped by available leverage |
| Gross leverage cap | **4×** equity across all open positions (FINRA intraday limit) |
| Max positions | 4 (one per ETF) |
| Overnight risk | None; every position is closed by 14:01 |
| Costs modelled | $0.0035 / share / side commission + 1 tick slippage on every stop/market fill |

### Worked example (illustrative numbers)
- **Setup:** SPY ATR = $6.00. Yesterday's close was $500.00 and today's open is $502.00, a gap of +$2.00 = **0.33 ATR ✓**.
- **Opening range:** 09:30–09:44 opens at $502.00, high $503.20, low $501.80, close $502.90. The candle is bullish and agrees with the gap ✓. The width is $1.40 = **0.23 ATR ✓**.
- **Trigger:** $503.20 + 0.02 × $6 = **$503.32**.
- **Entry:** the 10:05 bar closes at $503.50, so you buy at **$503.51**.
- **Stop:** the OR midpoint, **$502.50**, so 1R = **$1.01** per share.
- **Size:** 1 % of $25,000 = $250 risk, which would mean 247 shares ($124k notional). The **4× cap** limits it to $100k, or **198 shares**, so the actual risk is about $200.
- **Breakeven:** once SPY trades at **$505.53** (+2R), the stop moves to $503.51.
- **Exit:** sell at the 14:00 bar's close, unless stopped out first.

### Performance (live / out-of-sample, 2017 → Sep 2026)
| Metric | ORB @ 1 % risk | SPY buy & hold |
|---|---|---|
| Mean per trade | +0.255 R (t = 5.26) | — |
| CAGR | 21.3 % | 14.7 % |
| Sharpe | 1.33 | 0.84 |
| Max drawdown | −11.9 % | −33.9 % |
| Correlation to SPY | −0.05 | 1.00 |

### Where it works best and worst
- **Best:** high-volatility markets (SPY 20-day vol > 20 %), SPY below its 200-day average, large gaps (> 1 ATR), SPY & QQQ.
- **Worst:** calm, rising markets (the most common condition, with the thinnest edge), late entries (10:30–11:30), wider opening ranges (0.35–0.5 ATR), and DIA.

### Known risks & caveats
- **Selection bias:** chosen as the best of ~13,440 configurations. Only the 2017+ results are a fair test.
- **Leverage:** at 1 % risk the 4× cap often binds. A $25,000 account sits at the US pattern-day-trader minimum, so a small drawdown could restrict day trading.
- **Execution:** the edge per trade is modest. Slippage worse than 1 tick, or slow fills on bar-close entries, would erode it.
- **Recent drawdowns:** two of the three worst drawdowns (in R) occurred in 2023–2026. That could be a low-volatility patch or the start of decay; monitor it.
- **Not proven to beat SPY:** the edge is statistically real, but its outperformance vs SPY is not statistically significant (Sharpe-difference p = 0.29).

In [7]:
# ═════════════════════════════════════════════════════════════════════════════
# WINNING ORB CONFIG — WHEN DOES IT WORK, AND WHEN DOES IT FAIL?
# Every trade is tagged with market conditions. Ex-ante conditions are known before entry (usable as
# filters); the EX-POST condition describes the day after the fact (explains results, not tradable).
# Results are in R (risk units, after costs).
# ═════════════════════════════════════════════════════════════════════════════
k_final = int(wf_log["chosen_config"].iloc[-1])
streak = wf_log["chosen_config"].iloc[::-1].eq(k_final).cumprod().astype(bool)
LIVE_START = pd.Timestamp(year=int(wf_log["trade_year"].iloc[::-1][streak].min()), month=1, day=1)
WIN_LOGIC, WIN_FILT = res.iloc[k_final]["_logic"], res.iloc[k_final]["_filt"]
win_trades = apply_filter(candidates(WIN_LOGIC), WIN_FILT).sort_values(["date", "entry_col"]).copy()
MIN_BUCKET_TRADES = 30
print(f"Winner (config #{k_final}, live from {LIVE_START.year}):\n  {WIN_LOGIC}\n  {WIN_FILT}")

# Market context from SPY (ex-ante: shifted so it is known at today's open)
spy = panels["SPY"]["daily"]
spy_ret = spy["close"].pct_change()
spy_tr = pd.concat([spy["high"] - spy["low"], (spy["high"] - spy["close"].shift(1)).abs(),
                    (spy["low"] - spy["close"].shift(1)).abs()], axis=1).max(axis=1)
ctx = pd.DataFrame({
    "spy_vol20": (spy_ret.rolling(20, min_periods=20).std() * np.sqrt(252)).shift(1),
    "spy_above_200d": (spy["close"] > spy["close"].rolling(200, min_periods=200).mean()).shift(1),
    "spy_ret_5d": spy["close"].pct_change(5).shift(1),
    "spy_day_move_atr": (spy["close"] - spy["open"]).abs()
                        / spy_tr.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean().shift(1),   # ex-post
}, index=spy.index)
tr = win_trades.join(ctx, on="date")


def bucket(s: pd.Series, edges: list, labels: list) -> pd.Series:
    return pd.cut(s, edges, labels=labels, right=False)


minute = tr["entry_col"] + RTH_OPEN
COND = {
    "Volatility regime (SPY 20d)": bucket(tr["spy_vol20"], [0, 0.12, 0.20, 0.30, np.inf],
                                          ["calm <12%", "normal 12–20%", "elevated 20–30%", "crisis >30%"]),
    "Market trend (SPY vs 200d)": tr["spy_above_200d"].map({True: "above 200d", False: "below 200d"}),
    "Prior week (SPY 5d return)": bucket(tr["spy_ret_5d"], [-np.inf, -0.02, 0.0, 0.02, np.inf],
                                         ["≤ −2%", "−2–0%", "0–2%", "≥ +2%"]),
    "ETF": tr["asset"],
    "Direction": tr["dir"].map({1: "long", -1: "short"}),
    "Gap size (|gap| / ATR)": bucket(tr["gap_atr"].abs(), [0, 0.3, 0.5, 1.0, np.inf], ["< 0.3", "0.3–0.5", "0.5–1.0", "> 1.0"]),
    "OR width (OR / ATR)": bucket(tr["or_atr"], [0, 0.2, 0.35, 0.5, np.inf], ["< 0.20", "0.20–0.35", "0.35–0.50", "> 0.50"]),
    "Entry time": bucket(minute, [0, 600, 630, 690, 24 * 60], ["before 10:00", "10:00–10:30", "10:30–11:30", "after 11:30"]),
    "Vol trend (ATR14/ATR50)": pd.Series(np.where(tr["atr_ratio"] > 1, "expanding", "contracting"), index=tr.index),
    "Day of week": pd.Series(pd.Categorical(tr["date"].dt.day_name().str[:3], ["Mon", "Tue", "Wed", "Thu", "Fri"],
                                            ordered=True), index=tr.index),
    "Day type (SPY |close−open| / ATR) — EX-POST": bucket(tr["spy_day_move_atr"], [0, 0.3, 0.7, np.inf],
                                                          ["range < 0.3", "0.3–0.7", "trend > 0.7"]),
}
live = tr["date"] >= LIVE_START


def group_stats(key: pd.Series) -> pd.DataFrame:
    g = tr.groupby(key, observed=True)["R"]
    out = pd.DataFrame({"trades": g.size(), "mean_R": g.mean(), "se": g.std(ddof=1) / np.sqrt(g.size()),
                        "hit_%": 100 * g.apply(lambda s: (s > 0).mean()), "total_R": g.sum()})
    out["t_stat"] = out["mean_R"] / out["se"]
    out["live_mean_R"] = tr.loc[live].groupby(key[live], observed=True)["R"].mean()
    return out


R = tr["R"]
print(f"\n{len(tr):,} trades {tr['date'].min().date()} → {tr['date'].max().date()} | mean {R.mean():+.3f} R "
      f"(t = {R.mean() / (R.std() / np.sqrt(len(R))):.2f}) | hit {100 * (R > 0).mean():.1f}% | "
      f"avg win {R[R > 0].mean():+.2f} R, avg loss {R[R <= 0].mean():+.2f} R | live-period mean {R[live].mean():+.3f} R")

tables = {c: group_stats(k) for c, k in COND.items()}
ranked = (pd.concat([t.assign(condition=c) for c, t in tables.items()]).reset_index(names="bucket")
          .query("trades >= @MIN_BUCKET_TRADES").sort_values("mean_R", ascending=False))
cols = ["condition", "bucket", "trades", "mean_R", "t_stat", "hit_%", "total_R", "live_mean_R"]
print(f"\nBEST conditions (≥ {MIN_BUCKET_TRADES} trades):")
display(ranked.head(6)[cols].round(3))
print("WORST conditions:")
display(ranked.tail(6).iloc[::-1][cols].round(3))
print(f"Caution: {len(ranked)} buckets are compared, so |t| ≈ 2 can occur by chance; treat |t| < 3, or buckets "
      "whose live-period mean disagrees in sign, as unconfirmed.")

# Figure 1 — mean R by condition (bars ± 1 SE)
ncols, nrows = 3, int(np.ceil(len(COND) / 3))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=list(COND), vertical_spacing=0.09, horizontal_spacing=0.06)
for i, c in enumerate(COND):
    t = tables[c]
    rr, cc = divmod(i, ncols)
    color = np.where(t["trades"] < MIN_BUCKET_TRADES, "#bdbdbd", np.where(t["mean_R"] >= 0, "#2ca02c", "#d62728"))
    fig.add_trace(go.Bar(x=t.index.astype(str), y=t["mean_R"], marker_color=color,
                         error_y=dict(type="data", array=t["se"], thickness=1, width=3),
                         text=[f"n={n}" for n in t["trades"]], textposition="outside", textfont=dict(size=9),
                         hovertemplate="%{x}<br>mean R %{y:.3f}<br>%{text}<extra></extra>", showlegend=False),
                  row=rr + 1, col=cc + 1)
    fig.add_hline(y=R.mean(), line_dash="dot", line_color="grey", line_width=1, row=rr + 1, col=cc + 1)
    fig.update_yaxes(title_text="mean R" if cc == 0 else None, zeroline=True, zerolinecolor="black", row=rr + 1, col=cc + 1)
fig.update_annotations(font_size=11)
fig.update_layout(title=("Winning ORB config — average result per trade by market condition"
                         "<br><sup>Green = profitable, red = losing, grey = < 30 trades · error bars ±1 SE · "
                         "dotted line = overall average · EX-POST panel is explanatory only</sup>"),
                  template="plotly_white", height=330 * nrows, bargap=0.35)
fig.show()

# Figure 2 — cumulative R with the 3 worst drawdowns and the best 12 months highlighted
cumR = tr.groupby("date")["R"].sum().reindex(CAL, fill_value=0.0).cumsum()
cumR = cumR.loc[cumR.index >= tr["date"].min()]
dd = cumR - cumR.cummax()
episodes, start = [], None
for dt, v in dd.items():
    if v < 0 and start is None:
        start = dt
    elif v == 0 and start is not None:
        episodes.append((cumR.loc[:start].idxmax(), dd.loc[start:dt].idxmin(), dd.loc[start:dt].min()))
        start = None
if start is not None:
    episodes.append((cumR.loc[:start].idxmax(), dd.loc[start:].idxmin(), dd.loc[start:].min()))
roll = cumR.diff(252)
best_end = roll.idxmax()
best_start = cumR.index[max(cumR.index.get_loc(best_end) - 252, 0)]
fig = go.Figure(go.Scatter(x=cumR.index, y=cumR.values, line=dict(color="#1f77b4", width=2), name="cumulative R"))
for k, (pk, trough, depth) in enumerate(sorted(episodes, key=lambda e: e[2])[:3], 1):
    fig.add_vrect(x0=pk, x1=trough, fillcolor="#d62728", opacity=0.15, line_width=0)
    fig.add_annotation(x=trough, y=cumR.loc[trough], text=f"DD #{k}: {depth:.1f} R<br>{pk:%b %Y}–{trough:%b %Y}",
                       showarrow=True, arrowhead=2, ax=0, ay=40, font=dict(size=10, color="#d62728"))
fig.add_vrect(x0=best_start, x1=best_end, fillcolor="#2ca02c", opacity=0.15, line_width=0)
fig.add_annotation(x=best_end, y=cumR.loc[best_end], text=f"best 12 months: +{roll.max():.1f} R", showarrow=True,
                   arrowhead=2, ax=0, ay=-40, font=dict(size=10, color="#2ca02c"))
fig.add_vline(x=LIVE_START, line_dash="dash", line_color="grey")
fig.add_annotation(x=LIVE_START, y=1, yref="y domain", text="  live →", showarrow=False, xanchor="left")
fig.update_layout(title="Winning ORB config — cumulative R (red = 3 worst drawdowns, green = best 12 months)",
                  yaxis_title="cumulative R (after costs)", template="plotly_white", height=480, showlegend=False)
fig.show()

Winner (config #9323, live from 2017):
  Logic(n=15, dir_mode='candle+gap', buffer=0.02, confirm='close', cutoff=120, stop=('or_mid', 0.0), be_R=2.0, trail_atr=0.0, target_R=0.0, exit_col=270)
  Filt(rv_min=0.0, or_atr_max=0.5, gap_min=0.3, regime='any', nr7=False, universe='equity', top_n=None, min_avg_vol=0.0, min_atr=0.0)

2,844 trades 2003-01-24 → 2026-09-21 | mean +0.227 R (t = 7.46) | hit 42.2% | avg win +1.79 R, avg loss -0.92 R | live-period mean +0.255 R

BEST conditions (≥ 30 trades):


,condition,bucket,trades,mean_R,t_stat,hit_%,total_R,live_mean_R
35,Day type (SPY |close−open| / ATR) — EX-POST,trend > 0.7,774,1.089,14.833,61.111,843.218,1.208
3,Volatility regime (SPY 20d),crisis >30%,175,0.494,3.834,50.286,86.411,0.264
18,Gap size (|gap| / ATR),> 1.0,153,0.413,2.808,43.137,63.122,0.718
2,Volatility regime (SPY 20d),elevated 20–30%,445,0.374,4.700,45.618,166.352,0.484
5,Market trend (SPY vs 200d),below 200d,701,0.366,5.717,45.364,256.806,0.326
32,Day of week,Fri,515,0.325,4.379,44.272,167.509,0.429


WORST conditions:


,condition,bucket,trades,mean_R,t_stat,hit_%,total_R,live_mean_R
33,Day type (SPY |close−open| / ATR) — EX-POST,range < 0.3,962,-0.431,-12.759,22.869,-414.605,-0.407
24,Entry time,10:30–11:30,370,0.102,1.593,47.297,37.790,0.219
21,OR width (OR / ATR),0.35–0.50,659,0.103,1.935,43.096,68.021,0.084
10,ETF,DIA,696,0.166,2.842,41.523,115.365,0.074
29,Day of week,Tue,603,0.169,2.682,41.459,101.835,0.229
1,Volatility regime (SPY 20d),normal 12–20%,1003,0.171,3.512,41.176,171.132,0.188


Caution: 35 buckets are compared, so |t| ≈ 2 can occur by chance; treat |t| < 3, or buckets whose live-period mean disagrees in sign, as unconfirmed.


#### **Where it does best**
Trend days, by a wide margin. On days when SPY closes far from its open, the average is +1.09R per trade (+1.21R in the live period, 2017+). On quiet, range-bound days it’s −0.43R. That label is only known after the fact, so it can’t be used as a filter. It does tell you what the strategy is: a bet that today will trend, and almost everything below is a proxy for that.
High volatility. When SPY’s 20-day volatility is above 20 %, trades average +0.37 to +0.49R, against about +0.17R in calm markets. The best single stretch was Oct 2007 – Oct 2008, at +121R.
Markets below their 200-day average: +0.37R vs +0.18R above it.
Bigger gaps and very narrow ranges. Gaps above 1 × ATR average +0.41R. Opening ranges under 0.35 × ATR do better than wider ones.
SPY and QQQ: +0.26R each, versus +0.17R for DIA.
Long and short are about equal (+0.23R vs +0.22R), so it isn’t just riding the bull market.

#### **Where it does worst**
Calm, rising markets. Volatility under 12 % with SPY above its 200-day average is the most common condition (1,203 trades), and the edge is thinnest there: about +0.18R.
Late entries (10:30–11:30), wide ranges (0.35–0.5 × ATR) and DIA. All three are positive but weak, with t-statistics below 3. Wide ranges and DIA nearly vanish in the live period (+0.08R and +0.07R).
The worst periods were 2004–05, mid-2023 to early 2024, and Aug 2025 to Apr 2026, each a drawdown of about 30R. Two of the three worst drawdowns are recent, and 2025 was roughly flat (+1R). That could be a low-volatility patch or the edge fading, and it’s too early to tell which.
#### **What this suggests**

*Two filters are worth testing next: skip entries after 10:30, and skip opening ranges wider than 0.35 × ATR. Both have a clear rationale, and both weak buckets also stayed weak in the live period. Adding them now would be fitting to the same data, though, so they would need confirming on new data.*

Combined conditions are thin at the extremes: crisis volatility while SPY is above its 200-day average has 1 trade, and calm volatility while SPY is below it has 13. Don't read anything into those combinations.

## Winner vs SPY buy & hold
Two realistic implementations of the winner against SPY:
* **ORB @ 1 % risk.** Integer shares, 1 % of equity risked per trade, 4× gross-leverage cap.
* **70 % SPY / 30 % ORB.** A buy-and-hold core with an ORB sleeve, rebalanced daily.

The live period (from `LIVE_START`) is genuinely out-of-sample.

In [8]:
CRISES = {"GFC 2007–09": ("2007-10-09", "2009-03-09"), "Aug 2011 downgrade": ("2011-07-22", "2011-10-03"),
          "Q4 2018 selloff": ("2018-09-20", "2018-12-24"), "COVID crash 2020": ("2020-02-19", "2020-03-23"),
          "2022 bear market": ("2022-01-03", "2022-10-12"), "2025 tariff shock": ("2025-02-19", "2025-04-08")}
BLEND_SPY = 0.70
ORB_NAME, BLEND_NAME, SPY_NAME = "ORB @ 1% risk (4× cap)", f"{BLEND_SPY:.0%} SPY / {1 - BLEND_SPY:.0%} ORB", f"{BENCHMARK} buy & hold"

cmp = pd.DataFrame(index=CAL)
cmp[SPY_NAME] = R_SPY
cmp[ORB_NAME] = run_risk(win_trades, dict(risk=0.01, gross_lev=4.0, max_trades=99, daily_loss=0.0,
                                          dd_throttle=0.0, eq_filter=False))
cmp[BLEND_NAME] = BLEND_SPY * cmp[SPY_NAME] + (1 - BLEND_SPY) * cmp[ORB_NAME]
cmp = cmp.loc[cmp.index >= win_trades["date"].min()]
COLORS = {SPY_NAME: "#d62728", ORB_NAME: "#1f77b4", BLEND_NAME: "#2ca02c"}

for label, m in ((f"FULL PERIOD (partly in-sample before {LIVE_START.year})", cmp.index >= cmp.index[0]),
                 (f"LIVE / OUT-OF-SAMPLE ({LIVE_START.year}+)", cmp.index >= LIVE_START)):
    sub = cmp.loc[m]
    print(f"\n{label}: {sub.index[0].date()} → {sub.index[-1].date()}")
    display(pd.DataFrame({c: perf_stats(sub[c], sub[SPY_NAME]) for c in sub}).round(2))
print("\nReturn during SPY's major drawdowns (%):")
display(pd.DataFrame({name: {c: 100 * ((1 + cmp.loc[a:b, c]).prod() - 1) for c in cmp}
                      for name, (a, b) in CRISES.items()}).T.round(1))

# Figure 1 — growth, drawdown, rolling 12-month return, rolling correlation
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, row_heights=[0.40, 0.20, 0.22, 0.18], vertical_spacing=0.035,
                    subplot_titles=("Growth of $25,000 (log scale)", "Drawdown (%)", "Rolling 12-month return (%)",
                                    "Rolling 12-month correlation of ORB to SPY"))
for c in cmp:
    eq = INITIAL_EQUITY * (1 + cmp[c]).cumprod()
    fig.add_trace(go.Scatter(x=eq.index, y=eq, name=c, line=dict(color=COLORS[c], width=2.5), legendgroup=c), 1, 1)
    fig.add_trace(go.Scatter(x=eq.index, y=100 * (eq / eq.cummax() - 1), name=c, showlegend=False,
                             line=dict(color=COLORS[c], width=1.2), legendgroup=c), 2, 1)
    fig.add_trace(go.Scatter(x=eq.index, y=100 * (eq / eq.shift(252) - 1), name=c, showlegend=False,
                             line=dict(color=COLORS[c], width=1.2), legendgroup=c), 3, 1)
fig.add_trace(go.Scatter(x=cmp.index, y=cmp[ORB_NAME].rolling(252).corr(cmp[SPY_NAME]), showlegend=False,
                         line=dict(color="#555555", width=1.2)), 4, 1)
for row in range(1, 5):
    fig.add_vrect(x0=LIVE_START, x1=cmp.index[-1], fillcolor="grey", opacity=0.07, line_width=0, row=row, col=1)
    for a, b in CRISES.values():
        fig.add_vrect(x0=a, x1=b, fillcolor="#d62728", opacity=0.06, line_width=0, row=row, col=1)
for row in (3, 4):
    fig.add_hline(y=0, line_color="black", line_width=0.8, row=row, col=1)
fig.add_annotation(x=LIVE_START, y=1, yref="y domain", text="  live / out-of-sample →", showarrow=False,
                   xanchor="left", row=1, col=1)
fig.update_yaxes(type="log", tickformat="$,.0f", row=1, col=1)
fig.update_yaxes(range=[-1, 1], row=4, col=1)
fig.update_layout(title="Winning ORB strategy vs SPY buy & hold<br><sup>Grey = live / out-of-sample · light red = SPY crisis windows</sup>",
                  template="plotly_white", height=1100, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0))
fig.show()

# Figure 2 — calendar-year returns side by side + monthly scatter
yearly = (1 + cmp).resample("YE").prod() - 1
yearly.index = yearly.index.year
monthly = (1 + cmp).resample("ME").prod() - 1
fig = make_subplots(rows=1, cols=2, column_widths=[0.64, 0.36], horizontal_spacing=0.08,
                    subplot_titles=("Calendar-year return (%)", "Monthly returns: ORB vs SPY (%)"))
for c in cmp:
    fig.add_trace(go.Bar(x=yearly.index, y=100 * yearly[c], name=c, marker_color=COLORS[c]), 1, 1)
fig.add_trace(go.Scatter(x=100 * monthly[SPY_NAME], y=100 * monthly[ORB_NAME], mode="markers", showlegend=False,
                         marker=dict(size=6, opacity=0.55, color=np.where(monthly.index >= LIVE_START, "#1f77b4", "#9ecae1")),
                         text=monthly.index.strftime("%b %Y"), hovertemplate="%{text}<br>SPY %{x:.1f}%<br>ORB %{y:.1f}%"), 1, 2)
fig.add_hline(y=0, line_color="black", line_width=0.8, row=1, col=2)
fig.add_vline(x=0, line_color="black", line_width=0.8, row=1, col=2)
fig.update_xaxes(title_text="SPY monthly return %", row=1, col=2)
fig.update_yaxes(title_text="ORB monthly return %", row=1, col=2)
fig.update_layout(template="plotly_white", height=520, barmode="group",
                  legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="left", x=0),
                  title=f"Year-by-year and month-by-month comparison (dark dots = live period, {LIVE_START.year}+)")
fig.show()


FULL PERIOD (partly in-sample before 2017): 2003-01-24 → 2026-09-21


,SPY buy & hold,ORB @ 1% risk (4× cap),70% SPY / 30% ORB
Total return %,1144.80,2372.17,1658.91
CAGR %,11.27,14.55,12.91
Volatility %,18.49,13.90,13.36
Sharpe,0.67,1.05,0.98
Sortino,0.94,2.31,1.47
Max drawdown %,-55.20,-30.82,-28.81
Calmar,0.20,0.47,0.45
Recovery (sessions),869.00,550.00,96.00
Worst day %,-11.63,-3.81,-8.14
Worst month %,-16.87,-6.34,-7.96



LIVE / OUT-OF-SAMPLE (2017+): 2017-01-03 → 2026-09-21


,SPY buy & hold,ORB @ 1% risk (4× cap),70% SPY / 30% ORB
Total return %,276.93,547.47,371.05
CAGR %,14.67,21.26,17.34
Volatility %,18.25,15.35,13.34
Sharpe,0.84,1.33,1.27
Sortino,1.17,3.15,1.91
Max drawdown %,-33.85,-11.88,-24.59
Calmar,0.43,1.79,0.71
Recovery (sessions),97.00,100.00,70.00
Worst day %,-11.63,-2.59,-8.14
Worst month %,-12.47,-6.34,-7.96



Return during SPY's major drawdowns (%):


,SPY buy & hold,ORB @ 1% risk (4× cap),70% SPY / 30% ORB
GFC 2007–09,-54.8,121.2,-24.9
Aug 2011 downgrade,-17.9,-2.4,-13.3
Q4 2018 selloff,-18.5,6.2,-11.6
COVID crash 2020,-33.5,0.4,-24.2
2022 bear market,-28.4,38.4,-11.6
2025 tariff shock,-18.4,-3.4,-14.0


### T-test
Determines whether the sharpe will actuially hold or not

In [9]:
# ═════════════════════════════════════════════════════════════════════════════
# T-TESTS ON THE WINNING ORB STRATEGY
#
# 1. Does it make money?   One-sample t-tests (H0: mean = 0) on
#      • per-trade R                       (treats every trade as independent)
#      • per-day R (trades summed by day)  (fixes same-day correlation across the 4 ETFs)
#      • daily portfolio returns @ 1 % risk (Newey-West SE for autocorrelation)
#    each for the full sample, the in-sample years, and the LIVE period.
# 2. Does it beat SPY (live)?  Paired t-test on daily return differences (Newey-West) and the
#    Jobson-Korkie / Memmel test for a difference in Sharpe ratios.
# 3. Has the edge changed?  Welch t-test, in-sample vs live per-trade R.
#
# Selection-bias note: the winner was picked from ~13,440 configurations using pre-live data, so
# full-sample / in-sample p-values are optimistic. The LIVE rows are the honest test. Harvey, Liu &
# Zhu (2016) suggest |t| > 3 for strategies found by search.
#
# Requires: the "Analyze winning strategy" cell (win_trades, LIVE_START) and the
#           "Winner vs SPY" cell (cmp, ORB_NAME, SPY_NAME).
# ═════════════════════════════════════════════════════════════════════════════
for _name in ("win_trades", "LIVE_START", "cmp", "ORB_NAME", "SPY_NAME"):
    if _name not in globals():
        raise RuntimeError("Run the 'Analyze winning strategy' and 'Winner vs SPY' cells first.")

N_BOOT = 10_000
ALPHA = 0.05
rng_t = np.random.default_rng(11)


def nw_se(x: np.ndarray, lags: int | None = None) -> tuple[float, int]:
    """Newey-West (Bartlett) standard error of the mean; default lag = floor(4·(n/100)^(2/9))."""
    x = np.asarray(x, float)
    n = len(x)
    e = x - x.mean()
    L = int(np.floor(4 * (n / 100) ** (2 / 9))) if lags is None else lags
    s = e @ e / n
    for l in range(1, L + 1):
        s += 2 * (1 - l / (L + 1)) * (e[l:] @ e[:-l]) / n
    return np.sqrt(s / n), L


def one_sample(x: np.ndarray, label: str, sample: str, hac: bool = False, annualize: bool = False) -> dict:
    x = np.asarray(x, float)
    n, mu = len(x), x.mean()
    if hac:
        se, lags = nw_se(x)
        t = mu / se
        p = 2 * stats.t.sf(abs(t), n - 1)
    else:
        t, p = stats.ttest_1samp(x, 0.0)
        se, lags = x.std(ddof=1) / np.sqrt(n), 0
    q = stats.t.ppf(1 - ALPHA / 2, n - 1)
    boot = x[rng_t.integers(0, n, size=(N_BOOT, n))].mean(axis=1)
    k = 252 if annualize else 1
    return {"test": label, "sample": sample, "n": n, "mean": mu * k, "t_stat": t, "p_two_sided": p,
            "p_one_sided (>0)": stats.t.sf(t, n - 1), "CI95_low": (mu - q * se) * k, "CI95_high": (mu + q * se) * k,
            "boot_CI95_low": np.percentile(boot, 2.5) * k, "boot_CI95_high": np.percentile(boot, 97.5) * k,
            "NW_lags": lags if hac else np.nan}


# ── 1. Does it make money? ───────────────────────────────────────────────────
tt = win_trades.copy()
live_mask = tt["date"] >= LIVE_START
day_R = tt.groupby("date")["R"].sum()
day_live = day_R.index >= LIVE_START
port = cmp[ORB_NAME]
port_live = port.index >= LIVE_START
samples = {"full": (slice(None), slice(None), slice(None)),
           f"in-sample (< {LIVE_START.year})": (~live_mask, ~day_live, ~port_live),
           f"LIVE ({LIVE_START.year}+)": (live_mask, day_live, port_live)}
rows = []
for name, (mt, md, mp) in samples.items():
    rows.append(one_sample(tt.loc[mt, "R"].to_numpy(), "per-trade R", name))
    rows.append(one_sample(day_R[md].to_numpy(), "per-day R (trades summed)", name))
    rows.append(one_sample(port[mp].to_numpy(), "daily return @1% (NW, annualised mean)", name, hac=True, annualize=True))
t1 = pd.DataFrame(rows).set_index(["test", "sample"])
print("1) DOES THE STRATEGY MAKE MONEY?  H0: mean = 0")
display(t1.round(4))

# ── 2. Does it beat SPY in the live period? ──────────────────────────────────
live_cmp = cmp.loc[cmp.index >= LIVE_START, [ORB_NAME, SPY_NAME]]
diff = (live_cmp[ORB_NAME] - live_cmp[SPY_NAME]).to_numpy()
se_d, lags_d = nw_se(diff)
t_d = diff.mean() / se_d
s1, s2 = live_cmp[ORB_NAME], live_cmp[SPY_NAME]
sr1, sr2, rho, T = s1.mean() / s1.std(), s2.mean() / s2.std(), s1.corr(s2), len(live_cmp)
var_jk = (2 - 2 * rho + 0.5 * (sr1 ** 2 + sr2 ** 2 - 2 * sr1 * sr2 * rho ** 2)) / T     # Memmel (2003)
z_sr = (sr1 - sr2) / np.sqrt(var_jk)
r_corr, p_corr = stats.pearsonr(s1, s2)
t2 = pd.DataFrame({
    "Mean daily return difference (ORB − SPY), annualised %": {
        "estimate": 100 * 252 * diff.mean(), "statistic": t_d, "p_two_sided": 2 * stats.t.sf(abs(t_d), T - 1),
        "note": f"paired, Newey-West ({lags_d} lags)"},
    "Sharpe ratio difference (ORB − SPY), annualised": {
        "estimate": np.sqrt(252) * (sr1 - sr2), "statistic": z_sr, "p_two_sided": 2 * stats.norm.sf(abs(z_sr)),
        "note": "Jobson-Korkie / Memmel z-test"},
    "Correlation ORB vs SPY": {
        "estimate": r_corr, "statistic": r_corr * np.sqrt((T - 2) / (1 - r_corr ** 2)), "p_two_sided": p_corr,
        "note": "H0: correlation = 0"},
}).T
print(f"\n2) DOES IT BEAT {SPY_NAME.upper()}?  Live period {LIVE_START.year}+ ({T:,} sessions), ORB @ 1 % risk")
display(t2)

# ── 3. Has the edge changed? ─────────────────────────────────────────────────
a, b = tt.loc[~live_mask, "R"], tt.loc[live_mask, "R"]
t_w, p_w = stats.ttest_ind(b, a, equal_var=False)
u, p_u = stats.mannwhitneyu(b, a, alternative="two-sided")
print(f"\n3) HAS THE EDGE CHANGED?  in-sample mean {a.mean():+.3f} R (n={len(a):,}) vs live {b.mean():+.3f} R "
      f"(n={len(b):,})\n   Welch t = {t_w:+.2f}, p = {p_w:.3f}   |   Mann-Whitney U p = {p_u:.3f} (distribution-free)")

# ── Verdict ──────────────────────────────────────────────────────────────────
live_trade = t1.loc[("per-trade R", f"LIVE ({LIVE_START.year}+)")]
live_day = t1.loc[("per-day R (trades summed)", f"LIVE ({LIVE_START.year}+)")]
print("\nVERDICT")
print(f"  • Live edge: mean {live_trade['mean']:+.3f} R per trade, t = {live_trade['t_stat']:.2f} "
      f"(per-day t = {live_day['t_stat']:.2f}) → "
      + ("significant at 5 %" if live_day["p_two_sided"] < ALPHA else "NOT significant at 5 %")
      + ("; clears the |t| > 3 bar for searched strategies." if live_day["t_stat"] > 3 else "; below the |t| > 3 bar for searched strategies."))
print(f"  • vs SPY (live): Sharpe difference p = {t2.loc['Sharpe ratio difference (ORB − SPY), annualised', 'p_two_sided']:.3f}, "
      f"return difference p = {t2.iloc[0]['p_two_sided']:.3f}.")
print(f"  • Edge change in-sample → live: p = {p_w:.3f} "
      + ("(no evidence of decay)." if p_w > ALPHA else "(significant change)."))

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE — bootstrap distribution of mean R (in-sample vs live) and mean R by year (95 % CI)
# ═════════════════════════════════════════════════════════════════════════════
boot_is = a.to_numpy()[rng_t.integers(0, len(a), size=(N_BOOT, len(a)))].mean(axis=1)
boot_live = b.to_numpy()[rng_t.integers(0, len(b), size=(N_BOOT, len(b)))].mean(axis=1)
yr = tt.groupby(tt["date"].dt.year)["R"].agg(["mean", "std", "size"])
yr["ci"] = stats.t.ppf(0.975, yr["size"] - 1) * yr["std"] / np.sqrt(yr["size"])

fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58], horizontal_spacing=0.08,
                    subplot_titles=(f"Bootstrap distribution of mean R per trade ({N_BOOT:,} resamples)",
                                    "Mean R per trade by year (±95 % CI)"))
for vals, name, col in ((boot_is, f"in-sample (< {LIVE_START.year})", "#9ecae1"),
                        (boot_live, f"live ({LIVE_START.year}+)", "#1f77b4")):
    fig.add_trace(go.Histogram(x=vals, name=name, marker_color=col, opacity=0.7, nbinsx=80,
                               histnorm="probability density"), 1, 1)
fig.add_vline(x=0, line_color="#d62728", line_width=2, row=1, col=1)
fig.add_annotation(x=0, y=1, yref="y domain", text="H0: mean = 0", showarrow=False, xanchor="left",
                   font=dict(color="#d62728"), row=1, col=1)
fig.add_trace(go.Bar(x=yr.index, y=yr["mean"], error_y=dict(type="data", array=yr["ci"], thickness=1),
                     marker_color=np.where(yr.index >= LIVE_START.year, "#1f77b4", "#9ecae1"),
                     text=[f"n={n}" for n in yr["size"]], textposition="outside", textfont=dict(size=8),
                     showlegend=False), 1, 2)
fig.add_hline(y=0, line_color="black", line_width=1, row=1, col=2)
fig.update_xaxes(title_text="mean R per trade", row=1, col=1)
fig.update_yaxes(title_text="mean R", row=1, col=2)
fig.update_layout(title=f"Winning ORB strategy — t-test evidence (dark = live {LIVE_START.year}+)",
                  template="plotly_white", height=480, barmode="overlay",
                  legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="left", x=0))
fig.show()

1) DOES THE STRATEGY MAKE MONEY?  H0: mean = 0


,,n,mean,t_stat,p_two_sided,p_one_sided (>0),CI95_low,CI95_high,boot_CI95_low,boot_CI95_high,NW_lags
test,sample,,,,,,,,,,
per-trade R,full,2844,0.2269,7.4624,0.0000,0.0000,0.1673,0.2865,0.1677,0.2867,NaN
per-day R (trades summed),full,1486,0.4342,5.4792,0.0000,0.0000,0.2788,0.5897,0.2820,0.5908,NaN
"daily return @1% (NW, annualised mean)",full,5952,0.1453,5.0982,0.0000,0.0000,0.0894,0.2012,0.0896,0.2013,9.0
per-trade R,in-sample (< 2017),1585,0.2044,5.2922,0.0000,0.0000,0.1286,0.2801,0.1293,0.2804,NaN
per-day R (trades summed),in-sample (< 2017),777,0.4169,3.7190,0.0002,0.0001,0.1969,0.6370,0.1975,0.6400,NaN
"daily return @1% (NW, annualised mean)",in-sample (< 2017),3510,0.1042,3.0456,0.0023,0.0012,0.0371,0.1713,0.0386,0.1717,8.0
per-trade R,LIVE (2017+),1259,0.2552,5.2606,0.0000,0.0000,0.1600,0.3504,0.1602,0.3507,NaN
per-day R (trades summed),LIVE (2017+),709,0.4532,4.0512,0.0001,0.0000,0.2336,0.6728,0.2440,0.6772,NaN
"daily return @1% (NW, annualised mean)",LIVE (2017+),2442,0.2043,4.2307,0.0000,0.0000,0.1096,0.2990,0.1070,0.3020,8.0



2) DOES IT BEAT SPY BUY & HOLD?  Live period 2017+ (2,442 sessions), ORB @ 1 % risk


,estimate,statistic,p_two_sided,note
"Mean daily return difference (ORB − SPY), annualised %",5.062679,0.693402,0.488124,"paired, Newey-West (8 lags)"
"Sharpe ratio difference (ORB − SPY), annualised",0.489308,1.048028,0.294626,Jobson-Korkie / Memmel z-test
Correlation ORB vs SPY,-0.053714,-2.6571,0.007933,H0: correlation = 0



3) HAS THE EDGE CHANGED?  in-sample mean +0.204 R (n=1,585) vs live +0.255 R (n=1,259)
   Welch t = +0.82, p = 0.412   |   Mann-Whitney U p = 0.000 (distribution-free)

VERDICT
  • Live edge: mean +0.255 R per trade, t = 5.26 (per-day t = 4.05) → significant at 5 %; clears the |t| > 3 bar for searched strategies.
  • vs SPY (live): Sharpe difference p = 0.295, return difference p = 0.488.
  • Edge change in-sample → live: p = 0.412 (no evidence of decay).


### T-test findings: winning ORB strategy

**Bottom line:** the strategy has a real, statistically significant edge that held up out of sample. However, it can't be shown to *beat* SPY. Its value is as an independent return stream, not a replacement for buy & hold.

#### 1. Does it make money? Yes, including in the live period
| Live period (2017+) | Mean | t | p |
|---|---|---|---|
| Per trade (1,259 trades) | +0.255 R | 5.26 | < 0.0001 |
| Per day, same-day trades combined (709 days) | +0.453 R | 4.05 | 0.0001 |
| Daily returns at 1 % risk (Newey-West) | +20.4 % / yr | 4.23 | < 0.0001 |

- All three live results clear the stricter **|t| > 3** bar recommended for strategies found by searching (Harvey, Liu & Zhu, 2016).
- Bootstrap confidence intervals match the t-test intervals (live R per trade: **+0.16 to +0.35**), so the skewed trade outcomes aren't distorting the result.

#### 2. Does it beat SPY? Not provably
| Live period (2017+), ORB at 1 % risk vs SPY | Estimate | p |
|---|---|---|
| Return difference | +5.1 % / yr | 0.49 |
| Sharpe ratio difference | +0.49 | 0.29 |
| Correlation to SPY | −0.05 | 0.008 |

- About ten years of daily data can't statistically separate two strategies whose Sharpe ratios differ by about 0.5.
- The near-zero correlation **is** significant. The defensible claim is a real, *uncorrelated* edge, not "beats SPY".

#### 3. Has the edge decayed? No evidence of that
- Mean R per trade was **+0.20** before 2017 and **+0.26** since (Welch t = 0.82, p = 0.41).
- A Mann-Whitney test flags a change in the *shape* of trade outcomes (p < 0.001) even though the average is unchanged. A possible cause is more near-breakeven exits in recent years. That's worth investigating, but it isn't a loss of edge.

#### Caveats
- **Selection bias:** this configuration was the best of ~13,440 tested, so full-sample and in-sample p-values are optimistic. Quote only the **live (2017+)** results.
- **Sizing:** the 1 % risk version needs up to 4× intraday leverage and margin. The per-trade R results don't depend on sizing.
- **Execution:** results assume 1 tick of slippage per fill. Worse real-world fills would shrink the edge.
- **Next step:** confirm with paper trading or new data before committing capital.

### Monte carlo simulations

In [11]:
# ═════════════════════════════════════════════════════════════════════════════
# MONTE CARLO SIMULATION — winning ORB strategy (raw: every trade, 1 % risk, 4× leverage cap)
#
# Method: stationary block bootstrap (Politis & Romano, 1994) of DAILY returns. Each simulated path strings
#   together random blocks of consecutive trading days (mean block length BLOCK_LEN), which preserves
#   volatility regimes, losing streaks and the zero-return days between trades. SPY is sampled on the SAME
#   days, so strategy-vs-SPY comparisons are like-for-like.
# Source: the live / out-of-sample period (LIVE_START+) by default — the honest estimate of future behaviour.
# Stress tests: every trade's result is cut by 0.05 R and 0.10 R (≈ 20 % and 40 % of the live edge) to show
#   what happens if the edge decays or fills are worse than modelled.
# Practical risk: with a $25,000 start the account sits at the US pattern-day-trader minimum, so the
#   probability of ever dropping below PDT_MIN is reported.
#
# Requires: the "Analyze winning strategy" cell (win_trades, LIVE_START) and the "Winner vs SPY" cell
#           (cmp, ORB_NAME, SPY_NAME).
# ═════════════════════════════════════════════════════════════════════════════
for _name in ("win_trades", "LIVE_START", "cmp", "ORB_NAME", "SPY_NAME"):
    if _name not in globals():
        raise RuntimeError("Run the 'Analyze winning strategy' and 'Winner vs SPY' cells first.")

N_PATHS = 10_000
HORIZONS = {"1 year": 252, "5 years": 1260}
BLOCK_LEN = 20                       # mean block length in trading days
SOURCE = "live"                      # "live" (LIVE_START+) or "full"
START_EQUITY = 25_000.0
PDT_MIN = 25_000.0
HAIRCUTS_R = {"Base (as backtested)": 0.0, "Edge −0.05 R / trade": 0.05, "Edge −0.10 R / trade": 0.10}
rng_mc = np.random.default_rng(2024)

# ── Source returns ───────────────────────────────────────────────────────────
src = cmp.loc[cmp.index >= (LIVE_START if SOURCE == "live" else cmp.index[0]), [ORB_NAME, SPY_NAME]]
n_trades_day = win_trades.groupby("date").size().reindex(src.index, fill_value=0).to_numpy()
r_day = win_trades.groupby("date")["R"].sum().reindex(src.index, fill_value=0.0).to_numpy()
r_orb, r_spy = src[ORB_NAME].to_numpy(), src[SPY_NAME].to_numpy()
# Empirical "fraction of equity per 1 R" (captures the leverage cap, which often shrinks actual risk below 1 %)
equity_per_R = r_orb.sum() / r_day.sum()
print(f"Source: {SOURCE} period {src.index[0].date()} → {src.index[-1].date()} ({len(src):,} sessions, "
      f"{int(n_trades_day.sum()):,} trades). Effective risk per trade ≈ {100 * equity_per_R:.2f} % of equity per 1 R.")


def stationary_bootstrap_idx(n_obs: int, n_paths: int, horizon: int, mean_block: int) -> np.ndarray:
    """Index matrix (n_paths × horizon) for the stationary bootstrap with geometric block lengths."""
    idx = np.empty((n_paths, horizon), dtype=np.int64)
    idx[:, 0] = rng_mc.integers(0, n_obs, n_paths)
    jump = rng_mc.random((n_paths, horizon)) < 1.0 / mean_block
    fresh = rng_mc.integers(0, n_obs, (n_paths, horizon))
    for h in range(1, horizon):
        idx[:, h] = np.where(jump[:, h], fresh[:, h], (idx[:, h - 1] + 1) % n_obs)
    return idx


H_MAX = max(HORIZONS.values())
IDX = stationary_bootstrap_idx(len(src), N_PATHS, H_MAX, BLOCK_LEN)
spy_paths = START_EQUITY * np.cumprod(1 + r_spy[IDX], axis=1)


def path_stats(eq: np.ndarray, h: int, spy_eq: np.ndarray) -> dict:
    e = eq[:, :h]
    peak = np.maximum.accumulate(np.concatenate([np.full((len(e), 1), START_EQUITY), e], axis=1), axis=1)[:, 1:]
    mdd = (e / peak - 1).min(axis=1)
    term = e[:, -1]
    cagr = (term / START_EQUITY) ** (252 / h) - 1
    return {
        "Median CAGR %": 100 * np.median(cagr),
        "5th pct CAGR %": 100 * np.percentile(cagr, 5),
        "95th pct CAGR %": 100 * np.percentile(cagr, 95),
        "Median ending $": np.median(term),
        "P(loss) %": 100 * (term < START_EQUITY).mean(),
        "P(beat SPY) %": 100 * (term > spy_eq[:, h - 1]).mean(),
        "Median max DD %": 100 * np.median(mdd),
        "95th pct max DD %": 100 * np.percentile(mdd, 5),
        "P(max DD > 20 %) %": 100 * (mdd < -0.20).mean(),
        f"P(ever < ${PDT_MIN:,.0f}) %": 100 * (e.min(axis=1) < PDT_MIN).mean(),
        "P(lose half) %": 100 * (e.min(axis=1) < 0.5 * START_EQUITY).mean(),
    }


results, scenario_paths = {}, {}
for scen, cut in HAIRCUTS_R.items():
    r_s = r_orb - cut * equity_per_R * n_trades_day          # remove `cut` R from every trade that day
    eq = START_EQUITY * np.cumprod(1 + r_s[IDX], axis=1)
    scenario_paths[scen] = eq
    for hz, h in HORIZONS.items():
        results[(scen, hz)] = path_stats(eq, h, spy_paths)
for hz, h in HORIZONS.items():
    results[(f"{SPY_NAME} (same days)", hz)] = path_stats(spy_paths, h, spy_paths)

table = pd.DataFrame(results).T
table.index.names = ["scenario", "horizon"]
print(f"\nMonte Carlo: {N_PATHS:,} paths, stationary block bootstrap (mean block {BLOCK_LEN} days), "
      f"start ${START_EQUITY:,.0f}")
display(table.round(1))

base = table.loc[("Base (as backtested)", "5 years")]
cut10 = table.loc[("Edge −0.10 R / trade", "5 years")]
print("\nKEY TAKEAWAYS (5-year horizon)")
print(f"  • Base case: median CAGR {base['Median CAGR %']:.1f} % (5th–95th pct {base['5th pct CAGR %']:.1f} % to "
      f"{base['95th pct CAGR %']:.1f} %), P(loss) {base['P(loss) %']:.1f} %, P(beat SPY) {base['P(beat SPY) %']:.0f} %.")
print(f"  • Drawdowns: median worst drawdown {base['Median max DD %']:.1f} %; 1-in-20 paths see "
      f"{base['95th pct max DD %']:.1f} % or worse.")
print(f"  • PDT risk: {base[f'P(ever < ${PDT_MIN:,.0f}) %']:.0f} % of paths dip below ${PDT_MIN:,.0f} at some point — "
      "start with a cushion above the PDT minimum.")
print(f"  • If the edge decays by 0.10 R per trade: median CAGR {cut10['Median CAGR %']:.1f} %, "
      f"P(loss) {cut10['P(loss) %']:.1f} %, P(beat SPY) {cut10['P(beat SPY) %']:.0f} %.")

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — fan chart of equity paths (base case) vs SPY median
# ═════════════════════════════════════════════════════════════════════════════
days = np.arange(1, H_MAX + 1)
base_eq = scenario_paths["Base (as backtested)"]
pct = {q: np.percentile(base_eq, q, axis=0) for q in (5, 25, 50, 75, 95)}
fig = go.Figure()
fig.add_trace(go.Scatter(x=days, y=pct[95], line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=days, y=pct[5], fill="tonexty", fillcolor="rgba(31,119,180,0.15)", line=dict(width=0),
                         name="ORB 5th–95th percentile"))
fig.add_trace(go.Scatter(x=days, y=pct[75], line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=days, y=pct[25], fill="tonexty", fillcolor="rgba(31,119,180,0.30)", line=dict(width=0),
                         name="ORB 25th–75th percentile"))
for i in rng_mc.choice(N_PATHS, 15, replace=False):
    fig.add_trace(go.Scatter(x=days, y=base_eq[i], line=dict(color="rgba(31,119,180,0.25)", width=0.8),
                             showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=days, y=pct[50], line=dict(color="#1f77b4", width=3), name="ORB median"))
for scen, col in (("Edge −0.05 R / trade", "#ff7f0e"), ("Edge −0.10 R / trade", "#9467bd")):
    fig.add_trace(go.Scatter(x=days, y=np.median(scenario_paths[scen], axis=0), line=dict(color=col, width=2, dash="dash"),
                             name=f"ORB median, {scen.lower()}"))
fig.add_trace(go.Scatter(x=days, y=np.median(spy_paths, axis=0), line=dict(color="#d62728", width=2.5),
                         name=f"{SPY_NAME} median"))
fig.add_hline(y=PDT_MIN, line_dash="dot", line_color="black", annotation_text=f"PDT minimum ${PDT_MIN:,.0f}",
              annotation_position="bottom right")
for hz, h in HORIZONS.items():
    fig.add_vline(x=h, line_dash="dot", line_color="grey", annotation_text=hz, annotation_position="top left")
fig.update_yaxes(type="log", tickformat="$,.0f", title_text="Equity (log scale)")
fig.update_xaxes(title_text="Trading days ahead")
fig.update_layout(title=f"Monte Carlo — {N_PATHS:,} simulated paths of the winning ORB strategy ({SOURCE} period, block bootstrap)",
                  template="plotly_white", height=620, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0))
fig.show()

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — 5-year distributions: CAGR and maximum drawdown, strategy scenarios vs SPY
# ═════════════════════════════════════════════════════════════════════════════
h5 = HORIZONS["5 years"]


def cagr_mdd(eq: np.ndarray, h: int) -> tuple[np.ndarray, np.ndarray]:
    e = eq[:, :h]
    peak = np.maximum.accumulate(np.concatenate([np.full((len(e), 1), START_EQUITY), e], axis=1), axis=1)[:, 1:]
    return 100 * ((e[:, -1] / START_EQUITY) ** (252 / h) - 1), 100 * (e / peak - 1).min(axis=1)


dist_cols = {"Base (as backtested)": "#1f77b4", "Edge −0.10 R / trade": "#9467bd", f"{SPY_NAME} (same days)": "#d62728"}
dist_src = {"Base (as backtested)": scenario_paths["Base (as backtested)"],
            "Edge −0.10 R / trade": scenario_paths["Edge −0.10 R / trade"], f"{SPY_NAME} (same days)": spy_paths}
fig2 = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                     subplot_titles=("5-year CAGR distribution (%)", "5-year maximum drawdown distribution (%)"))
for name, eq in dist_src.items():
    cg, md = cagr_mdd(eq, h5)
    fig2.add_trace(go.Histogram(x=cg, name=name, marker_color=dist_cols[name], opacity=0.55, nbinsx=80,
                                histnorm="probability", legendgroup=name), 1, 1)
    fig2.add_trace(go.Histogram(x=md, name=name, marker_color=dist_cols[name], opacity=0.55, nbinsx=80,
                                histnorm="probability", legendgroup=name, showlegend=False), 1, 2)
fig2.add_vline(x=0, line_color="black", line_width=1, row=1, col=1)
fig2.add_vline(x=-20, line_dash="dot", line_color="black", row=1, col=2)
fig2.update_layout(barmode="overlay", template="plotly_white", height=460,
                   title="Monte Carlo — 5-year outcome distributions (strategy scenarios vs SPY on the same simulated days)",
                   legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="left", x=0))
fig2.show()

Source: live period 2017-01-03 → 2026-09-21 (2,442 sessions, 1,259 trades). Effective risk per trade ≈ 0.62 % of equity per 1 R.

Monte Carlo: 10,000 paths, stationary block bootstrap (mean block 20 days), start $25,000


Median CAGR %  5th pct CAGR %  95th pct CAGR %  Median ending $  P(loss) %  P(beat SPY) %  Median max DD %  95th pct max DD %  P(max DD > 20 %) %  P(ever < $25,000) %  P(lose half) %
scenario                   horizon                                                                                                                                                                                        
Base (as backtested)       1 year            20.1            -2.8             56.4          30020.3        7.8           57.4             -7.8              -13.9                 0.4                 84.2             0.0
                           5 years           20.9             9.4             35.3          64659.5        0.1           70.3            -12.6              -20.4                 5.6                 84.6             0.0
Edge −0.05 R / trade       1 year            15.4            -6.6             50.1          28852.5       13.5           50.0             -8.7              -15.7                 1.0                 87.0             0.0
                           5 years           16.2             5.0             30.0          52931.3        0.6           55.2            -14.5              -24.1                15.2                 87.9             0.0
Edge −0.10 R / trade       1 year            10.9           -10.3             44.2          27736.0       21.2           43.0             -9.7              -17.6                 2.3                 89.8             0.0
                           5 years           11.6             0.9             24.9          43356.0        3.6           39.9            -17.1              -29.1                31.0                 91.0             0.0
SPY buy & hold (same days) 1 year            15.8           -13.6             44.8          28957.9       18.4            0.0            -12.7              -32.7                18.3                 91.4             0.1
                           5 years           14.8             1.5             28.0          49867.1        3.2            0.0            -25.3              -43.3                70.1                 92.8             0.3


KEY TAKEAWAYS (5-year horizon)
  • Base case: median CAGR 20.9 % (5th–95th pct 9.4 % to 35.3 %), P(loss) 0.1 %, P(beat SPY) 70 %.
  • Drawdowns: median worst drawdown -12.6 %; 1-in-20 paths see -20.4 % or worse.
  • PDT risk: 85 % of paths dip below $25,000 at some point — start with a cushion above the PDT minimum.
  • If the edge decays by 0.10 R per trade: median CAGR 11.6 %, P(loss) 3.6 %, P(beat SPY) 40 %.


### Monte Carlo findings: winning ORB strategy

**Setup:** 10,000 simulated paths, $25,000 start, 1 % risk per trade with the 4× leverage cap (actual risk averages **0.62 %** per trade because the cap often binds). Paths are built from the **2017–2026 live period** using a block bootstrap (~20-day blocks), which keeps volatility regimes and losing streaks intact. SPY is sampled on the same days.

### Expected outcomes
| Scenario | Horizon | Median CAGR | 5th–95th pct CAGR | P(loss) | P(beat SPY) | Median max DD | 1-in-20 worst DD |
|---|---|---|---|---|---|---|---|
| **ORB as backtested** | 1 yr | 20.1 % | −2.8 % to 56.4 % | 7.8 % | 57 % | −7.8 % | −13.9 % |
| **ORB as backtested** | 5 yr | **20.9 %** | 9.4 % to 35.3 % | **0.1 %** | **70 %** | −12.6 % | −20.4 % |
| ORB, edge −0.05 R / trade | 5 yr | 16.2 % | 5.0 % to 30.0 % | 0.6 % | 55 % | −14.5 % | −24.1 % |
| ORB, edge −0.10 R / trade | 5 yr | 11.6 % | 0.9 % to 24.9 % | 3.6 % | 40 % | −17.1 % | −29.1 % |
| SPY buy & hold | 1 yr | 15.8 % | −13.6 % to 44.8 % | 18.4 % | — | −12.7 % | −32.7 % |
| SPY buy & hold | 5 yr | 14.8 % | 1.5 % to 28.0 % | 3.2 % | — | −25.3 % | −43.3 % |

### Key takeaways
- **If the live edge holds, five years almost certainly ends in profit.** Only 0.1 % of paths lose money. The strategy beats SPY on **70 %** of paths, with about **half SPY's drawdowns** (−12.6 % vs −25.3 % median).
- **One year is much less certain.** About 1 in 13 paths lose money in the first year, and the strategy beats SPY only 57 % of the time.
- **It's sensitive to edge decay.** Removing 0.10R per trade (about 40 % of the edge) cuts the median CAGR to 11.6 % and drops it **below SPY on the majority of paths** (40 % win rate). Slippage worse than one tick, or a weaker regime, could do this.
- **Tail risk is contained.** Even the 1-in-20 worst 5-year drawdown (−20.4 %) is milder than SPY's *median* drawdown. Essentially no path loses half the account.
- **The pattern-day-trader minimum is the biggest practical risk.** About **85 %** of paths dip below $25,000 at some point. Starting at exactly the minimum, an early loss would block day trading. **Start with a cushion (e.g. $30k or more).**

### Limitations
- **It resamples history.** It can't produce a crash or regime worse than anything in 2017–2026. Set `SOURCE = "full"` to include 2003–2016, which adds the 2008 crash and the weaker 2004–05 years.
- **It assumes the live-period edge is the true edge.** That edge came from a configuration selected out of ~13,440, so treat the base case as optimistic and the decay scenarios as plausible.
- **Execution isn't stress-tested beyond the R haircuts.** Gaps, halts and partial fills at bar-close entries aren't modelled.
```

## Machine Learning implementations
***
utilizes forest tree classification to determine whether the strategy should or should not trade or double down on sizing.

### Double down ML feature
Utilizes a random forest tree to double down on high probable setups

In [10]:
# ═════════════════════════════════════════════════════════════════════════════
# RANDOM FOREST DOUBLE-DOWN — take every trade of the winning ORB strategy, double only high-probability ones
#
# The ORB strategy generates every signal and EVERY trade is taken (nothing is skipped). A Random Forest
# estimates, at the moment of entry, the probability that the trade becomes a BIG winner (R ≥ TARGET_R,
# i.e. a trend-day trade), and the policy is:
#     p ≥ double threshold  → DOUBLE the risk (2 % instead of 1 %, still inside the 4× leverage cap)
#     otherwise             → normal size (1 %)
#
# Control: doubling ANY trades adds risk and therefore (on average) profit. The model only adds value if its
# picks beat doubling the same number of trades at RANDOM — tested with a Monte-Carlo of random doubling.
#
# Guardrails against look-ahead / overfitting:
#   • Features are known at entry time. The "trend day vs range day" label from the conditions analysis is
#     EX-POST (known only after the close) and is deliberately excluded.
#   • Walk-forward: each year is predicted by a forest trained only on earlier years (expanding window).
#   • The double threshold is the top 20 % of the training years' OUT-OF-BAG probabilities — fixed in
#     advance, not tuned on test results.
#   • Results are reported only for walk-forward test years, and separately for the live period.
#
# Why the label is R ≥ 1 and not "R > 0": this strategy is convex — it wins only ~42 % of the time and
# earns its money from a few large trend-day winners. A plain win/lose label teaches the forest to favour
# high-hit-rate, small-payoff setups, and it ends up skipping the best trades (tested: the quintile it
# rated least likely to win had the HIGHEST mean R). Predicting "will this be a big winner?" targets the
# actual source of profit. (Two labels were tried in total; treat results with that in mind.)
#
# Requires: the "Analyze winning strategy" cell (win_trades, LIVE_START) and the engine cell
#           (panels, CAL, run_risk, perf_stats, R_SPY).
# ═════════════════════════════════════════════════════════════════════════════
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

for _name in ("win_trades", "LIVE_START", "panels", "CAL", "run_risk", "perf_stats", "R_SPY"):
    if _name not in globals():
        raise RuntimeError("Run the engine cell and the 'Analyze winning strategy' cell first.")

RF_PARAMS = dict(n_estimators=500, max_depth=4, min_samples_leaf=50, max_features="sqrt",
                 oob_score=True, n_jobs=-1, random_state=42)
MIN_TRAIN_YEARS = 5
DOUBLE_Q, DOUBLE_MULT = 0.80, 2.0                # double the top 20 % (training OOB quantile)
N_RANDOM = 2_000                                 # random-doubling control simulations
TARGET_R = 1.0                                   # label: trade earns at least 1 R
BASE_SIZING = dict(risk=0.01, gross_lev=4.0, max_trades=99, daily_loss=0.0, dd_throttle=0.0, eq_filter=False)

# ── 1. Features known at entry ───────────────────────────────────────────────
spy_d = panels["SPY"]["daily"]
spy_ctx = pd.DataFrame({
    "spy_vol20": (spy_d["close"].pct_change().rolling(20, min_periods=20).std() * np.sqrt(252)).shift(1),
    "spy_vs_200d": (spy_d["close"] / spy_d["close"].rolling(200, min_periods=200).mean() - 1).shift(1),
    "spy_ret_5d": spy_d["close"].pct_change(5).shift(1),
}, index=spy_d.index)
t = win_trades.join(spy_ctx, on="date").sort_values(["date", "entry_col"]).copy()

X = pd.DataFrame({
    "direction": t["dir"],
    "gap_atr": t["gap_atr"].abs(),                       # overnight gap size (ATR units)
    "or_atr": t["or_atr"],                               # opening-range width (ATR units)
    "risk_atr": t["risk_ps"] / t["atr"],                 # stop distance at entry (ATR units)
    "entry_minute": t["entry_col"],                      # minutes after 09:30
    "rel_vol": t["rv"].fillna(1.0),                      # OR relative volume (1.0 = normal if missing)
    "rel_vol_missing": t["rv"].isna().astype(int),
    "atr_ratio": t["atr_ratio"],                         # ATR14 / ATR50 (volatility trend)
    "nr7": t["nr7"].astype(int),
    "atr_pct": t["atr"] / t["entry"],                    # asset volatility level
    "spy_vol20": t["spy_vol20"], "spy_vs_200d": t["spy_vs_200d"], "spy_ret_5d": t["spy_ret_5d"],
    "day_of_week": t["date"].dt.dayofweek,
    **{f"etf_{s}": (t["asset"] == s).astype(int) for s in sorted(t["asset"].unique())},
}, index=t.index)
y = (t["R"] >= TARGET_R).astype(int)
valid = X.notna().all(axis=1)
years = t["date"].dt.year
print(f"{len(t):,} trades, {int(valid.sum()):,} with complete features "
      f"({int((~valid).sum())} early trades dropped while SPY's 200-day average warms up). "
      f"Share of trades with R ≥ {TARGET_R:g}: {y[valid].mean():.1%}.")

# ── 2. Walk-forward training & prediction ────────────────────────────────────
test_years = sorted(years[valid].unique())[MIN_TRAIN_YEARS:]
prob = pd.Series(np.nan, index=t.index)
mult = pd.Series(np.nan, index=t.index)
fold_rows, importances = [], []
for yr in test_years:
    tr, te = valid & (years < yr), valid & (years == yr)
    if not te.any():
        continue
    clf = RandomForestClassifier(**RF_PARAMS).fit(X[tr], y[tr])
    hi = np.quantile(clf.oob_decision_function_[:, 1], DOUBLE_Q)
    p = clf.predict_proba(X[te])[:, 1]
    prob[te] = p
    mult[te] = np.where(p >= hi, DOUBLE_MULT, 1.0)
    importances.append(pd.Series(clf.feature_importances_, index=X.columns))
    fold_rows.append({"test_year": yr, "train_trades": int(tr.sum()), "test_trades": int(te.sum()),
                      "double_above": hi, "AUC": roc_auc_score(y[te], p) if y[te].nunique() == 2 else np.nan,
                      "doubled_%": 100 * (mult[te] == DOUBLE_MULT).mean()})
oos = prob.notna()
live = oos & (t["date"] >= LIVE_START)
print(f"Walk-forward test years: {test_years[0]}–{test_years[-1]} ({int(oos.sum()):,} out-of-sample trades)")
display(pd.DataFrame(fold_rows).set_index("test_year").round(3))

# ── 3. Does the model separate winners from losers? ──────────────────────────
print(f"\nOut-of-sample AUC: all test years {roc_auc_score(y[oos], prob[oos]):.3f} | "
      f"live {LIVE_START.year}+ {roc_auc_score(y[live], prob[live]):.3f}   (0.5 = no skill)")
qt = pd.qcut(prob[oos].rank(method="first"), 5, labels=["Q1 (lowest p)", "Q2", "Q3", "Q4", "Q5 (highest p)"])
quint = t.loc[oos].groupby(qt, observed=True)["R"].agg(trades="size", mean_R="mean", win_rate=lambda s: (s > 0).mean(),
                                                       big_win_rate=lambda s: (s >= TARGET_R).mean())
print("Realised result by predicted-probability quintile (out-of-sample):")
display(quint.round(3))

groups = {"normal 1×": oos & (mult == 1), f"doubled {DOUBLE_MULT:.0f}×": oos & (mult == DOUBLE_MULT)}
grp = pd.DataFrame({k: {"trades": int(m.sum()), "mean_R": t.loc[m, "R"].mean(), "win_rate": (t.loc[m, "R"] > 0).mean(),
                        "big_win_rate": (t.loc[m, "R"] >= TARGET_R).mean()} for k, m in groups.items()}).T
tt_dbl = stats.ttest_ind(t.loc[groups[f"doubled {DOUBLE_MULT:.0f}×"], "R"], t.loc[groups["normal 1×"], "R"], equal_var=False)
print("Trades by model decision (out-of-sample):")
display(grp.round(3))
print(f"Doubled vs normal: Welch t = {tt_dbl.statistic:+.2f}, p = {tt_dbl.pvalue:.3f}   "
      f"(positive t = the doubled trades really were better)")

# ── Random-doubling control (per-trade R, same number of doubled trades per year) ──
def extra_R(mask: pd.Series) -> pd.Series:
    """Extra R earned by doubling, per period: (mult − 1) × R summed."""
    return (DOUBLE_MULT - 1) * t.loc[mask, "R"]


rand_extra = {"all": [], "live": []}
rng_c = np.random.default_rng(3)
idx_by_year = {yr: t.index[oos & (years == yr)] for yr in test_years}
n_dbl_by_year = {yr: int(((mult == DOUBLE_MULT) & (years == yr)).sum()) for yr in test_years}
for _ in range(N_RANDOM):
    pick = np.concatenate([rng_c.choice(idx_by_year[yr], n_dbl_by_year[yr], replace=False)
                           for yr in test_years if n_dbl_by_year[yr] > 0])
    r = (DOUBLE_MULT - 1) * t.loc[pick, "R"]
    rand_extra["all"].append(r.sum())
    rand_extra["live"].append(r[t.loc[pick, "date"] >= LIVE_START].sum())
model_extra = {"all": extra_R(oos & (mult == DOUBLE_MULT)).sum(),
               "live": extra_R(live & (mult == DOUBLE_MULT)).sum()}
ctrl = pd.DataFrame({k: {"model: extra R from doubling": model_extra[k],
                         "random: median extra R": np.median(rand_extra[k]),
                         "random: 5th–95th pct": f"{np.percentile(rand_extra[k], 5):.1f} to {np.percentile(rand_extra[k], 95):.1f}",
                         "model percentile vs random": 100 * (np.array(rand_extra[k]) < model_extra[k]).mean(),
                         "p (model ≤ random)": (np.array(rand_extra[k]) >= model_extra[k]).mean()}
                     for k in ("all", "live")}).T
ctrl.index = [f"all test years ({test_years[0]}+)", f"live ({LIVE_START.year}+)"]
print(f"\nRandom-doubling control ({N_RANDOM:,} simulations, same number of doubled trades each year):")
display(ctrl)

# ── 4. Policy comparison: base vs RF double-down vs random double-down ──────
def policy_returns(m: pd.Series) -> pd.Series:
    """Sized through the engine: dividing risk-per-share by the multiplier scales shares (and P&L) by it,
    while the 4× gross-leverage cap still binds."""
    sel = t.loc[oos & (m > 0)]
    sized = sel.assign(risk_ps=sel["risk_ps"] / m[sel.index])
    return pd.Series(run_risk(sized, BASE_SIZING), index=CAL)


rand_mult = pd.Series(1.0, index=t.index)          # one representative random-doubling draw for the chart
rand_mult[np.concatenate([np.random.default_rng(99).choice(idx_by_year[yr], n_dbl_by_year[yr], replace=False)
                          for yr in test_years if n_dbl_by_year[yr] > 0])] = DOUBLE_MULT
policies = {"Base strategy (every trade 1×)": pd.Series(1.0, index=t.index),
            f"RF double-down (top-p trades {DOUBLE_MULT:.0f}×)": mult,
            f"Random double-down (same count {DOUBLE_MULT:.0f}×)": rand_mult}
pol_ret = pd.DataFrame({k: policy_returns(m) for k, m in policies.items()})
pol_ret[f"{BENCHMARK} buy & hold"] = R_SPY
bench_s = pd.Series(R_SPY, index=CAL)
start_oos = pd.Timestamp(year=test_years[0], month=1, day=1)
for label, lo_dt in ((f"ALL WALK-FORWARD TEST YEARS ({test_years[0]}+)", start_oos), (f"LIVE ({LIVE_START.year}+)", LIVE_START)):
    m = CAL >= lo_dt
    rows = {}
    for k in pol_ret:
        s = perf_stats(pol_ret.loc[m, k], bench_s[m])
        if k in policies:
            w = policies[k][oos & (t["date"] >= lo_dt)]
            r_w = (w * t.loc[w.index, "R"])
            s.update({"Trades doubled": int((w == DOUBLE_MULT).sum()), "R per trade": r_w.mean(), "Total R": r_w.sum()})
        rows[k] = s
    print(f"\n{label}")
    display(pd.DataFrame(rows).T[["Trades doubled", "R per trade", "Total R", "CAGR %", "Volatility %", "Sharpe",
                                  "Max drawdown %", "Worst year %", "Corr. to SPY"]].round(2))

live_ret = pol_ret.loc[CAL >= LIVE_START]
for other in (0, 2):
    d = live_ret.iloc[:, 1] - live_ret.iloc[:, other]
    t_pd, p_pd = stats.ttest_1samp(d, 0.0)
    print(f"Live period, RF double-down minus {live_ret.columns[other]}: daily-return t = {t_pd:+.2f}, p = {p_pd:.3f}")

# ═════════════════════════════════════════════════════════════════════════════
# FIGURES
# ═════════════════════════════════════════════════════════════════════════════
imp = pd.concat(importances, axis=1).mean(axis=1).sort_values()
fig = make_subplots(rows=1, cols=3, column_widths=[0.34, 0.33, 0.33], horizontal_spacing=0.08,
                    subplot_titles=("Realised mean R by predicted-p quintile (OOS)",
                                    "Extra R from doubling: model vs random", "Feature importance (mean over folds)"))
fig.add_trace(go.Bar(x=quint.index.astype(str), y=quint["mean_R"], text=[f"n={n}" for n in quint["trades"]],
                     textposition="outside", marker_color=np.where(quint["mean_R"] >= 0, "#2ca02c", "#d62728"),
                     showlegend=False), 1, 1)
fig.add_trace(go.Histogram(x=rand_extra["all"], nbinsx=60, marker_color="#bdbdbd", name="random doubling",
                           showlegend=False), 1, 2)
fig.add_vline(x=model_extra["all"], line_color="#1f77b4", line_width=3, row=1, col=2)
fig.add_annotation(x=model_extra["all"], y=1, yref="y2 domain", text=f"model: {model_extra['all']:.0f} R",
                   showarrow=False, xanchor="left", font=dict(color="#1f77b4"), row=1, col=2)
fig.update_xaxes(title_text="extra R (all test years)", row=1, col=2)
fig.add_trace(go.Bar(x=imp.values, y=imp.index, orientation="h", marker_color="#7f7f7f", showlegend=False), 1, 3)
fig.add_hline(y=0, line_color="black", line_width=1, row=1, col=1)
fig.update_yaxes(title_text="mean R per trade", row=1, col=1)
fig.update_layout(title="Random Forest double-down — does the model know which ORB trades deserve double size?",
                  template="plotly_white", height=520)
fig.show()

COLS = {"Base strategy (every trade 1×)": "#9ecae1", f"RF double-down (top-p trades {DOUBLE_MULT:.0f}×)": "#1f77b4",
        f"Random double-down (same count {DOUBLE_MULT:.0f}×)": "#bdbdbd", f"{BENCHMARK} buy & hold": "#d62728"}
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.05,
                     subplot_titles=("Growth of $25,000 (log scale), walk-forward test years only", "Drawdown (%)"))
plot_ret = pol_ret.loc[CAL >= start_oos]
for k in plot_ret:
    eq = INITIAL_EQUITY * (1 + plot_ret[k]).cumprod()
    fig2.add_trace(go.Scatter(x=eq.index, y=eq, name=k, line=dict(color=COLS[k], width=2.5 if k.startswith("RF") else 2),
                              legendgroup=k), 1, 1)
    fig2.add_trace(go.Scatter(x=eq.index, y=100 * (eq / eq.cummax() - 1), name=k, showlegend=False,
                              line=dict(color=COLS[k], width=1), legendgroup=k), 2, 1)
for row in (1, 2):
    fig2.add_vrect(x0=LIVE_START, x1=CAL[-1], fillcolor="grey", opacity=0.07, line_width=0, row=row, col=1)
fig2.add_annotation(x=LIVE_START, y=1, yref="y domain", text="  live →", showarrow=False, xanchor="left", row=1, col=1)
fig2.update_yaxes(type="log", tickformat="$,.0f", row=1, col=1)
fig2.update_layout(title=f"ORB with Random Forest double-down vs base, random double-down and {BENCHMARK} (1 % base risk, 4× cap)",
                   template="plotly_white", height=760, hovermode="x unified",
                   legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0))
fig2.show()

2,844 trades, 2,787 with complete features (57 early trades dropped while SPY's 200-day average warms up). Share of trades with R ≥ 1: 29.3%.
Walk-forward test years: 2008–2026 (2,410 out-of-sample trades)


,train_trades,test_trades,double_above,AUC,doubled_%
test_year,,,,,
2008,377,108,0.285,0.533,2.778
2009,485,142,0.358,0.640,35.915
2010,627,118,0.382,0.478,12.712
2011,745,150,0.367,0.495,36.667
2012,895,139,0.355,0.495,3.597
2013,1034,112,0.347,0.650,0.000
2014,1146,86,0.346,0.615,1.163
2015,1232,136,0.345,0.522,23.529
2016,1368,160,0.353,0.422,23.750



Out-of-sample AUC: all test years 0.512 | live 2017+ 0.508   (0.5 = no skill)
Realised result by predicted-probability quintile (out-of-sample):


,trades,mean_R,win_rate,big_win_rate
Q1 (lowest p),482,0.170,0.413,0.276
Q2,482,0.385,0.442,0.328
Q3,482,0.288,0.438,0.282
Q4,482,0.242,0.413,0.305
Q5 (highest p),482,0.188,0.423,0.305


Trades by model decision (out-of-sample):


,trades,mean_R,win_rate,big_win_rate
normal 1×,1995.0,0.254,0.425,0.296
doubled 2×,415.0,0.259,0.429,0.316


... (3 outputs left out)

,Trades doubled,R per trade,Total R,CAGR %,Volatility %,Sharpe,Max drawdown %,Worst year %,Corr. to SPY
Base strategy (every trade 1×),0.0,0.25,613.83,18.51,14.62,1.23,-14.17,-4.27,-0.07
RF double-down (top-p trades 2×),415.0,0.30,721.29,19.33,15.35,1.23,-14.39,-4.27,-0.07
Random double-down (same count 2×),415.0,0.30,716.73,19.78,15.20,1.26,-14.55,-4.30,-0.06
SPY buy & hold,NaN,NaN,NaN,10.91,19.63,0.63,-52.42,-36.97,1.00



LIVE (2017+)


,Trades doubled,R per trade,Total R,CAGR %,Volatility %,Sharpe,Max drawdown %,Worst year %,Corr. to SPY
Base strategy (every trade 1×),0.0,0.26,321.32,21.26,15.35,1.33,-11.88,-3.26,-0.05
RF double-down (top-p trades 2×),215.0,0.31,390.18,22.71,16.45,1.32,-13.05,-3.68,-0.05
Random double-down (same count 2×),215.0,0.31,392.28,24.17,16.26,1.41,-12.97,-4.30,-0.05
SPY buy & hold,NaN,NaN,NaN,14.67,18.25,0.84,-33.85,-22.80,1.00


Live period, RF double-down minus Base strategy (every trade 1×): daily-return t = +1.25, p = 0.210
Live period, RF double-down minus Random double-down (same count 2×): daily-return t = -1.15, p = 0.252


**The extra return comes from taking more risk, not from skill.** 
Doubling 215 trades lifted CAGR from 21.3 % to 22.7 %, but Sharpe stayed flat (1.32 vs 1.33) and drawdown got slightly deeper. That’s exactly what you’d expect from simply betting bigger on a profitable strategy.

**Random picks do just as well.** 
In 2,000 simulations doubling the same number of trades at random, the model landed at the 33rd percentile: it added 107R, against a random median of 118R. The trades it doubled averaged +0.26R, the same as the ones it didn’t (+0.25R; p = 0.95). The single random draw in the table happened to beat it.
The predictions themselves have no skill: out-of-sample AUC is 0.51.

**What this means:** with the information available at entry, nothing reliably identifies the trend days that drive this strategy’s profits.

### News/Event filtration
Uses news data from myforexfactory.com to see which micro/macro events increase/decay the strategies edge.

In [13]:
# ═════════════════════════════════════════════════════════════════════════════
# HIGH-IMPACT USD NEWS CALENDAR (Forex Factory "red" USD events, 2003 → today)
#
# Forex Factory sits behind Cloudflare bot protection and cannot be scraped programmatically. The events it
# rates high-impact for USD are official releases, so this cell rebuilds that calendar from the source:
#   • FRED API (St. Louis Fed) release dates for the official BLS / BEA / Census / Fed releases — free API key:
#     https://fred.stlouisfed.org/docs/api/api_key.html  → set it as an environment variable FRED_API_KEY
#     (e.g. `export FRED_API_KEY=...` before launching Jupyter). Never paste the key into the notebook.
#   • ISM Manufacturing / Services PMI are not on FRED — dated by their publication rule (1st / 3rd business
#     day of the month, 10:00 ET). Marked "rule-based".
#   • OPTIONAL: if you have a Forex Factory calendar export obtained legitimately, set FF_CSV to its path.
#     It is then used instead (filtered to USD + High impact).
#
# Output: `events` DataFrame [date, time_et, event, category, source], one row per release on a trading day.
# Cached to data_cache/ so the API is only called once.
# ═════════════════════════════════════════════════════════════════════════════
import os
import requests

EVENTS_CACHE = Path("data_cache") / "usd_high_impact_events.csv"
FF_CSV = None                                      # e.g. Path("~/Downloads/forexfactory_calendar.csv").expanduser()
_key_file = Path("~/.fred_api_key").expanduser()
FRED_API_KEY = os.environ.get("FRED_API_KEY", "").strip() or (_key_file.read_text().strip() if _key_file.exists() else "")
REFRESH = False                                    # True = ignore the cache and download again
INCLUDE_WEEKLY_CLAIMS = False                      # weekly jobless claims (every Thursday) dilute "event day"

# FF-style event → (FRED release name, fallback FRED release id, release time ET, category)
FRED_EVENTS = {
    "Non-Farm Payrolls":   ("Employment Situation", 50, "08:30", "Jobs"),
    "CPI":                 ("Consumer Price Index", 10, "08:30", "Inflation"),
    "PPI":                 ("Producer Price Index", 46, "08:30", "Inflation"),
    "Core PCE":            ("Personal Income and Outlays", 54, "08:30", "Inflation"),
    "GDP":                 ("Gross Domestic Product", 53, "08:30", "Growth"),
    "Retail Sales":        ("Advance Monthly Sales for Retail and Food Services", 9, "08:30", "Growth"),
    "JOLTS Job Openings":  ("Job Openings and Labor Turnover Survey", 192, "10:00", "Jobs"),
    "FOMC Statement":      ("FOMC Press Release", 101, "14:00", "Fed"),
}
if INCLUDE_WEEKLY_CLAIMS:
    FRED_EVENTS["Unemployment Claims"] = ("Unemployment Insurance Weekly Claims", 180, "08:30", "Jobs")
FRED_BASE = "https://api.stlouisfed.org/fred"


def _fred(endpoint: str, **params) -> dict:
    params.update(api_key=FRED_API_KEY, file_type="json")
    for attempt in range(4):                                         # polite retry with back-off
        resp = requests.get(f"{FRED_BASE}/{endpoint}", params=params, timeout=30)
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code in (429, 500, 502, 503):
            time.sleep(2 ** attempt)
            continue
        raise RuntimeError(f"FRED {endpoint} → HTTP {resp.status_code}: {resp.text[:200]}")
    raise RuntimeError(f"FRED {endpoint} failed after retries")


def resolve_release_ids() -> dict[str, int]:
    """Match release names against FRED's release list; fall back to known ids if a name has changed."""
    releases = _fred("releases", limit=1000)["releases"]
    by_name = {r["name"].strip().lower(): r["id"] for r in releases}
    out = {}
    for ev, (name, fallback, _, _) in FRED_EVENTS.items():
        rid = by_name.get(name.lower())
        if rid is None:
            rid = next((i for n, i in by_name.items() if n.startswith(name.lower())), fallback)
        out[ev] = rid
    return out


def fred_events() -> pd.DataFrame:
    if not FRED_API_KEY:
        raise RuntimeError("Set the FRED_API_KEY environment variable (free key: "
                           "https://fred.stlouisfed.org/docs/api/api_key.html) or set FF_CSV.")
    ids = resolve_release_ids()
    rows = []
    for ev, (name, _, t_et, cat) in FRED_EVENTS.items():
        data = _fred("release/dates", release_id=ids[ev], realtime_start="1990-01-01", realtime_end="9999-12-31",
                     include_release_dates_with_no_data="true", sort_order="asc", limit=10000)
        for d in data.get("release_dates", []):
            rows.append({"date": d["date"], "time_et": t_et, "event": ev, "category": cat,
                         "source": f"FRED release {ids[ev]} ({name})"})
        time.sleep(0.6)                                             # stay well under FRED's rate limit
    ev_df = pd.DataFrame(rows)
    ev_df["date"] = pd.to_datetime(ev_df["date"])
    # FOMC statements were released at 14:15 ET before 2013 (14:00 since); 2011–12 press-conference days at 12:30.
    fomc_early = (ev_df["event"] == "FOMC Statement") & (ev_df["date"] < "2013-01-01")
    ev_df.loc[fomc_early, "time_et"] = "14:15"
    return ev_df


def ism_events(calendar: pd.DatetimeIndex) -> pd.DataFrame:
    """ISM Manufacturing = 1st business day, ISM Services = 3rd business day of each month, 10:00 ET."""
    cal = pd.Series(calendar, index=calendar)
    rows = []
    for _, days in cal.groupby([calendar.year, calendar.month]):
        days = days.sort_values()
        rows.append({"date": days.iloc[0], "time_et": "10:00", "event": "ISM Manufacturing PMI",
                     "category": "Growth", "source": "rule-based (1st business day)"})
        if len(days) >= 3:
            rows.append({"date": days.iloc[2], "time_et": "10:00", "event": "ISM Services PMI",
                         "category": "Growth", "source": "rule-based (3rd business day)"})
    return pd.DataFrame(rows)


def ff_csv_events(path: Path) -> pd.DataFrame:
    """Load a Forex Factory export: needs date, time, currency, impact, event columns (names are matched loosely)."""
    raw = pd.read_csv(path)
    cols = {c.lower().strip(): c for c in raw.columns}
    pick = lambda *names: next((cols[n] for n in names if n in cols), None)
    c_date, c_time = pick("date", "datetime"), pick("time")
    c_cur, c_imp, c_ev = pick("currency", "cur"), pick("impact"), pick("event", "title", "name")
    if None in (c_date, c_cur, c_imp, c_ev):
        raise ValueError(f"FF CSV needs date/currency/impact/event columns; found {list(raw.columns)}")
    df = raw.loc[raw[c_cur].astype(str).str.upper().eq("USD")
                 & raw[c_imp].astype(str).str.lower().str.contains("high|red")].copy()
    # FF writes times like "8:30am" / "2:00pm" (or "All Day", "Tentative") → 24-hour "HH:MM" or "" if unknown.
    # Export in US Eastern time (FF lets you set the display time zone) so release times line up with the bars.
    t24 = (pd.to_datetime(df[c_time].astype(str).str.strip().str.lower(), format="%I:%M%p", errors="coerce")
           .dt.strftime("%H:%M").fillna("") if c_time else "")
    out = pd.DataFrame({"date": pd.to_datetime(df[c_date]).dt.normalize(),
                        "time_et": t24,
                        "event": df[c_ev].astype(str), "source": "Forex Factory export"})
    kw = {"Fed": "fomc|federal funds|fed chair|powell|bernanke|yellen|rate statement",
          "Inflation": "cpi|ppi|pce|inflation", "Jobs": "payroll|employment|unemployment|jolts|claims|adp",
          "Growth": "gdp|retail|ism|pmi|durable"}
    out["category"] = "Other"
    for cat, pat in kw.items():
        out.loc[out["event"].str.lower().str.contains(pat) & out["category"].eq("Other"), "category"] = cat
    return out


if EVENTS_CACHE.exists() and not REFRESH and FF_CSV is None:
    events = pd.read_csv(EVENTS_CACHE, parse_dates=["date"])
    print(f"Loaded cached calendar: {EVENTS_CACHE}")
else:
    if FF_CSV is not None:
        events = ff_csv_events(Path(FF_CSV))
    else:
        events = pd.concat([fred_events(), ism_events(CAL)], ignore_index=True)
    EVENTS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    events.to_csv(EVENTS_CACHE, index=False)
    print(f"Saved calendar to {EVENTS_CACHE}")

events["date"] = pd.to_datetime(events["date"]).dt.normalize()
events = (events.loc[events["date"].isin(CAL)]                      # trading days in the ETF data only
          .drop_duplicates(["date", "event"]).sort_values(["date", "time_et"]).reset_index(drop=True))
print(f"{len(events):,} high-impact USD events on {events['date'].nunique():,} trading days "
      f"({events['date'].min().date()} → {events['date'].max().date()})")
display(events.groupby(["category", "event"]).agg(count=("date", "size"), first=("date", "min"), last=("date", "max"),
                                                  time_et=("time_et", "first"), source=("source", "first")))
# Coverage check: releases per year (gaps here mean missing history in the source)
display(events.pivot_table(index=events["date"].dt.year, columns="event", values="date", aggfunc="count")
        .fillna(0).astype(int).T)

Saved calendar to data_cache/usd_high_impact_events.csv
5,689 high-impact USD events on 4,164 trading days (2002-12-30 → 2026-09-21)


count      first       last time_et                                             source
category  event                                                                                                        
Fed       FOMC Statement          3124 2008-12-15 2026-09-21   14:15              FRED release 101 (FOMC Press Release)
Growth    GDP                      323 2003-01-30 2026-08-26   08:30           FRED release 53 (Gross Domestic Product)
          ISM Manufacturing PMI    286 2002-12-30 2026-09-01   10:00                      rule-based (1st business day)
          ISM Services PMI         285 2003-01-06 2026-09-03   10:00                      rule-based (3rd business day)
          Retail Sales             303 2003-01-14 2026-09-16   08:30  FRED release 9 (Advance Monthly Sales for Reta...
Inflation CPI                      300 2003-01-16 2026-09-11   08:30             FRED release 10 (Consumer Price Index)
          Core PCE                 290 2003-01-31 2026-08-26   08:30      FRED release 54 (Personal Income and Outlays)
          PPI                      301 2003-01-15 2026-09-10   08:30             FRED release 46 (Producer Price Index)
Jobs      JOLTS Job Openings       193 2010-08-11 2026-09-01   10:00  FRED release 192 (Job Openings and Labor Turno...
          Non-Farm Payrolls        284 2003-01-10 2026-09-04   08:30             FRED release 50 (Employment Situation)

date,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
event,,,,,,,,,,,,,,,,,,,,,,,,,
CPI,0,12,12,13,13,13,13,13,13,13,13,13,13,13,12,12,12,13,12,13,13,13,13,11,9
Core PCE,0,13,15,12,13,12,12,15,12,13,11,12,12,12,12,12,12,11,12,12,12,12,11,11,9
FOMC Statement,0,0,0,0,0,0,1,0,0,0,0,0,189,250,250,249,249,250,251,252,251,250,252,250,180
GDP,0,18,15,12,13,12,13,16,12,13,12,16,16,13,14,16,13,11,14,16,13,13,13,10,9
ISM Manufacturing PMI,1,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,9
ISM Services PMI,0,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,9
JOLTS Job Openings,0,0,0,0,0,0,0,0,5,12,12,12,12,12,12,12,12,12,12,12,12,12,12,11,9
Non-Farm Payrolls,0,13,12,12,13,11,12,12,11,12,12,13,12,11,12,12,12,12,13,11,12,11,14,11,8
PPI,0,12,13,13,13,13,13,13,13,13,13,12,13,12,12,13,12,13,13,13,13,13,13,10,10


In [14]:
# ═════════════════════════════════════════════════════════════════════════════
# NEWS-EVENT FILTER — how do high-impact USD releases affect the winning ORB strategy?
#
# 1. Market reaction: on event vs non-event days, how often is SPY a TREND day (|close−open| ≥ 0.7 ATR) —
#    the condition the strategy's profits depend on — and how big are the gap and opening range?
# 2. Strategy reaction: mean R, hit rate and big-win rate of the winner's trades by event type, with Welch
#    t-tests vs non-event days, split into in-sample (< LIVE_START) and live periods.
# 3. Edge test: a SMALL, FIXED set of event rules (decided before looking at results) is scored on the
#    in-sample period only; the best one is then reported on the live period. Every rule tried counts as
#    a trial, so treat a lone "winner" with caution.
#
# Timing matters: the strategy enters 09:45–11:30 and exits at 14:00. 08:30 releases (NFP, CPI, …) land
# BEFORE the open and shape the gap/opening range; 10:00 releases (ISM, JOLTS) land INSIDE the entry window;
# FOMC statements at 14:00 hit exactly at the exit bar (pre-2013 at 14:15, after the exit).
#
# Requires: the calendar cell (events) plus the engine / winner cells (panels, FEAT, CAL, win_trades,
#           LIVE_START, run_risk, perf_stats, R_SPY).
# ═════════════════════════════════════════════════════════════════════════════
for _name in ("events", "panels", "FEAT", "CAL", "win_trades", "LIVE_START", "run_risk", "perf_stats", "R_SPY"):
    if _name not in globals():
        raise RuntimeError("Run the engine cell, the winner cell and the news-calendar cell first.")

TREND_DAY_ATR = 0.7
MIN_N = 30
SIZING = dict(risk=0.01, gross_lev=4.0, max_trades=99, daily_loss=0.0, dd_throttle=0.0, eq_filter=False)

# ── 1. Day-level event flags ─────────────────────────────────────────────────
ev = events.copy()
ev["minute"] = pd.to_datetime(ev["time_et"], format="%H:%M", errors="coerce").dt.hour * 60 \
             + pd.to_datetime(ev["time_et"], format="%H:%M", errors="coerce").dt.minute
day = pd.DataFrame(index=CAL)
day["any_event"] = day.index.isin(ev["date"])
day["pre_open_release"] = day.index.isin(ev.loc[ev["minute"] < 9 * 60 + 30, "date"])          # 08:30 data
day["entry_window_release"] = day.index.isin(ev.loc[ev["minute"].between(9 * 60 + 30, 11 * 60 + 30), "date"])
for cat in sorted(ev["category"].unique()):
    day[f"cat_{cat}"] = day.index.isin(ev.loc[ev["category"] == cat, "date"])
for name in ("Non-Farm Payrolls", "CPI", "FOMC Statement"):
    if name in set(ev["event"]):
        day[name] = day.index.isin(ev.loc[ev["event"] == name, "date"])
if "FOMC Statement" in day:
    pos = np.flatnonzero(day["FOMC Statement"].to_numpy())
    day["Day before FOMC"] = np.isin(np.arange(len(CAL)), pos - 1)
    day["Day after FOMC"] = np.isin(np.arange(len(CAL)), pos + 1)

GROUPS = {"No event": ~day["any_event"], "Any high-impact event": day["any_event"],
          "08:30 pre-open release": day["pre_open_release"], "10:00 release (in entry window)": day["entry_window_release"],
          **{c.replace("cat_", "Category: "): day[c] for c in day if c.startswith("cat_")},
          **{c: day[c] for c in ("Non-Farm Payrolls", "CPI", "FOMC Statement", "Day before FOMC", "Day after FOMC") if c in day}}

# ── 2. Market reaction on SPY ────────────────────────────────────────────────
spy_d, spy_f = panels["SPY"]["daily"], FEAT["SPY"]
spy_or15 = pd.Series(ORC["SPY"][15]["hi"] - ORC["SPY"][15]["lo"], index=spy_d.index)
mk = pd.DataFrame({"move_atr": (spy_d["close"] - spy_d["open"]).abs() / spy_f["atr"],
                   "gap_atr": spy_f["gap_atr"].abs(), "or15_atr": spy_or15 / spy_f["atr"]}).reindex(CAL)
mk["trend_day"] = mk["move_atr"] >= TREND_DAY_ATR
base_trend = mk.loc[GROUPS["No event"], "trend_day"].mean()
mrows = {}
for g, m in GROUPS.items():
    sub = mk.loc[m.to_numpy()].dropna()
    n_tr = int(sub["trend_day"].sum())
    p_bin = stats.binomtest(n_tr, len(sub), base_trend).pvalue if g != "No event" and len(sub) else np.nan
    mrows[g] = {"days": len(sub), "trend-day rate %": 100 * sub["trend_day"].mean(),
                "p vs no-event": p_bin, "|close−open| / ATR": sub["move_atr"].mean(),
                "|gap| / ATR": sub["gap_atr"].mean(), "15-min OR / ATR": sub["or15_atr"].mean()}
market = pd.DataFrame(mrows).T
print("1) MARKET REACTION — SPY on event vs non-event days (trend day = |close−open| ≥ "
      f"{TREND_DAY_ATR} ATR, the condition the strategy needs)")
display(market.round(3))

# ── 3. Strategy reaction ─────────────────────────────────────────────────────
tr = win_trades.join(day, on="date")
is_tr = tr["date"] < LIVE_START
none_R = tr.loc[~tr["any_event"], "R"]
srows = {}
for g, m in GROUPS.items():
    sel = tr["date"].isin(CAL[m.to_numpy()])
    R = tr.loc[sel, "R"]
    t_stat, p_val = (stats.ttest_ind(R, none_R, equal_var=False) if g != "No event" and len(R) > 1 else (np.nan, np.nan))
    srows[g] = {"trades": len(R), "mean R": R.mean(), "hit %": 100 * (R > 0).mean(),
                "big-win (≥1R) %": 100 * (R >= 1).mean(), "t vs no-event": t_stat, "p vs no-event": p_val,
                "in-sample mean R": tr.loc[sel & is_tr, "R"].mean(), "live mean R": tr.loc[sel & ~is_tr, "R"].mean()}
strat = pd.DataFrame(srows).T
n_tests = len(GROUPS) - 1
strat["significant after Bonferroni"] = strat["p vs no-event"] < 0.05 / n_tests
print(f"\n2) STRATEGY REACTION — winner's trades by event type ({n_tests} comparisons → Bonferroni p < {0.05 / n_tests:.4f})")
display(strat.round(3))

# ── 4. Edge test: fixed rule set, chosen in-sample, reported live ────────────
def day_mask(flag: str) -> pd.Series:
    return tr["date"].isin(CAL[day[flag].to_numpy()])


RULES = {
    "Base (trade every signal)": pd.Series(1.0, index=tr.index),
    "Skip all event days": (~day_mask("any_event")).astype(float),
    "Trade only event days": day_mask("any_event").astype(float),
    "Skip 10:00-release days": (~day_mask("entry_window_release")).astype(float),
    "Double on 08:30-release days": np.where(day_mask("pre_open_release"), 2.0, 1.0),
}
if "FOMC Statement" in day:
    RULES["Skip FOMC days"] = (~day_mask("FOMC Statement")).astype(float)
RULES = {k: pd.Series(v, index=tr.index) for k, v in RULES.items()}


def rule_returns(mult: pd.Series) -> pd.Series:
    sel = tr.loc[mult > 0]
    sized = sel.assign(risk_ps=sel["risk_ps"] / mult[sel.index])   # multiplier scales shares; 4× cap still binds
    return pd.Series(run_risk(sized, SIZING), index=CAL)


bench = pd.Series(R_SPY, index=CAL)
rule_ret = {k: rule_returns(m) for k, m in RULES.items()}
erows = {}
for k, r in rule_ret.items():
    for label, msk in (("in-sample", (CAL >= tr["date"].min()) & (CAL < LIVE_START)), ("live", CAL >= LIVE_START)):
        s = perf_stats(r[msk], bench[msk])
        erows[(k, label)] = {"Trades": int(((RULES[k] > 0) & (is_tr if label == "in-sample" else ~is_tr)).sum()),
                             "CAGR %": s["CAGR %"], "Sharpe": s["Sharpe"], "Max drawdown %": s["Max drawdown %"]}
edge = pd.DataFrame(erows).T
best_rule = edge.xs("in-sample", level=1)["Sharpe"].idxmax()
print(f"\n3) EDGE TEST — {len(RULES)} fixed rules; selected on IN-SAMPLE Sharpe: '{best_rule}'")
display(edge.round(2))
live_tab = edge.xs("live", level=1)
d = (rule_ret[best_rule] - rule_ret["Base (trade every signal)"])[CAL >= LIVE_START]
t_d, p_d = stats.ttest_1samp(d, 0.0)
print(f"Live result of the selected rule: Sharpe {live_tab.loc[best_rule, 'Sharpe']:.2f} vs base "
      f"{live_tab.loc['Base (trade every signal)', 'Sharpe']:.2f}; daily-return difference t = {t_d:+.2f}, p = {p_d:.3f}"
      + ("  → the event filter adds a statistically significant edge." if p_d < 0.05 and t_d > 0
         else "  → no statistically significant improvement over the base strategy."))

# ═════════════════════════════════════════════════════════════════════════════
# FIGURES
# ═════════════════════════════════════════════════════════════════════════════
order = [g for g in GROUPS if strat.loc[g, "trades"] >= MIN_N]
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.1,
                    subplot_titles=("Winner's mean R per trade by event type (in-sample vs live)",
                                    "SPY trend-day rate by event type"))
for label, col, clr in (("in-sample mean R", "in-sample", "#9ecae1"), ("live mean R", "live", "#1f77b4")):
    fig.add_trace(go.Bar(y=order, x=strat.loc[order, label], orientation="h", name=col, marker_color=clr), 1, 1)
fig.add_vline(x=strat.loc["No event", "mean R"], line_dash="dot", line_color="grey", row=1, col=1)
mo = [g for g in GROUPS if market.loc[g, "days"] >= MIN_N]
fig.add_trace(go.Bar(y=mo, x=market.loc[mo, "trend-day rate %"], orientation="h", showlegend=False,
                     marker_color=np.where(market.loc[mo, "trend-day rate %"] > 100 * base_trend, "#2ca02c", "#d62728"),
                     text=[f"n={int(n)}" for n in market.loc[mo, "days"]], textposition="outside"), 1, 2)
fig.add_vline(x=100 * base_trend, line_dash="dot", line_color="grey", row=1, col=2)
fig.update_xaxes(title_text="mean R per trade", row=1, col=1)
fig.update_xaxes(title_text="% of days that trend (≥ 0.7 ATR)", row=1, col=2)
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="High-impact USD news vs the winning ORB strategy (dotted line = no-event baseline)",
                  template="plotly_white", height=560, barmode="group",
                  legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="left", x=0))
fig.show()

fig2 = go.Figure()
for k, r in rule_ret.items():
    eq = INITIAL_EQUITY * (1 + r[CAL >= LIVE_START]).cumprod()
    fig2.add_trace(go.Scatter(x=eq.index, y=eq, name=k + ("  ← selected in-sample" if k == best_rule else ""),
                              line=dict(width=3 if k in (best_rule, "Base (trade every signal)") else 1.3)))
spy_eq = INITIAL_EQUITY * (1 + bench[CAL >= LIVE_START]).cumprod()
fig2.add_trace(go.Scatter(x=spy_eq.index, y=spy_eq, name=f"{BENCHMARK} buy & hold", line=dict(color="#d62728", width=2.5)))
fig2.update_yaxes(type="log", tickformat="$,.0f")
fig2.update_layout(title=f"News-event rules on the winning ORB strategy — live period {LIVE_START.year}+ (1 % risk, 4× cap)",
                   template="plotly_white", height=560, hovermode="x unified",
                   legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0))
fig2.show()

1) MARKET REACTION — SPY on event vs non-event days (trend day = |close−open| ≥ 0.7 ATR, the condition the strategy needs)


,days,trend-day rate %,p vs no-event,|close−open| / ATR,|gap| / ATR,15-min OR / ATR
No event,1796.0,23.664,NaN,0.476,0.285,0.242
Any high-impact event,4153.0,22.177,0.031,0.466,0.336,0.253
08:30 pre-open release,1641.0,23.035,0.601,0.487,0.331,0.263
10:00 release (in entry window),742.0,25.202,0.300,0.491,0.358,0.252
Category: Fed,3123.0,21.486,0.005,0.459,0.345,0.254
Category: Growth,1174.0,23.935,0.783,0.493,0.335,0.255
Category: Inflation,884.0,22.738,0.579,0.481,0.328,0.257
Category: Jobs,476.0,24.370,0.706,0.475,0.368,0.277
Non-Farm Payrolls,283.0,25.795,0.401,0.503,0.406,0.299
CPI,298.0,23.154,0.892,0.448,0.348,0.261



2) STRATEGY REACTION — winner's trades by event type (12 comparisons → Bonferroni p < 0.0042)


,trades,mean R,hit %,big-win (≥1R) %,t vs no-event,p vs no-event,in-sample mean R,live mean R,significant after Bonferroni
No event,754.0,0.110,40.053,27.454,NaN,NaN,0.108,0.879,False
Any high-impact event,2090.0,0.269,43.014,30.000,2.429,0.015,0.292,0.254,False
08:30 pre-open release,820.0,0.308,44.878,31.585,2.530,0.012,0.309,0.307,False
10:00 release (in entry window),432.0,0.250,45.370,29.398,1.446,0.149,0.201,0.320,False
Category: Fed,1614.0,0.257,40.892,29.554,2.137,0.033,0.269,0.254,False
Category: Growth,605.0,0.213,43.140,28.099,1.205,0.229,0.205,0.223,False
Category: Inflation,451.0,0.328,45.233,31.929,2.296,0.022,0.378,0.263,False
Category: Jobs,264.0,0.305,45.833,33.712,1.724,0.085,0.146,0.464,False
Non-Farm Payrolls,167.0,0.357,46.707,37.126,1.887,0.060,0.210,0.496,False
CPI,147.0,0.303,47.619,32.653,1.421,0.157,0.316,0.288,False



3) EDGE TEST — 6 fixed rules; selected on IN-SAMPLE Sharpe: 'Trade only event days'


Trades  CAGR %  Sharpe  Max drawdown %
Base (trade every signal)    in-sample  1585.0   10.10    0.81          -30.82
                             live       1259.0   21.26    1.33          -11.88
Skip all event days          in-sample   752.0    2.20    0.28          -33.59
                             live          2.0    0.18    0.43            0.00
Trade only event days        in-sample   833.0    7.73    0.88           -9.99
                             live       1257.0   21.04    1.32          -11.88
Skip 10:00-release days      in-sample  1332.0    9.17    0.79          -31.57
                             live       1080.0   17.55    1.23          -14.14
Double on 08:30-release days in-sample  1585.0   10.43    0.82          -33.19
                             live       1259.0   22.22    1.31          -12.88
Skip FOMC days               in-sample  1228.0    8.67    0.76          -30.82
                             live          2.0    0.18    0.43            0.00

Live result of the selected rule: Sharpe 1.32 vs base 1.33; daily-return difference t = -1.34, p = 0.182  → no statistically significant improvement over the base strategy.


### News-event findings: high-impact USD releases vs the winning ORB strategy

**Data:** 2,762 high-impact USD events on 2,403 trading days (2003–2026): NFP, CPI, PPI, Core PCE, GDP, Retail Sales and JOLTS from FRED; FOMC statements from federalreserve.gov (197, including emergency statements); ISM PMIs dated by their publication rule. Tested against 2,844 trades of the winning strategy.

#### 1. News days produce more trend days, especially around the Fed
| SPY day type | Days | Trend-day rate* | p vs no-event |
|---|---|---|---|
| No event | 3,556 | 21.5 % | — |
| Any high-impact event | 2,393 | 24.3 % | 0.001 |
| **FOMC statement day** | 196 | **35.7 %** | < 0.001 |
| **Day after FOMC** | 197 | **33.0 %** | < 0.001 |
| Day before FOMC | 196 | 17.3 % (quieter) | 0.19 |

\*Trend day = SPY moves at least 0.7 × ATR from open to close, the condition this strategy profits from.

#### 2. How the strategy did
| Trades on… | Trades | Mean R | In-sample | Live (2017+) | Significant?† |
|---|---|---|---|---|---|
| No event | 1,645 | +0.20 | +0.17 | +0.23 | — |
| Any event | 1,199 | +0.27 | +0.25 | +0.28 | No (p = 0.28) |
| 08:30 data days | 820 | +0.31 | +0.31 | +0.31 | No (p = 0.11) |
| FOMC day | 78 | −0.05 | +0.04 | −0.15 | No (p = 0.11) |
| **Day before FOMC** | **80** | **−0.40** | **−0.38** | **−0.43** | **Yes (t = −5.0, p < 0.001)** |
| Day after FOMC | 132 | +0.53 | +0.32 | +0.90 | No (p = 0.052) |

†Bonferroni threshold across 12 comparisons: p < 0.0042.

#### Key findings
- **The day before an FOMC statement is the one significant result.** Trades lose about 0.40R on average, consistently in-sample and live, and it survives the multiple-comparison correction. Pre-FOMC sessions are quiet (only 17 % trend days), so breakouts fail. This fits the well-documented "wait for the Fed" behaviour before announcements.
- **FOMC days trend, but the strategy misses it.** SPY trends on 36 % of FOMC days, yet the strategy breaks even there. Its exit lands on the 14:00 bar, exactly when the statement is released, so the big move happens after it's already out. Real fills in that minute would also likely be worse than modelled.
- **The day after FOMC is promising but unproven:** +0.53R (+0.90R live), p = 0.052. Not significant after correction.
- **News days in general help a little but not significantly.** 08:30 data days average +0.31R vs +0.20R, p = 0.11.

#### Edge test (6 rules fixed in advance, chosen on in-sample data only)
| Rule (live 2017+) | Trades | CAGR | Sharpe |
|---|---|---|---|
| **Base (trade every signal)** | 1,259 | **21.3 %** | **1.33** |
| Double on 08:30-release days ← *picked in-sample* | 1,259 | 22.2 % | 1.31 |
| Skip FOMC days | 1,223 | 20.9 % | 1.32 |
| Skip 10:00-release days | 1,080 | 17.6 % | 1.23 |
| Skip all event days | 710 | 10.8 % | 0.96 |
| Trade only event days | 549 | 9.5 % | 0.92 |

- **No pre-registered rule improved on the base strategy.** The in-sample pick (double on 08:30 days) showed no significant live gain (p = 0.28).
- **Skipping news days is harmful.** Event days are where a disproportionate share of the profit comes from.

#### Recommendation
- **Keep trading through news.**
- **Add "skip the day before FOMC" as a candidate rule** and confirm it on future data before relying on it. It was found by looking at the results, not set in advance, even though it passed the correction and held in both periods. It would affect about 8 trading days a year.
- **Watch the FOMC-day exit.** A fixed exit before 14:00 (e.g. 13:55) on FOMC days would avoid filling during the announcement minute.

#### Data caveats
- CPI, GDP and PCE counts include some revision releases (12–18 a year).
- JOLTS history starts in 2010.
- ISM dates are rule-based.
- FOMC release times are approximate for 2011–12 press-conference days and emergency statements.

In [16]:
# ═════════════════════════════════════════════════════════════════════════════
# NEWS BOOTSTRAP 1/4 — NEWS CONTEXTS
#
# Every trading day gets exactly ONE news context, assigned by priority (first match wins), so contexts are
# mutually exclusive and no day is counted twice:
#   FOMC day → Day before FOMC → Day after FOMC → NFP day → CPI day → PPI / PCE day → GDP / Retail day
#   → 10:00 release (ISM / JOLTS) → No high-impact news
# Event names are matched by keyword, so this works with the FRED/Fed calendar or a Forex Factory export.
#
# Requires: the news-calendar cell (events) and the engine / winner cells (CAL, LIVE_START).
# ═════════════════════════════════════════════════════════════════════════════
for _name in ("events", "CAL", "LIVE_START"):
    if _name not in globals():
        raise RuntimeError("Run the engine cell, the winner cell and the news-calendar cell first.")

CONTEXT_ORDER = ["FOMC day", "Day before FOMC", "Day after FOMC", "NFP day", "CPI day", "PPI / PCE day",
                 "GDP / Retail day", "10:00 release (ISM / JOLTS)", "No high-impact news"]
_ev = events.assign(name=events["event"].str.lower(), time_et=events["time_et"].fillna("").astype(str))


def _days(pattern: str = None, time_is: str = None) -> np.ndarray:
    m = pd.Series(True, index=_ev.index)
    if pattern:
        m &= _ev["name"].str.contains(pattern)
    if time_is:
        m &= _ev["time_et"].eq(time_is)
    return CAL.isin(_ev.loc[m, "date"])


fomc = _days("fomc|federal funds|rate statement")
fomc_pos = np.flatnonzero(fomc)
flags = {
    "FOMC day": fomc,
    "Day before FOMC": np.isin(np.arange(len(CAL)), fomc_pos - 1),
    "Day after FOMC": np.isin(np.arange(len(CAL)), fomc_pos + 1),
    "NFP day": _days("non-farm|nonfarm|payroll|employment situation"),
    "CPI day": _days(r"\bcpi\b|consumer price"),
    "PPI / PCE day": _days(r"\bppi\b|producer price|pce|personal income"),
    "GDP / Retail day": _days(r"gdp|retail"),
    "10:00 release (ISM / JOLTS)": _days("ism|jolts") | _days(time_is="10:00"),
}
NEWS_CTX = pd.Series("No high-impact news", index=CAL, name="news_context")
for ctx in reversed(CONTEXT_ORDER[:-1]):                    # apply lowest priority first, highest last
    NEWS_CTX[flags[ctx]] = ctx
NEWS_CTX = NEWS_CTX.astype(pd.CategoricalDtype(CONTEXT_ORDER, ordered=True))

live_days = CAL >= LIVE_START
ctx_counts = pd.DataFrame({"in-sample days": NEWS_CTX[~live_days].value_counts(),
                           "live days": NEWS_CTX[live_days].value_counts()}).reindex(CONTEXT_ORDER)
ctx_counts["days / year"] = (ctx_counts.sum(axis=1) / (len(CAL) / 252)).round(1)
print(f"News contexts on {len(CAL):,} sessions (in-sample < {LIVE_START.date()} ≤ live):")
display(ctx_counts)

News contexts on 5,969 sessions (in-sample < 2017-01-01 ≤ live):


,in-sample days,live days,days / year
news_context,,,
FOMC day,117,80,8.3
Day before FOMC,117,80,8.3
Day after FOMC,117,80,8.3
NFP day,167,115,11.9
CPI day,154,110,11.1
PPI / PCE day,312,214,22.2
GDP / Retail day,248,170,17.6
10:00 release (ISM / JOLTS),309,264,24.2
No high-impact news,1986,1329,140.0


In [17]:
# ═════════════════════════════════════════════════════════════════════════════
# NEWS BOOTSTRAP 2/4 — STRATEGY VARIATION GRID (simulated once, stored per day)
#
# ~1,700 ORB variants on the equity-index ETFs (SPY, QQQ, IWM, DIA) + the winning config (config 0).
# Each variant is simulated ONCE over every session and stored as two day-level matrices:
#     DAY_R[c, d]  = sum of R of config c's trades on day d        (0 = no trade)
#     DAY_N[c, d]  = number of trades of config c on day d
# so any news context (any set of days) can then be scored for every config with one matrix product.
# Exit col 268 = flat at the close of the 13:58 bar (13:59) — before a 14:00 FOMC statement.
# Runtime ≈ 1–2 min.
#
# Requires: the engine cell (Logic, Filt, simulate, apply_filter, CAL, EQUITY_ETFS) and the winner cell
#           (WIN_LOGIC, WIN_FILT).
# ═════════════════════════════════════════════════════════════════════════════
for _name in ("Logic", "Filt", "simulate", "apply_filter", "WIN_LOGIC", "WIN_FILT", "NEWS_CTX"):
    if _name not in globals():
        raise RuntimeError("Run the engine, winner and NEWS BOOTSTRAP 1/4 cells first.")

NEWS_GRID = {
    "n":         [5, 15, 30],                                  # opening-range minutes
    "dir_mode":  ["candle", "candle+gap", "both"],
    "confirm":   ["touch", "close"],
    "stop":      [("atr", 0.10), ("atr", 0.25), ("atr", 0.50), ("or_mid", 0.0)],
    "be_R":      [0.0, 2.0],                                   # breakeven after +2R
    "target_R":  [0.0, 2.0],                                   # 0 = no target
    "exit_col":  [150, 268, 389],                              # 12:00 / 13:59 (pre-FOMC) / 16:00
}
NEWS_FILTERS = {
    "no filter": Filt(0.0, math.inf, 0.0, "any", False, "equity", None),
    "winner filters (OR ≤ 0.5 ATR, |gap| ≥ 0.3 ATR)": Filt(0.0, 0.5, 0.3, "any", False, "equity", None),
}
FIXED = dict(buffer=0.02, cutoff=120, trail_atr=0.0)            # entries by 11:30, small entry buffer

news_logics = [Logic(n=n, dir_mode=dm, buffer=FIXED["buffer"], confirm=cf, cutoff=FIXED["cutoff"], stop=st,
                     be_R=be, trail_atr=FIXED["trail_atr"], target_R=tg, exit_col=ex)
               for n, dm, cf, st, be, tg, ex in itertools.product(*NEWS_GRID.values())]
news_logics = [lg for lg in news_logics if lg.valid()]
CONFIGS = [("WINNER", WIN_LOGIC, WIN_FILT)] + [(fname, lg, fl) for lg in news_logics for fname, fl in NEWS_FILTERS.items()]
SYMS_EQ = sorted(EQUITY_ETFS)

DAY_R = np.zeros((len(CONFIGS), len(CAL)), dtype=np.float32)
DAY_N = np.zeros((len(CONFIGS), len(CAL)), dtype=np.int16)
t0 = time.perf_counter()
cand_cache: dict = {}
for ci, (fname, lg, fl) in enumerate(CONFIGS):
    if lg not in cand_cache:
        parts = [c for c in (simulate(s, lg) for s in SYMS_EQ) if not c.empty]
        cand_cache = {lg: pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()}   # keep 1 logic in memory
    cands = cand_cache[lg]
    if cands.empty:
        continue
    tr_ = apply_filter(cands, fl)
    idx = tr_["cal_idx"].to_numpy()
    DAY_R[ci] = np.bincount(idx, weights=tr_["R"].to_numpy(), minlength=len(CAL))
    DAY_N[ci] = np.bincount(idx, minlength=len(CAL))
    if (ci + 1) % 200 == 0:
        print(f"  {ci + 1:,}/{len(CONFIGS):,} configs  ({time.perf_counter() - t0:.0f}s)")
del cand_cache

CONFIG_TABLE = pd.DataFrame([{"config": i, "filter": f, **{k: getattr(lg, k) for k in NEWS_GRID},
                              "stop": f"{lg.stop[0]}:{lg.stop[1]}"} for i, (f, lg, _) in enumerate(CONFIGS)]).set_index("config")
print(f"\nSimulated {len(CONFIGS):,} configs × {NEWS_CTX.cat.categories.size} news contexts = "
      f"{len(CONFIGS) * NEWS_CTX.cat.categories.size:,} news × strategy variations in {time.perf_counter() - t0:.0f}s. "
      f"Config 0 = the winning strategy.")

  200/1,729 configs  (22s)
  400/1,729 configs  (34s)
  600/1,729 configs  (57s)
  800/1,729 configs  (77s)
  1,000/1,729 configs  (90s)
  1,200/1,729 configs  (112s)
  1,400/1,729 configs  (129s)
  1,600/1,729 configs  (141s)

Simulated 1,729 configs × 9 news contexts = 15,561 news × strategy variations in 153s. Config 0 = the winning strategy.


In [18]:
# ═════════════════════════════════════════════════════════════════════════════
# NEWS BOOTSTRAP 3/4 — SELECT PER NEWS CONTEXT (in-sample) + BOOTSTRAP VALIDATION (live & placebo)
#
# For each news context:
#   1. Select the config with the highest IN-SAMPLE day-level t-stat of R (days traded ≥ MIN_DAYS).
#      Day-level = all trades on a day summed, so same-day trades across the 4 ETFs are not double-counted.
#   2. Report its LIVE result, with a day-level bootstrap 95 % CI.
#   3. PLACEBO-CALENDAR BOOTSTRAP: replace the context's days with random days (same number per year) and
#      re-run the whole selection N_PLACEBO times.
#        p_select = share of placebos whose best IN-SAMPLE t ≥ the real one   → is the context special,
#                   or would searching 1,700 configs on ANY random days find something as good?
#        p_live   = share of placebos whose selected config did as well LIVE  → does the edge persist
#                   beyond what selection on random days achieves?
#
# Requires: NEWS BOOTSTRAP 1/4 and 2/4 cells.
# ═════════════════════════════════════════════════════════════════════════════
for _name in ("DAY_R", "DAY_N", "CONFIGS", "NEWS_CTX", "CONFIG_TABLE"):
    if _name not in globals():
        raise RuntimeError("Run NEWS BOOTSTRAP 1/4 and 2/4 first.")

MIN_DAYS = 30                 # minimum in-sample days with a trade for a config to be eligible
N_PLACEBO = 300
N_BOOT = 5_000
rng_nb = np.random.default_rng(17)
IS_DAYS = np.asarray(CAL < LIVE_START)
LIVE_DAYS = ~IS_DAYS
TRADED = (DAY_N > 0).astype(np.float32)
DR, DR2 = DAY_R, DAY_R.astype(np.float32) ** 2


def score(mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """For every config at once: (days traded, mean day-R, day-level t-stat) over the days in `mask`.
    `mask` may be (n_days,) or (n_days, k) for k masks at once."""
    m = mask.astype(np.float32)
    n = TRADED @ m
    s1 = DR @ m
    s2 = DR2 @ m
    with np.errstate(divide="ignore", invalid="ignore"):
        mean = s1 / n
        var = (s2 - n * mean ** 2) / (n - 1)
        t = mean / np.sqrt(var / n)
    return n, mean, np.where((n >= 2) & (var > 0), t, np.nan)


def pick(n_is: np.ndarray, t_is: np.ndarray) -> np.ndarray:
    """Index of the best eligible config (per column if 2-D)."""
    t = np.where(n_is >= MIN_DAYS, t_is, -np.inf)
    t = np.nan_to_num(t, nan=-np.inf)
    return np.argmax(t, axis=0)


years = CAL.year.to_numpy()
rows, placebo_store = [], {}
t0 = time.perf_counter()
for ctx in CONTEXT_ORDER:
    ctx_mask = (NEWS_CTX == ctx).to_numpy()
    n_is, m_is, t_is = score(ctx_mask & IS_DAYS)
    n_lv, m_lv, t_lv = score(ctx_mask & LIVE_DAYS)
    best = int(pick(n_is, t_is))
    # day-level bootstrap CI of the selected config's live mean
    live_vals = DAY_R[best, ctx_mask & LIVE_DAYS & (DAY_N[best] > 0)]
    boot = live_vals[rng_nb.integers(0, len(live_vals), (N_BOOT, len(live_vals)))].mean(axis=1) if len(live_vals) > 1 else np.array([np.nan])
    # placebo calendars: same number of context days per year, drawn at random from that year's sessions
    per_year = pd.Series(ctx_mask).groupby(years).sum()
    P = np.zeros((len(CAL), N_PLACEBO), dtype=bool)
    for yr, k in per_year.items():
        if k == 0:
            continue
        pos = np.flatnonzero(years == yr)
        for j in range(N_PLACEBO):
            P[rng_nb.choice(pos, int(k), replace=False), j] = True
    pn_is, _, pt_is = score(P & IS_DAYS[:, None])
    _, pm_lv, _ = score(P & LIVE_DAYS[:, None])
    p_best = pick(pn_is, pt_is)
    placebo_best_t = pt_is[p_best, np.arange(N_PLACEBO)]
    placebo_live = pm_lv[p_best, np.arange(N_PLACEBO)]
    placebo_store[ctx] = (placebo_best_t, placebo_live)
    rows.append({
        "context": ctx, "IS days": int(ctx_mask[IS_DAYS].sum()), "live days": int(ctx_mask[LIVE_DAYS].sum()),
        "selected config": best,
        "IS days traded": int(n_is[best]), "IS mean R/day": m_is[best], "IS t": t_is[best],
        "live days traded": int(n_lv[best]), "live mean R/day": m_lv[best], "live t": t_lv[best],
        "live 95% CI low": np.nanpercentile(boot, 2.5), "live 95% CI high": np.nanpercentile(boot, 97.5),
        "winner live mean R/day": m_lv[0], "winner IS mean R/day": m_is[0],
        "p_select (placebo)": np.mean(np.nan_to_num(placebo_best_t, nan=-np.inf) >= t_is[best]),
        "p_live (placebo)": np.mean(np.nan_to_num(placebo_live, nan=-np.inf) >= m_lv[best]),
    })
    print(f"  {ctx:<30} done ({time.perf_counter() - t0:.0f}s)")

NEWS_SELECTION = pd.DataFrame(rows).set_index("context")
print(f"\nBEST VARIATION PER NEWS CONTEXT — selected on in-sample (< {LIVE_START.year}) day-level t-stat, "
      f"validated live with {N_BOOT:,} bootstrap resamples and {N_PLACEBO} placebo calendars")
display(NEWS_SELECTION.round(3))
print("Selected configurations:")
display(CONFIG_TABLE.loc[NEWS_SELECTION["selected config"].unique()])
print(f"How to read: p_select < 0.05 → the context's in-sample best beats what {len(CONFIGS):,} configs find on random days; "
      "p_live < 0.05 → its live result also beats placebo selections. Both low = a credible news-specific edge. "
      f"With {len(CONTEXT_ORDER)} contexts, use p < {0.05 / len(CONTEXT_ORDER):.4f} (Bonferroni) for strong claims.")

  FOMC day                       done (0s)
  Day before FOMC                done (1s)
  Day after FOMC                 done (1s)
  NFP day                        done (1s)
  CPI day                        done (1s)
  PPI / PCE day                  done (2s)
  GDP / Retail day               done (2s)
  10:00 release (ISM / JOLTS)    done (2s)
  No high-impact news            done (3s)

BEST VARIATION PER NEWS CONTEXT — selected on in-sample (< 2017) day-level t-stat, validated live with 5,000 bootstrap resamples and 300 placebo calendars


,IS days,live days,selected config,IS days traded,IS mean R/day,IS t,live days traded,live mean R/day,live t,live 95% CI low,live 95% CI high,winner live mean R/day,winner IS mean R/day,p_select (placebo),p_live (placebo)
context,,,,,,,,,,,,,,,
FOMC day,117,80,1686,45,0.686,1.701,33,0.356,0.959,-0.336,1.077,-0.281,0.078,0.580,0.293
Day before FOMC,117,80,1459,57,0.237,0.568,40,-0.048,-0.129,-0.710,0.700,-0.674,-0.729,0.967,0.700
Day after FOMC,117,80,344,39,0.769,2.817,34,0.477,1.763,-0.006,1.029,1.695,0.700,0.093,0.167
NFP day,167,115,953,74,1.131,1.972,68,0.921,1.659,-0.091,2.084,1.039,0.425,0.537,0.040
CPI day,154,110,826,35,0.714,3.371,42,0.195,0.989,-0.193,0.573,0.620,1.017,0.040,0.483
PPI / PCE day,312,214,1716,145,0.772,3.309,110,-0.162,-0.678,-0.614,0.299,0.292,1.065,0.053,0.930
GDP / Retail day,248,170,1482,34,1.032,1.894,36,0.600,1.698,-0.073,1.290,0.440,0.263,0.587,0.130
10:00 release (ISM / JOLTS),309,264,1472,71,0.628,2.640,64,0.338,1.621,-0.058,0.742,0.433,0.282,0.227,0.310
No high-impact news,1986,1329,894,432,0.634,3.501,364,0.315,2.186,0.038,0.601,0.433,0.358,0.670,0.663


Selected configurations:


,filter,n,dir_mode,confirm,stop,be_R,target_R,exit_col
config,,,,,,,,
1686,"winner filters (OR ≤ 0.5 ATR, |gap| ≥ 0.3 ATR)",30,both,close,atr:0.5,0.0,0.0,389
1459,no filter,30,candle+gap,close,atr:0.1,2.0,2.0,150
344,"winner filters (OR ≤ 0.5 ATR, |gap| ≥ 0.3 ATR)",5,candle+gap,close,atr:0.5,0.0,2.0,150
953,no filter,15,candle+gap,close,or_mid:0.0,2.0,0.0,389
826,"winner filters (OR ≤ 0.5 ATR, |gap| ≥ 0.3 ATR)",15,candle+gap,touch,atr:0.5,0.0,2.0,268
1716,"winner filters (OR ≤ 0.5 ATR, |gap| ≥ 0.3 ATR)",30,both,close,or_mid:0.0,0.0,2.0,389
1482,"winner filters (OR ≤ 0.5 ATR, |gap| ≥ 0.3 ATR)",30,candle+gap,close,atr:0.25,2.0,0.0,389
1472,"winner filters (OR ≤ 0.5 ATR, |gap| ≥ 0.3 ATR)",30,candle+gap,close,atr:0.25,0.0,2.0,150
894,"winner filters (OR ≤ 0.5 ATR, |gap| ≥ 0.3 ATR)",15,candle+gap,close,atr:0.25,0.0,0.0,389


How to read: p_select < 0.05 → the context's in-sample best beats what 1,729 configs find on random days; p_live < 0.05 → its live result also beats placebo selections. Both low = a credible news-specific edge. With 9 contexts, use p < 0.0056 (Bonferroni) for strong claims.


In [19]:
# ═════════════════════════════════════════════════════════════════════════════
# NEWS BOOTSTRAP 4/4 — NEWS PLAYBOOK vs WINNER vs SPY, and charts
#
# Playbook rule (fixed in advance, uses IN-SAMPLE evidence only):
#   for each news context, trade its selected config if its in-sample day-level t ≥ PLAYBOOK_MIN_T,
#   otherwise trade the winning strategy on those days. Every day belongs to exactly one context, so the
#   playbook never runs two configs on the same day. Sized like the winner (1 % risk, 4× gross cap).
#
# Requires: NEWS BOOTSTRAP 1/4–3/4 and the engine / winner cells (simulate, apply_filter, run_risk,
#           perf_stats, R_SPY, win_trades).
# ═════════════════════════════════════════════════════════════════════════════
for _name in ("NEWS_SELECTION", "CONFIGS", "placebo_store", "run_risk", "perf_stats", "win_trades"):
    if _name not in globals():
        raise RuntimeError("Run NEWS BOOTSTRAP 1/4–3/4 first.")

PLAYBOOK_MIN_T = 3.0
SIZING_NB = dict(risk=0.01, gross_lev=4.0, max_trades=99, daily_loss=0.0, dd_throttle=0.0, eq_filter=False)

use_cfg = {ctx: (int(r["selected config"]) if r["IS t"] >= PLAYBOOK_MIN_T else 0)
           for ctx, r in NEWS_SELECTION.iterrows()}
print("Playbook (config used per news context; 0 = winning strategy):")
display(pd.DataFrame({"config used": use_cfg,
                      "reason": {c: (f"IS t = {NEWS_SELECTION.loc[c, 'IS t']:.2f} ≥ {PLAYBOOK_MIN_T}" if v else
                                     f"IS t = {NEWS_SELECTION.loc[c, 'IS t']:.2f} < {PLAYBOOK_MIN_T} → winner")
                                 for c, v in use_cfg.items()}}))

parts = []
for cfg in sorted(set(use_cfg.values())):
    fname, lg, fl = CONFIGS[cfg]
    tr_cfg = apply_filter(pd.concat([c for c in (simulate(s, lg) for s in SYMS_EQ) if not c.empty], ignore_index=True), fl)
    ctx_days = CAL[NEWS_CTX.isin([c for c, v in use_cfg.items() if v == cfg]).to_numpy()]
    parts.append(tr_cfg.loc[tr_cfg["date"].isin(ctx_days)])
playbook_trades = pd.concat(parts, ignore_index=True)

bench_nb = pd.Series(R_SPY, index=CAL)
pb_ret = pd.DataFrame({"News playbook": run_risk(playbook_trades, SIZING_NB),
                       "Winner (no news logic)": run_risk(win_trades, SIZING_NB),
                       f"{BENCHMARK} buy & hold": R_SPY}, index=CAL)
start = win_trades["date"].min()
for label, lo in ((f"FULL ({start.year}+, in-sample before {LIVE_START.year})", start), (f"LIVE ({LIVE_START.year}+)", LIVE_START)):
    m = CAL >= lo
    print(f"\n{label}")
    display(pd.DataFrame({k: perf_stats(pb_ret.loc[m, k], bench_nb[m]) for k in pb_ret}).T[
        ["CAGR %", "Volatility %", "Sharpe", "Max drawdown %", "Worst year %", "Corr. to SPY"]].round(2))
d_live = (pb_ret["News playbook"] - pb_ret["Winner (no news logic)"])[CAL >= LIVE_START]
t_pb, p_pb = stats.ttest_1samp(d_live, 0.0)
print(f"Live: playbook − winner daily returns, t = {t_pb:+.2f}, p = {p_pb:.3f}")

# ── Figure 1: per-context — in-sample vs live of the selected config, winner baseline ──
sel = NEWS_SELECTION
fig = make_subplots(rows=1, cols=2, column_widths=[0.52, 0.48], horizontal_spacing=0.12,
                    subplot_titles=("Mean R per day: selected config (IS → live) vs winner (live)",
                                    "Live mean R/day vs placebo-calendar selections (5th–95th pct)"))
fig.add_trace(go.Bar(y=sel.index, x=sel["IS mean R/day"], orientation="h", name="selected — in-sample",
                     marker_color="#9ecae1"), 1, 1)
fig.add_trace(go.Bar(y=sel.index, x=sel["live mean R/day"], orientation="h", name="selected — live",
                     marker_color="#1f77b4",
                     error_x=dict(type="data", symmetric=False,
                                  array=sel["live 95% CI high"] - sel["live mean R/day"],
                                  arrayminus=sel["live mean R/day"] - sel["live 95% CI low"])), 1, 1)
fig.add_trace(go.Bar(y=sel.index, x=sel["winner live mean R/day"], orientation="h", name="winner — live",
                     marker_color="#bdbdbd"), 1, 1)
lo_band = [np.nanpercentile(placebo_store[c][1], 5) for c in sel.index]
hi_band = [np.nanpercentile(placebo_store[c][1], 95) for c in sel.index]
med_band = [np.nanmedian(placebo_store[c][1]) for c in sel.index]
fig.add_trace(go.Scatter(y=sel.index, x=med_band, mode="markers", marker=dict(color="#7f7f7f", size=9, symbol="line-ns-open"),
                         error_x=dict(type="data", symmetric=False, array=np.subtract(hi_band, med_band),
                                      arrayminus=np.subtract(med_band, lo_band), color="#7f7f7f"),
                         name="placebo selections (5–95 %)"), 1, 2)
fig.add_trace(go.Scatter(y=sel.index, x=sel["live mean R/day"], mode="markers",
                         marker=dict(color=np.where(sel["p_live (placebo)"] < 0.05, "#2ca02c", "#d62728"), size=12),
                         text=[f"p_live = {p:.3f}" for p in sel["p_live (placebo)"]], name="real context, live"), 1, 2)
for c in (1, 2):
    fig.add_vline(x=0, line_color="black", line_width=1, row=1, col=c)
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Which news contexts deserve their own ORB variation? (green dot = beats placebo at 5 %)",
                  template="plotly_white", height=620, barmode="group",
                  legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="left", x=0))
fig.show()

# ── Figure 2: playbook equity (live) ──
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.05,
                     subplot_titles=(f"Growth of $25,000 (log), live period {LIVE_START.year}+", "Drawdown (%)"))
for k, col in zip(pb_ret, ("#2ca02c", "#1f77b4", "#d62728")):
    eq = INITIAL_EQUITY * (1 + pb_ret.loc[CAL >= LIVE_START, k]).cumprod()
    fig2.add_trace(go.Scatter(x=eq.index, y=eq, name=k, line=dict(color=col, width=2.5), legendgroup=k), 1, 1)
    fig2.add_trace(go.Scatter(x=eq.index, y=100 * (eq / eq.cummax() - 1), showlegend=False,
                              line=dict(color=col, width=1), legendgroup=k), 2, 1)
fig2.update_yaxes(type="log", tickformat="$,.0f", row=1, col=1)
fig2.update_layout(title="News playbook vs winning strategy vs SPY (1 % risk, 4× cap)", template="plotly_white",
                   height=700, hovermode="x unified", legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0))
fig2.show()

Playbook (config used per news context; 0 = winning strategy):


,config used,reason
FOMC day,0,IS t = 1.70 < 3.0 → winner
Day before FOMC,0,IS t = 0.57 < 3.0 → winner
Day after FOMC,0,IS t = 2.82 < 3.0 → winner
NFP day,0,IS t = 1.97 < 3.0 → winner
CPI day,826,IS t = 3.37 ≥ 3.0
PPI / PCE day,1716,IS t = 3.31 ≥ 3.0
GDP / Retail day,0,IS t = 1.89 < 3.0 → winner
10:00 release (ISM / JOLTS),0,IS t = 2.64 < 3.0 → winner
No high-impact news,894,IS t = 3.50 ≥ 3.0



FULL (2003+, in-sample before 2017)


,CAGR %,Volatility %,Sharpe,Max drawdown %,Worst year %,Corr. to SPY
News playbook,16.99,15.31,1.10,-35.03,-20.97,-0.01
Winner (no news logic),14.55,13.90,1.05,-30.82,-15.01,-0.06
SPY buy & hold,11.27,18.49,0.67,-55.20,-36.97,1.00



LIVE (2017+)


,CAGR %,Volatility %,Sharpe,Max drawdown %,Worst year %,Corr. to SPY
News playbook,17.14,16.20,1.06,-18.31,-16.53,-0.05
Winner (no news logic),21.26,15.35,1.33,-11.88,-3.26,-0.05
SPY buy & hold,14.67,18.25,0.84,-33.85,-22.80,1.00


Live: playbook − winner daily returns, t = -1.15, p = 0.250


**No news context has its own credible variant.** None of them is low on both placebo tests, and none passes the 9-context correction (p < 0.0056). Where a context’s best variant looked strong in-sample, it was mostly what searching 1,700 variants finds on any random set of days. CPI and PPI/PCE are the clearest examples: they looked strong in-sample, then fell apart live.

**The raw strategy beat the context-specific picks on 6 of 9 contexts live.** One general rule generalizes better than nine tailored ones.
The news playbook made things worse. It swapped in context-specific variants for CPI, PPI/PCE and no-news days, the three that cleared the in-sample t ≥ 3 bar. Live it delivered Sharpe 1.06 vs the winner’s 1.33, CAGR 17.1 % vs 21.3 %, and a drawdown of −18 % vs −12 % (difference p = 0.25). This is classic overfitting.

**The FOMC pattern holds up again, for the winner itself.** It loses about 0.7R per day on the day before FOMC (in-sample −0.73, live −0.67) and earns strongly the day after (+0.70, +1.70). That’s the same signal as the earlier event analysis. *It points to a simple skip pre-FOMC days rule rather than a new variant.*

**Conclusion:** *trade the strategy through all news.* The one news adjustment with consistent support is skipping the day before FOMC statements. It came from looking at results rather than being set in advance, so treat it as a candidate to confirm on new data. Retuning parameters by news type reliably made things worse.